# 3\.3\.2 Calibrate and Generate DBSCAN Anomalies

3\.3\.2 is where we calibrate DBSCAN inside the shared anomaly framework established in 3\.3\.1\. The notebook works with the inherited PCA and UMAP representation branches, the shared calibration samples, and the selected comparison\-universe defaults to evaluate how DBSCAN behaves across Bus, Subway, Taxi, FHVHV, and Multimodal mobility\-state surfaces\.

The focus is on local density structure, reasonable parameter regions, stability under small parameter changes, and the shape of the anomaly outputs that DBSCAN produces within each modality and representation\. By the end of the notebook, we want a set of modality\-specific and representation\-specific DBSCAN configurations that are coherent enough to carry forward into downstream anomaly generation and later cross\-framework comparison\.

In [1]:
# ---------------------------------------------------------------------
# Import libraries for DBSCAN calibration and anomaly generation
# ---------------------------------------------------------------------

from pathlib import Path
from time import perf_counter
import json
import gc
import shutil

import duckdb
import numpy as np
import pandas as pd

import geopandas as gpd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from shapely import wkb

from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

from project_branding import (
    BRAND_COLORS,
    BRAND_COLOR_SEQUENCE,
    BRAND_DIVERGING_SEQUENCE,
    BRAND_MAP_COLORS,
)

px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = BRAND_COLOR_SEQUENCE

BRAND_SEQUENTIAL_SCALE = [
    BRAND_COLORS["ice"],
    BRAND_COLORS["pale_peach"],
    BRAND_COLORS["terracotta"],
    BRAND_COLORS["dark_teal"],
]


def get_feature_set_color_map(feature_sets=None):
    if feature_sets is None:
        feature_sets = MODEL_FEATURE_SET_NAMES

    return {
        feature_set_name: BRAND_COLOR_SEQUENCE[index % len(BRAND_COLOR_SEQUENCE)]
        for index, feature_set_name in enumerate(feature_sets)
    }


def apply_brand_plotly_layout(
    fig,
    *,
    title=None,
    width=None,
    height=None,
    legend_title=None,
    margin=None,
):
    fig.update_layout(
        template="plotly_white",
        paper_bgcolor="white",
        plot_bgcolor=BRAND_COLORS["ice"],
        font={"color": BRAND_COLORS["dark_teal"]},
        title={
            "text": title,
            "font": {"color": BRAND_COLORS["dark_teal"]},
        } if title else None,
        width=width,
        height=height,
        margin=margin or {"l": 70, "r": 40, "t": 80, "b": 70},
    )

    fig.update_xaxes(
        gridcolor="rgba(11, 78, 74, 0.14)",
        zerolinecolor="rgba(11, 78, 74, 0.20)",
        tickfont={"color": BRAND_COLORS["dark_teal"]},
        title_font={"color": BRAND_COLORS["dark_teal"]},
    )
    fig.update_yaxes(
        gridcolor="rgba(11, 78, 74, 0.14)",
        zerolinecolor="rgba(11, 78, 74, 0.20)",
        tickfont={"color": BRAND_COLORS["dark_teal"]},
        title_font={"color": BRAND_COLORS["dark_teal"]},
    )

    if legend_title is not None:
        fig.update_layout(
            legend={
                "title": {"text": legend_title},
                "orientation": "h",
                "yanchor": "top",
                "y": -0.18,
                "xanchor": "center",
                "x": 0.5,
                "bgcolor": "rgba(255,255,255,0.85)",
                "bordercolor": BRAND_COLORS["seafoam"],
                "borderwidth": 1,
                "font": {"color": BRAND_COLORS["dark_teal"]},
            }
        )

    return fig


print("DBSCAN imports and branding helpers loaded.")

DBSCAN imports and branding helpers loaded.


In [2]:
# ---------------------------------------------------------------------
# Define project paths and 3.3.2 output directories
# ---------------------------------------------------------------------

PROJECT_ROOT = Path("/datasets/_deepnote_work")
PIPELINE_DATA_DIR = PROJECT_ROOT / "pipeline_data"

FINAL_311_DIR = PIPELINE_DATA_DIR / "3.1.1.final_tables"
FINAL_321_DIR = PIPELINE_DATA_DIR / "3.2.1.final_tables"
FINAL_322_DIR = PIPELINE_DATA_DIR / "3.2.2.final_tables"
FINAL_324_DIR = PIPELINE_DATA_DIR / "3.2.4.final_tables"
FINAL_331_DIR = PIPELINE_DATA_DIR / "3.3.1.final_tables"

INTERMEDIATE_332_DIR = PIPELINE_DATA_DIR / "3.3.2.intermediate_tables"
FINAL_332_DIR = PIPELINE_DATA_DIR / "3.3.2.final_tables"

INTERMEDIATE_332_DIR.mkdir(parents=True, exist_ok=True)
FINAL_332_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# ---------------------------------------------------------------------
# Define inherited anomaly-framework defaults for 3.3.2
# ---------------------------------------------------------------------

MODEL_FEATURE_SET_NAMES = ["bus", "subway", "taxi", "fhvhv", "multimodal"]
DBSCAN_REPRESENTATION_NAMES = ["pca_80pct", "umap_pca_to_umap_10d"]

MODEL_ID_COLUMN = "modeling_row_id"
TEMPORAL_BUCKET_COLUMN = "temporal_bucket"
DATE_COLUMN = "date"
ZONE_ID_COLUMN = "taxi_zone_id"
ZONE_COLUMN = "zone"
BOROUGH_COLUMN = "borough"
PRE_POST_CP_COLUMN = "pre_post_cp"
POLICY_GEOGRAPHY_COLUMN = "policy_geography_class"

DEFAULT_METADATA_COLUMNS = [
    MODEL_ID_COLUMN,
    DATE_COLUMN,
    TEMPORAL_BUCKET_COLUMN,
    PRE_POST_CP_COLUMN,
    ZONE_ID_COLUMN,
    ZONE_COLUMN,
    BOROUGH_COLUMN,
    POLICY_GEOGRAPHY_COLUMN,
]

# ------------------------------------------------------------------
# Inherited downstream framework decisions from 3.3.1
# ------------------------------------------------------------------

SHARED_DBSCAN_COMPARISON_UNIVERSE = "same_temporal_bucket_and_policy_geography"
SHARED_DBSCAN_GROUPING_COLUMNS = [TEMPORAL_BUCKET_COLUMN, POLICY_GEOGRAPHY_COLUMN]

COMPARISON_UNIVERSE_GROUPING_COLUMNS = {
    "global_all_rows": [],
    "same_temporal_bucket": [TEMPORAL_BUCKET_COLUMN],
    "same_temporal_bucket_and_pre_post_cp": [TEMPORAL_BUCKET_COLUMN, PRE_POST_CP_COLUMN],
    "same_temporal_bucket_and_policy_geography": [TEMPORAL_BUCKET_COLUMN, POLICY_GEOGRAPHY_COLUMN],
}

TAXI_POLICY_SANITY_UNIVERSE = "same_temporal_bucket_and_pre_post_cp"
TAXI_POLICY_SANITY_GROUPING_COLUMNS = [TEMPORAL_BUCKET_COLUMN, PRE_POST_CP_COLUMN]

TAXI_PCA_HANDLING_DECISION = "split_prepost_period"

# We are calibrating DBSCAN separately by modality and representation.
DBSCAN_CONFIGURATION_SCOPE = "per_modality_x_representation"

DBSCAN_DENSITY_REPRESENTATION_NAMES = ["pca_80pct", "umap_pca_to_umap_10d"]
DBSCAN_DENSITY_GROUPING_COLUMNS = [
    TEMPORAL_BUCKET_COLUMN,
    POLICY_GEOGRAPHY_COLUMN,
]
DBSCAN_DENSITY_GROUP_KEY_COLUMN = "comparison_group_key"

DBSCAN_DENSITY_PROFILE_PANEL_PATH = (
    INTERMEDIATE_332_DIR / "dbscan_density_profile_panel.parquet"
)
DBSCAN_DENSITY_GROUP_COUNTS_PATH = (
    INTERMEDIATE_332_DIR / "dbscan_density_group_counts.parquet"
)

# Shared anomaly representations advancing downstream.
REPRESENTATION_ROLE_BY_NAME = {
    "pca_80pct": "primary_linear_representation",
    "umap_pca_to_umap_10d": "nonlinear_complement_representation",
}

# ------------------------------------------------------------------
# Sampling and calibration defaults
# ------------------------------------------------------------------

ANOMALY_CALIBRATION_SAMPLE_ROWS = 50_000
DBSCAN_RANDOM_STATE = 42

# k-distance profiling defaults
DBSCAN_K_DISTANCE_K_VALUES = [5, 10, 15, 20]
DBSCAN_DEFAULT_MIN_SAMPLES_GRID = [10, 15, 20, 30]

# We will usually propose eps grids from empirical distance summaries later.
DBSCAN_EPS_GRID_QUANTILES = [0.90, 0.92, 0.94, 0.95, 0.96, 0.97, 0.98]

# Lightweight diagnostic cap for expensive per-group pilot runs.
DBSCAN_PILOT_MAX_ROWS_PER_GROUP = 4_000
DBSCAN_PILOT_MIN_ROWS_PER_GROUP = 100

# ------------------------------------------------------------------
# Rebuild / write toggles
# ------------------------------------------------------------------

REBUILD_DBSCAN_INPUT_SUMMARIES = False
REBUILD_DBSCAN_DENSITY_DIAGNOSTICS = False
REBUILD_DBSCAN_PARAMETER_CALIBRATION = False
REBUILD_DBSCAN_STABILITY_DIAGNOSTICS = False
REBUILD_DBSCAN_REPRESENTATION_COMPARISONS = False
REBUILD_DBSCAN_GEOMETRIC_EXPLANATION = False
REBUILD_DBSCAN_TOP_FEATURE_DRIVERS = False
REBUILD_DBSCAN_FINAL_OUTPUTS = False
WRITE_332_OUTPUTS = True

# ------------------------------------------------------------------
# Inherited asset paths from 3.3.1 / upstream notebooks
# ------------------------------------------------------------------

ANOMALY_FRAMEWORK_FINAL_EXPORT_PATHS = {
    "shared_representation_defaults": FINAL_331_DIR / "shared_representation_defaults.csv",
    "shared_calibration_defaults": FINAL_331_DIR / "shared_calibration_defaults.csv",
    "shared_comparison_universe_defaults": FINAL_331_DIR / "shared_comparison_universe_defaults.csv",
    "shared_framework_interpretation_defaults": FINAL_331_DIR / "shared_framework_interpretation_defaults.csv",
    "shared_framework_asset_manifest": FINAL_331_DIR / "shared_framework_asset_manifest.csv",
}

ROW_METADATA_PATHS_BY_SET = {
    feature_set: FINAL_311_DIR / f"{feature_set}_row_metadata.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

ANOMALY_CALIBRATION_ID_PATHS_BY_SET = {
    feature_set_name: FINAL_331_DIR / f"{feature_set_name}_anomaly_calibration_ids.parquet"
    for feature_set_name in MODEL_FEATURE_SET_NAMES
}

ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET = {
    feature_set_name: FINAL_331_DIR / f"{feature_set_name}_anomaly_calibration_metadata.parquet"
    for feature_set_name in MODEL_FEATURE_SET_NAMES
}

PCA_80_FINAL_PATHS_BY_SET = {
    "bus": FINAL_321_DIR / "bus_pca_modeling_scores.parquet",
    "subway": FINAL_321_DIR / "subway_pca_modeling_scores.parquet",
    "taxi": FINAL_321_DIR / "taxi_pca_modeling_scores.parquet",
    "fhvhv": FINAL_321_DIR / "fhvhv_pca_modeling_scores.parquet",
    "multimodal": FINAL_321_DIR / "multimodal_pca_modeling_scores.parquet",
}

TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD = {
    "pre_cp": FINAL_322_DIR / "taxi_pre_cp_pca_80pct_modeling_scores.parquet",
    "post_cp": FINAL_322_DIR / "taxi_post_cp_pca_80pct_modeling_scores.parquet",
}

UMAP_PCA_TO_UMAP_10D_FINAL_PATHS_BY_SET = {
    feature_set_name: FINAL_324_DIR / f"{feature_set_name}_umap_pca_to_umap_10d.parquet"
    for feature_set_name in MODEL_FEATURE_SET_NAMES
}

SOURCE_DATA_DIR = Path("source_data")
TAXI_ZONE_SOURCE_CSV_PATH = SOURCE_DATA_DIR / "NYC_Taxi_Zones_20260526.csv"
TAXI_ZONE_GEOMETRY_PATH = PIPELINE_DATA_DIR / "1.1.6.nyc_taxi_zones.parquet"
MERGED_FINAL_DIR = PIPELINE_DATA_DIR / "1.3.1.final_tables"
HARMONIZED_TAXI_ZONES_PATH = MERGED_FINAL_DIR / "nyc_taxi_zones_harmonized.parquet"

# ------------------------------------------------------------------
# 3.3.2 intermediate / final output paths
# ------------------------------------------------------------------

DBSCAN_INPUT_ASSET_SUMMARY_PATH = INTERMEDIATE_332_DIR / "dbscan_input_asset_summary.parquet"
DBSCAN_DENSITY_PROFILE_PATH = INTERMEDIATE_332_DIR / "dbscan_density_profile.parquet"
DBSCAN_K_DISTANCE_PROFILE_PATH = INTERMEDIATE_332_DIR / "dbscan_k_distance_profile.parquet"
DBSCAN_PARAMETER_GRID_SUMMARY_PATH = INTERMEDIATE_332_DIR / "dbscan_parameter_grid_summary.parquet"
DBSCAN_STABILITY_SUMMARY_PATH = INTERMEDIATE_332_DIR / "dbscan_stability_summary.parquet"
DBSCAN_REPRESENTATION_COMPARISON_PATH = INTERMEDIATE_332_DIR / "dbscan_representation_comparison.parquet"

DBSCAN_SELECTED_CONFIGURATION_PATH = FINAL_332_DIR / "selected_dbscan_configurations.csv"
DBSCAN_ANOMALY_EXPORT_MANIFEST_PATH = FINAL_332_DIR / "dbscan_anomaly_export_manifest.csv"

dbscan_inspection_rows_path = (
    INTERMEDIATE_332_DIR / "dbscan_representative_inspection_rows.parquet"
)
dbscan_geometric_explanation_path = (
    INTERMEDIATE_332_DIR / "dbscan_representative_geometric_explanation.parquet"
)
dbscan_geometric_explanation_validation_path = (
    INTERMEDIATE_332_DIR / "dbscan_representative_geometric_explanation_validation.parquet"
)
dbscan_top_feature_driver_validation_path = (
    INTERMEDIATE_332_DIR / "dbscan_representative_top_feature_driver_validation.parquet"
)

In [4]:
# ---------------------------------------------------------------------
# Define helper functions for shared asset loading and rebuild logic
# ---------------------------------------------------------------------

def sql_path(path_like) -> str:
    return str(Path(path_like)).replace("\\", "/")

def duckdb_identifier(name: str) -> str:
    escaped_name = str(name).replace('"', '""')
    return f'"{escaped_name}"'

def should_rebuild(rebuild_toggle: bool, output_path: Path) -> bool:
    return rebuild_toggle or (not output_path.exists())

def load_parquet_columns(parquet_path: Path) -> list[str]:
    with duckdb.connect() as con:
        schema_df = con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{sql_path(parquet_path)}')"
        ).fetchdf()
    return schema_df["column_name"].tolist()

def load_shared_framework_table(table_name: str) -> pd.DataFrame:
    table_path = ANOMALY_FRAMEWORK_FINAL_EXPORT_PATHS[table_name]
    return pd.read_csv(table_path)

def get_representation_path(feature_set_name: str, representation_name: str, *, pre_post_cp: str | None = None) -> Path:
    # Taxi PCA is the one special inherited case. Later blocks can pass
    # pre_post_cp when they need the split-period branch.
    if representation_name == "pca_80pct":
        if feature_set_name == "taxi" and TAXI_PCA_HANDLING_DECISION == "split_prepost_period":
            if pre_post_cp not in {"pre_cp", "post_cp"}:
                raise ValueError(
                    "Taxi PCA path resolution requires pre_post_cp='pre_cp' or 'post_cp' "
                    "when split pre/post handling is active."
                )
            return TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD[pre_post_cp]
        return PCA_80_FINAL_PATHS_BY_SET[feature_set_name]

    if representation_name == "umap_pca_to_umap_10d":
        return UMAP_PCA_TO_UMAP_10D_FINAL_PATHS_BY_SET[feature_set_name]

    raise ValueError(f"Unsupported representation_name: {representation_name}")

def get_numeric_representation_columns(parquet_path: Path) -> list[str]:
    schema_df = pd.DataFrame(
        {
            "column_name": load_parquet_columns(parquet_path)
        }
    )

    # We only want learned representation columns, never ids or metadata.
    excluded_columns = set(DEFAULT_METADATA_COLUMNS)

    numeric_prefixes = ("PC", "umap_")

    selected_columns = [
        column_name
        for column_name in schema_df["column_name"].tolist()
        if column_name not in excluded_columns
        and str(column_name).startswith(numeric_prefixes)
    ]

    return selected_columns

def load_representation_rows(
    feature_set_name: str,
    representation_name: str,
    *,
    selected_columns: list[str] | None = None,
    pre_post_cp: str | None = None,
) -> pd.DataFrame:
    representation_path = get_representation_path(
        feature_set_name,
        representation_name,
        pre_post_cp=pre_post_cp,
    )

    if selected_columns is None:
        selected_columns = get_numeric_representation_columns(representation_path)

    select_columns = [MODEL_ID_COLUMN, *selected_columns]

    with duckdb.connect() as con:
        query = f"""
        SELECT {", ".join(duckdb_identifier(column_name) for column_name in select_columns)}
        FROM read_parquet('{sql_path(representation_path)}')
        """
        return con.execute(query).fetchdf()

def load_calibration_metadata(feature_set_name: str) -> pd.DataFrame:
    metadata_path = ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name]

    with duckdb.connect() as con:
        query = f"""
        SELECT {", ".join(duckdb_identifier(column_name) for column_name in DEFAULT_METADATA_COLUMNS)}
        FROM read_parquet('{sql_path(metadata_path)}')
        """
        return con.execute(query).fetchdf()

def get_final_pca_representation_df(
    feature_set_name: str,
    metadata_df: pd.DataFrame,
) -> tuple[pd.DataFrame, list[str]]:
    """
    Build the row-aligned PCA frame for downstream DBSCAN review blocks.
    Taxi remains the special case because PCA was split pre/post CP.
    """
    if feature_set_name == "taxi" and TAXI_PCA_HANDLING_DECISION == "split_prepost_period":
        pre_metadata_df = metadata_df.loc[
            metadata_df[PRE_POST_CP_COLUMN].eq("pre_cp")
        ].copy()
        post_metadata_df = metadata_df.loc[
            metadata_df[PRE_POST_CP_COLUMN].eq("post_cp")
        ].copy()

        pre_pca_df = load_representation_rows(
            feature_set_name="taxi",
            representation_name="pca_80pct",
            pre_post_cp="pre_cp",
        )
        post_pca_df = load_representation_rows(
            feature_set_name="taxi",
            representation_name="pca_80pct",
            pre_post_cp="post_cp",
        )

        representation_columns = [
            column_name
            for column_name in pre_pca_df.columns
            if column_name != MODEL_ID_COLUMN
        ]

        combined_pca_df = pd.concat(
            [
                pre_metadata_df[[MODEL_ID_COLUMN]].merge(
                    pre_pca_df,
                    on=MODEL_ID_COLUMN,
                    how="inner",
                ),
                post_metadata_df[[MODEL_ID_COLUMN]].merge(
                    post_pca_df,
                    on=MODEL_ID_COLUMN,
                    how="inner",
                ),
            ],
            ignore_index=True,
        )
        return combined_pca_df, representation_columns

    pca_df = load_representation_rows(
        feature_set_name=feature_set_name,
        representation_name="pca_80pct",
    )
    representation_columns = [
        column_name
        for column_name in pca_df.columns
        if column_name != MODEL_ID_COLUMN
    ]
    return pca_df, representation_columns

def load_full_row_metadata(feature_set_name: str) -> pd.DataFrame:
    metadata_path = ROW_METADATA_PATHS_BY_SET[feature_set_name]

    with duckdb.connect() as con:
        query = f"""
        SELECT {", ".join(duckdb_identifier(column_name) for column_name in DEFAULT_METADATA_COLUMNS)}
        FROM read_parquet('{sql_path(metadata_path)}')
        """
        return con.execute(query).fetchdf()

This small helper block loads the retained final DBSCAN setting table as soon as the notebook starts if it already exists on disk\. That makes the selected modality\-level DBSCAN configurations available to later sections even when we rerun only part of the notebook, while still allowing the formal setting\-selection section to rebuild and rewrite the canonical table when needed\.

In [5]:
# ---------------------------------------------------------------------
# Load retained final DBSCAN settings if they already exist
# ---------------------------------------------------------------------

if DBSCAN_SELECTED_CONFIGURATION_PATH.exists():
    dbscan_final_pca_settings_df = pd.read_csv(DBSCAN_SELECTED_CONFIGURATION_PATH)

    numeric_setting_cols = [
        "min_samples",
        "eps_value",
        "mean_neighbor_jaccard",
        "mean_noise_delta",
        "mean_cluster_delta",
        "mean_largest_cluster_share_delta",
        "weighted_noise_share",
        "weighted_cluster_count",
        "weighted_largest_cluster_share",
    ]
    for col in numeric_setting_cols:
        if col in dbscan_final_pca_settings_df.columns:
            dbscan_final_pca_settings_df[col] = pd.to_numeric(
                dbscan_final_pca_settings_df[col],
                errors="coerce",
            )

    print(
        f"Loaded retained DBSCAN final settings from "
        f"{DBSCAN_SELECTED_CONFIGURATION_PATH.name}."
    )
else:
    print("No retained DBSCAN final settings file found yet; it will be created later in 3.3.2.4.")

Loaded retained DBSCAN final settings from selected_dbscan_configurations.csv.


## 3\.3\.2\.1 Load Shared Anomaly Framework Assets

Let’s start by loading the framework that 3\.3\.1 already settled\. This notebook inherits the shared calibration samples, the downstream PCA and UMAP representation branches, the selected comparison\-universe defaults, and the aligned metadata layer that DBSCAN will need for both calibration and interpretation\.

This section confirms that those inherited assets are complete, aligned, and in the right format so the later DBSCAN sections can focus directly on density behavior and parameter calibration\.

This first block loads the compact framework tables exported by 3\.3\.1\. It gives us one place to verify the inherited defaults before we start reading larger representation and sample assets\.

In [6]:
# ---------------------------------------------------------------------
# Load shared anomaly framework export tables
# ---------------------------------------------------------------------

shared_representation_defaults_df = load_shared_framework_table(
    "shared_representation_defaults"
)
shared_calibration_defaults_df = load_shared_framework_table(
    "shared_calibration_defaults"
)
shared_comparison_universe_defaults_df = load_shared_framework_table(
    "shared_comparison_universe_defaults"
)
shared_framework_interpretation_defaults_df = load_shared_framework_table(
    "shared_framework_interpretation_defaults"
)
shared_framework_asset_manifest_df = load_shared_framework_table(
    "shared_framework_asset_manifest"
)

framework_table_inventory_df = pd.DataFrame(
    [
        {
            "table_name": "shared_representation_defaults",
            "rows": len(shared_representation_defaults_df),
            "columns": len(shared_representation_defaults_df.columns),
        },
        {
            "table_name": "shared_calibration_defaults",
            "rows": len(shared_calibration_defaults_df),
            "columns": len(shared_calibration_defaults_df.columns),
        },
        {
            "table_name": "shared_comparison_universe_defaults",
            "rows": len(shared_comparison_universe_defaults_df),
            "columns": len(shared_comparison_universe_defaults_df.columns),
        },
        {
            "table_name": "shared_framework_interpretation_defaults",
            "rows": len(shared_framework_interpretation_defaults_df),
            "columns": len(shared_framework_interpretation_defaults_df.columns),
        },
        {
            "table_name": "shared_framework_asset_manifest",
            "rows": len(shared_framework_asset_manifest_df),
            "columns": len(shared_framework_asset_manifest_df.columns),
        },
    ]
)

display(framework_table_inventory_df)

,table_name,rows,columns
0,shared_representation_defaults,15,7
1,shared_calibration_defaults,5,8
2,shared_comparison_universe_defaults,6,6
3,shared_framework_interpretation_defaults,5,10
4,shared_framework_asset_manifest,8,4


Findings\. The five shared framework tables from 3\.3\.1 loaded successfully and are ready to serve as the inherited defaults for 3\.3\.2\.

Now let’s verify that the actual downstream assets referenced by the framework are present for every modality\. This is the basic existence check before we move on to row alignment and format validation\.

In [7]:
# ---------------------------------------------------------------------
# Validate shared framework asset availability
# ---------------------------------------------------------------------

framework_asset_availability_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    calibration_id_exists = ANOMALY_CALIBRATION_ID_PATHS_BY_SET[feature_set_name].exists()
    calibration_metadata_exists = ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name].exists()
    umap_exists = UMAP_PCA_TO_UMAP_10D_FINAL_PATHS_BY_SET[feature_set_name].exists()

    if feature_set_name == "taxi" and TAXI_PCA_HANDLING_DECISION == "split_prepost_period":
        pca_exists = (
            TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD["pre_cp"].exists()
            and TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD["post_cp"].exists()
        )
        pca_scope = "split_prepost_period"
    else:
        pca_exists = PCA_80_FINAL_PATHS_BY_SET[feature_set_name].exists()
        pca_scope = "shared_full_period"

    framework_asset_availability_records.append(
        {
            "feature_set": feature_set_name,
            "calibration_ids_exist": calibration_id_exists,
            "calibration_metadata_exist": calibration_metadata_exists,
            "pca_asset_scope": pca_scope,
            "pca_assets_exist": pca_exists,
            "umap_assets_exist": umap_exists,
            "status": (
                "pass"
                if calibration_id_exists
                and calibration_metadata_exists
                and pca_exists
                and umap_exists
                else "review"
            ),
        }
    )

framework_asset_availability_df = pd.DataFrame(
    framework_asset_availability_records
)

display(framework_asset_availability_df)

assert framework_asset_availability_df["status"].eq("pass").all(), (
    "One or more shared anomaly framework assets is missing."
)

,feature_set,calibration_ids_exist,calibration_metadata_exist,pca_asset_scope,pca_assets_exist,umap_assets_exist,status
0,bus,True,True,shared_full_period,True,True,pass
1,subway,True,True,shared_full_period,True,True,pass
2,taxi,True,True,split_prepost_period,True,True,pass
3,fhvhv,True,True,shared_full_period,True,True,pass
4,multimodal,True,True,shared_full_period,True,True,pass


Findings\. All inherited calibration, PCA, and UMAP assets are present, including Taxi’s split pre/post PCA branch\.

With the files present, let’s check that the representation branches and the shared calibration metadata actually line up on rows\. This is the alignment check that protects the rest of the notebook from silent join drift\.

In [8]:
# ---------------------------------------------------------------------
# Validate row alignment between calibration metadata and representations
# ---------------------------------------------------------------------

framework_alignment_records = []
calibration_metadata_schema_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    metadata_path = ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name]
    calibration_id_path = ANOMALY_CALIBRATION_ID_PATHS_BY_SET[feature_set_name]

    with duckdb.connect() as con:
        metadata_schema_df = con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{sql_path(metadata_path)}')"
        ).fetchdf()

        metadata_columns = metadata_schema_df["column_name"].tolist()

        calibration_metadata_schema_records.append(
            {
                "feature_set": feature_set_name,
                "metadata_columns": ", ".join(metadata_columns),
                "has_modeling_row_id": MODEL_ID_COLUMN in metadata_columns,
            }
        )

        metadata_ids = con.execute(
            f"""
            SELECT {duckdb_identifier(MODEL_ID_COLUMN)}
            FROM read_parquet('{sql_path(metadata_path)}')
            """
        ).fetchdf()

        calibration_ids = con.execute(
            f"""
            SELECT {duckdb_identifier(MODEL_ID_COLUMN)}
            FROM read_parquet('{sql_path(calibration_id_path)}')
            """
        ).fetchdf()

    # Shared UMAP branch: restrict to sampled ids before validating counts.
    umap_df = load_representation_rows(
        feature_set_name=feature_set_name,
        representation_name="umap_pca_to_umap_10d",
    )
    umap_sample_ids = calibration_ids.merge(
        umap_df[[MODEL_ID_COLUMN]],
        on=MODEL_ID_COLUMN,
        how="inner",
    )

    if feature_set_name == "taxi" and TAXI_PCA_HANDLING_DECISION == "split_prepost_period":
        with duckdb.connect() as con:
            pre_sample_ids = con.execute(
                f"""
                SELECT {duckdb_identifier(MODEL_ID_COLUMN)}
                FROM read_parquet('{sql_path(calibration_id_path)}')
                WHERE {duckdb_identifier(PRE_POST_CP_COLUMN)} = 'pre_cp'
                """
            ).fetchdf()

            post_sample_ids = con.execute(
                f"""
                SELECT {duckdb_identifier(MODEL_ID_COLUMN)}
                FROM read_parquet('{sql_path(calibration_id_path)}')
                WHERE {duckdb_identifier(PRE_POST_CP_COLUMN)} = 'post_cp'
                """
            ).fetchdf()

        pre_pca_df = load_representation_rows(
            feature_set_name="taxi",
            representation_name="pca_80pct",
            pre_post_cp="pre_cp",
        )
        post_pca_df = load_representation_rows(
            feature_set_name="taxi",
            representation_name="pca_80pct",
            pre_post_cp="post_cp",
        )

        pca_sample_ids = pd.concat(
            [
                pre_sample_ids.merge(
                    pre_pca_df[[MODEL_ID_COLUMN]],
                    on=MODEL_ID_COLUMN,
                    how="inner",
                ),
                post_sample_ids.merge(
                    post_pca_df[[MODEL_ID_COLUMN]],
                    on=MODEL_ID_COLUMN,
                    how="inner",
                ),
            ],
            ignore_index=True,
        )

        missing_from_pca = int(len(calibration_ids) - len(pca_sample_ids))

    else:
        pca_df = load_representation_rows(
            feature_set_name=feature_set_name,
            representation_name="pca_80pct",
        )
        pca_sample_ids = calibration_ids.merge(
            pca_df[[MODEL_ID_COLUMN]],
            on=MODEL_ID_COLUMN,
            how="inner",
        )

        missing_from_pca = int(len(calibration_ids) - len(pca_sample_ids))

    missing_from_umap = int(len(calibration_ids) - len(umap_sample_ids))

    framework_alignment_records.append(
        {
            "feature_set": feature_set_name,
            "metadata_rows": int(len(metadata_ids)),
            "calibration_id_rows": int(len(calibration_ids)),
            "pca_sample_rows": int(len(pca_sample_ids)),
            "umap_sample_rows": int(len(umap_sample_ids)),
            "missing_from_pca": missing_from_pca,
            "missing_from_umap": missing_from_umap,
            "status": (
                "pass"
                if MODEL_ID_COLUMN in metadata_columns
                and len(metadata_ids) == len(calibration_ids)
                and len(pca_sample_ids) == len(calibration_ids)
                and len(umap_sample_ids) == len(calibration_ids)
                and missing_from_pca == 0
                and missing_from_umap == 0
                else "review"
            ),
        }
    )

calibration_metadata_schema_df = pd.DataFrame(calibration_metadata_schema_records)
framework_alignment_df = pd.DataFrame(framework_alignment_records)

display(calibration_metadata_schema_df)
display(framework_alignment_df)

assert framework_alignment_df["status"].eq("pass").all(), (
    "One or more inherited representation branches is not aligned to the shared calibration sample."
)

,feature_set,metadata_columns,has_modeling_row_id
0,bus,"modeling_row_id, taxi_zone_id, zone, borough, ...",True
1,subway,"modeling_row_id, taxi_zone_id, zone, borough, ...",True
2,taxi,"modeling_row_id, taxi_zone_id, zone, borough, ...",True
3,fhvhv,"modeling_row_id, taxi_zone_id, zone, borough, ...",True
4,multimodal,"modeling_row_id, taxi_zone_id, zone, borough, ...",True


,feature_set,metadata_rows,calibration_id_rows,pca_sample_rows,umap_sample_rows,missing_from_pca,missing_from_umap,status
0,bus,50000,50000,50000,50000,0,0,pass
1,subway,50000,50000,50000,50000,0,0,pass
2,taxi,50000,50000,50000,50000,0,0,pass
3,fhvhv,50000,50000,50000,50000,0,0,pass
4,multimodal,50000,50000,50000,50000,0,0,pass


Findings\. The shared calibration metadata aligns cleanly with the inherited PCA and UMAP branches for all five modalities, with no missing or extra sample rows\.

> 📝 Note\. This notebook uses the shared 50,000\-row anomaly calibration samples created in 3\.3\.1\.2 as its common row set\. Those samples preserve temporal\_bucket and pre\_post\_cp coverage and stay aligned across the downstream representation branches\. The geography\-aware decision made later in 3\.3\.1\.3 changes how rows are compared within that shared sample — using temporal\_bucket \+ policy\_geography\_class as the scoring baseline — rather than requiring a different calibration panel\.

The shared calibration panel was designed to preserve temporal\-bucket and pre/post coverage, while the downstream DBSCAN baseline later became temporal\_bucket \+ policy\_geography\_class\. Before moving on, it helps to check whether the inherited 50,000\-row sample still tracks that finer joint distribution closely enough to support the geography\-aware baseline without rebuilding the panel itself\.

In [9]:
# ---------------------------------------------------------------------
# Validate temporal-bucket x policy-geography distribution in the
# shared calibration panel
# ---------------------------------------------------------------------

TEMPORAL_GEO_SAMPLE_DELTA_REVIEW_THRESHOLD = 0.015

temporal_geo_distribution_summary_records = []
temporal_geo_distribution_detail_frames = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    source_metadata_path = ROW_METADATA_PATHS_BY_SET[feature_set_name]
    sample_metadata_path = ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name]

    with duckdb.connect() as con:
        source_distribution_df = con.execute(
            f"""
            SELECT
                {duckdb_identifier(TEMPORAL_BUCKET_COLUMN)} AS temporal_bucket,
                {duckdb_identifier(POLICY_GEOGRAPHY_COLUMN)} AS policy_geography_class,
                COUNT(*) AS source_rows
            FROM read_parquet('{sql_path(source_metadata_path)}')
            GROUP BY 1, 2
            ORDER BY 1, 2
            """
        ).fetchdf()

        sample_distribution_df = con.execute(
            f"""
            SELECT
                {duckdb_identifier(TEMPORAL_BUCKET_COLUMN)} AS temporal_bucket,
                {duckdb_identifier(POLICY_GEOGRAPHY_COLUMN)} AS policy_geography_class,
                COUNT(*) AS sample_rows
            FROM read_parquet('{sql_path(sample_metadata_path)}')
            GROUP BY 1, 2
            ORDER BY 1, 2
            """
        ).fetchdf()

    source_total_rows = int(source_distribution_df["source_rows"].sum())
    sample_total_rows = int(sample_distribution_df["sample_rows"].sum())

    merged_distribution_df = (
        source_distribution_df.merge(
            sample_distribution_df,
            on=["temporal_bucket", "policy_geography_class"],
            how="outer",
        )
        .fillna({"source_rows": 0, "sample_rows": 0})
        .copy()
    )

    merged_distribution_df["source_rows"] = merged_distribution_df["source_rows"].astype(int)
    merged_distribution_df["sample_rows"] = merged_distribution_df["sample_rows"].astype(int)

    merged_distribution_df["source_row_share"] = (
        merged_distribution_df["source_rows"] / source_total_rows
    )
    merged_distribution_df["sample_row_share"] = (
        merged_distribution_df["sample_rows"] / sample_total_rows
    )
    merged_distribution_df["share_delta"] = (
        merged_distribution_df["sample_row_share"]
        - merged_distribution_df["source_row_share"]
    )
    merged_distribution_df["abs_share_delta"] = (
        merged_distribution_df["share_delta"].abs()
    )
    merged_distribution_df["feature_set"] = feature_set_name

    temporal_geo_distribution_detail_frames.append(
        merged_distribution_df[
            [
                "feature_set",
                "temporal_bucket",
                "policy_geography_class",
                "source_rows",
                "source_row_share",
                "sample_rows",
                "sample_row_share",
                "share_delta",
                "abs_share_delta",
            ]
        ].sort_values(
            ["feature_set", "temporal_bucket", "policy_geography_class"]
        )
    )

    temporal_geo_distribution_summary_records.append(
        {
            "feature_set": feature_set_name,
            "joint_strata": int(len(merged_distribution_df)),
            "max_abs_share_delta": float(merged_distribution_df["abs_share_delta"].max()),
            "mean_abs_share_delta": float(merged_distribution_df["abs_share_delta"].mean()),
            "review_threshold": TEMPORAL_GEO_SAMPLE_DELTA_REVIEW_THRESHOLD,
            "status": (
                "pass"
                if merged_distribution_df["abs_share_delta"].max()
                <= TEMPORAL_GEO_SAMPLE_DELTA_REVIEW_THRESHOLD
                else "review"
            ),
        }
    )

temporal_geo_distribution_summary_df = pd.DataFrame(
    temporal_geo_distribution_summary_records
)

temporal_geo_distribution_detail_df = pd.concat(
    temporal_geo_distribution_detail_frames,
    ignore_index=True,
)

display(temporal_geo_distribution_summary_df)
display(temporal_geo_distribution_detail_df)

assert temporal_geo_distribution_summary_df["status"].eq("pass").all(), (
    "The shared calibration panel differs more than expected from the full "
    "temporal-bucket x policy-geography distribution."
)

,feature_set,joint_strata,max_abs_share_delta,mean_abs_share_delta,review_threshold,status
0,bus,30,0.000911,0.000409,0.015,pass
1,subway,30,0.001119,0.000548,0.015,pass
2,taxi,30,0.000734,0.000463,0.015,pass
3,fhvhv,30,0.000734,0.000463,0.015,pass
4,multimodal,30,0.000734,0.000463,0.015,pass


,feature_set,temporal_bucket,policy_geography_class,source_rows,source_row_share,sample_rows,sample_row_share,share_delta,abs_share_delta
0,bus,weekday_am_peak,cbd,32148,0.021826,1080,0.02160,-0.000226,0.000226
1,bus,weekday_am_peak,gateway_or_adjacent,15228,0.010338,487,0.00974,-0.000598,0.000598
2,bus,weekday_am_peak,outside,164970,0.112000,5640,0.11280,0.000800,0.000800
3,bus,weekday_evening,cbd,31302,0.021251,1097,0.02194,0.000689,0.000689
4,bus,weekday_evening,gateway_or_adjacent,14382,0.009764,471,0.00942,-0.000344,0.000344
...,...,...,...,...,...,...,...,...,...
145,multimodal,weekend_overnight,gateway_or_adjacent,6441,0.004178,214,0.00428,0.000102,0.000102
146,multimodal,weekend_overnight,outside,68139,0.044194,2237,0.04474,0.000546,0.000546
147,multimodal,weekend_pm_peak,cbd,13560,0.008795,407,0.00814,-0.000655,0.000655
148,multimodal,weekend_pm_peak,gateway_or_adjacent,6441,0.004178,214,0.00428,0.000102,0.000102


The last thing we should check here is format\. We want to make sure the downstream representation tables contain only the numeric learned coordinates we actually expect DBSCAN to use, while metadata stays in the aligned sample metadata layer\.

In [10]:
# ---------------------------------------------------------------------
# Validate downstream representation format contract
# ---------------------------------------------------------------------

representation_format_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    # --------------------------------------------------------------
    # Validate PCA branch
    # --------------------------------------------------------------
    if feature_set_name == "taxi" and TAXI_PCA_HANDLING_DECISION == "split_prepost_period":
        pca_path_items = [
            ("pre_cp", TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD["pre_cp"]),
            ("post_cp", TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD["post_cp"]),
        ]
    else:
        pca_path_items = [("shared", PCA_80_FINAL_PATHS_BY_SET[feature_set_name])]

    for period_scope, pca_path in pca_path_items:
        pca_columns = load_parquet_columns(pca_path)

        learned_pca_columns = [
            column_name
            for column_name in pca_columns
            if str(column_name).startswith("PC")
        ]

        disallowed_pca_columns = [
            column_name
            for column_name in pca_columns
            if column_name not in {MODEL_ID_COLUMN, *learned_pca_columns}
        ]

        representation_format_records.append(
            {
                "feature_set": feature_set_name,
                "representation_name": "pca_80pct",
                "period_scope": period_scope,
                "path": str(pca_path),
                "learned_column_count": len(learned_pca_columns),
                "has_modeling_row_id": MODEL_ID_COLUMN in pca_columns,
                "disallowed_extra_columns": (
                    ", ".join(disallowed_pca_columns)
                    if disallowed_pca_columns
                    else "none"
                ),
                "status": (
                    "pass"
                    if MODEL_ID_COLUMN in pca_columns
                    and len(learned_pca_columns) > 0
                    and len(disallowed_pca_columns) == 0
                    else "review"
                ),
            }
        )

    # --------------------------------------------------------------
    # Validate UMAP branch
    # --------------------------------------------------------------
    umap_path = UMAP_PCA_TO_UMAP_10D_FINAL_PATHS_BY_SET[feature_set_name]
    umap_columns = load_parquet_columns(umap_path)

    learned_umap_columns = [
        column_name
        for column_name in umap_columns
        if str(column_name).startswith("umap_")
    ]

    disallowed_umap_columns = [
        column_name
        for column_name in umap_columns
        if column_name not in {MODEL_ID_COLUMN, *learned_umap_columns}
    ]

    representation_format_records.append(
        {
            "feature_set": feature_set_name,
            "representation_name": "umap_pca_to_umap_10d",
            "period_scope": "shared",
            "path": str(umap_path),
            "learned_column_count": len(learned_umap_columns),
            "has_modeling_row_id": MODEL_ID_COLUMN in umap_columns,
            "disallowed_extra_columns": (
                ", ".join(disallowed_umap_columns)
                if disallowed_umap_columns
                else "none"
            ),
            "status": (
                "pass"
                if MODEL_ID_COLUMN in umap_columns
                and len(learned_umap_columns) > 0
                and len(disallowed_umap_columns) == 0
                else "review"
            ),
        }
    )

representation_format_df = pd.DataFrame(representation_format_records)

display(representation_format_df)

assert representation_format_df["status"].eq("pass").all(), (
    "One or more inherited representation assets does not match the downstream format contract."
)

,feature_set,representation_name,period_scope,path,learned_column_count,has_modeling_row_id,disallowed_extra_columns,status
0,bus,pca_80pct,shared,/datasets/_deepnote_work/pipeline_data/3.2.1.f...,14,True,none,pass
1,bus,umap_pca_to_umap_10d,shared,/datasets/_deepnote_work/pipeline_data/3.2.4.f...,10,True,none,pass
2,subway,pca_80pct,shared,/datasets/_deepnote_work/pipeline_data/3.2.1.f...,11,True,none,pass
3,subway,umap_pca_to_umap_10d,shared,/datasets/_deepnote_work/pipeline_data/3.2.4.f...,10,True,none,pass
4,taxi,pca_80pct,pre_cp,/datasets/_deepnote_work/pipeline_data/3.2.2.f...,15,True,none,pass
5,taxi,pca_80pct,post_cp,/datasets/_deepnote_work/pipeline_data/3.2.2.f...,15,True,none,pass
6,taxi,umap_pca_to_umap_10d,shared,/datasets/_deepnote_work/pipeline_data/3.2.4.f...,10,True,none,pass
7,fhvhv,pca_80pct,shared,/datasets/_deepnote_work/pipeline_data/3.2.1.f...,13,True,none,pass
8,fhvhv,umap_pca_to_umap_10d,shared,/datasets/_deepnote_work/pipeline_data/3.2.4.f...,10,True,none,pass
9,multimodal,pca_80pct,shared,/datasets/_deepnote_work/pipeline_data/3.2.1.f...,46,True,none,pass


Findings\. The downstream PCA and UMAP assets satisfy the expected representation\-table contract: each table contains modeling\_row\_id plus learned numeric coordinates only, with no stray metadata columns embedded in the representation surface\.

Now that the assets have passed the basic checks, let’s put the inherited framework in one compact summary table so the rest of the notebook can refer back to it easily\.

In [11]:
# ---------------------------------------------------------------------
# Summarize inherited DBSCAN framework defaults
# ---------------------------------------------------------------------

dbscan_framework_summary_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    if feature_set_name == "taxi":
        pca_handling = TAXI_PCA_HANDLING_DECISION
    else:
        pca_handling = "shared_full_period"

    dbscan_framework_summary_records.append(
        {
            "feature_set": feature_set_name,
            "representations_advancing": ", ".join(DBSCAN_REPRESENTATION_NAMES),
            "shared_comparison_universe": SHARED_DBSCAN_COMPARISON_UNIVERSE,
            "shared_grouping_columns": ", ".join(SHARED_DBSCAN_GROUPING_COLUMNS),
            "calibration_sample_rows": ANOMALY_CALIBRATION_SAMPLE_ROWS,
            "taxi_pca_handling": pca_handling,
            "configuration_scope": DBSCAN_CONFIGURATION_SCOPE,
        }
    )

dbscan_framework_summary_df = pd.DataFrame(dbscan_framework_summary_records)

display(dbscan_framework_summary_df)

,feature_set,representations_advancing,shared_comparison_universe,shared_grouping_columns,calibration_sample_rows,taxi_pca_handling,configuration_scope
0,bus,"pca_80pct, umap_pca_to_umap_10d",same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",50000,shared_full_period,per_modality_x_representation
1,subway,"pca_80pct, umap_pca_to_umap_10d",same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",50000,shared_full_period,per_modality_x_representation
2,taxi,"pca_80pct, umap_pca_to_umap_10d",same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",50000,split_prepost_period,per_modality_x_representation
3,fhvhv,"pca_80pct, umap_pca_to_umap_10d",same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",50000,shared_full_period,per_modality_x_representation
4,multimodal,"pca_80pct, umap_pca_to_umap_10d",same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",50000,shared_full_period,per_modality_x_representation


Findings\. 3\.3\.2 will calibrate DBSCAN separately for each modality and for each downstream representation branch, using the shared temporal\_bucket \+ policy\_geography\_class comparison universe and the inherited 50,000\-row calibration samples\.

### Section Summary

The inherited anomaly framework is now loaded and validated\. DBSCAN calibration can proceed using the agreed downstream setup: PCA and UMAP branches for all five modalities, the shared temporal\-plus\-policy\-geography comparison universe, the shared calibration samples, and Taxi’s split pre/post PCA handling\.

## 3\.3\.2\.2 Evaluate Local Density Structure

This section checks the core assumption that DBSCAN needs in order to behave sensibly: observations within a comparison universe should have reasonably comparable local density\. Using the shared calibration samples and the inherited temporal\_bucket \+ policy\_geography\_class baseline, we’ll profile local\-neighbor distance behavior across Bus, Subway, Taxi, FHVHV, and Multimodal under both PCA and UMAP\.

The purpose here is to understand the density landscape before parameter tuning begins\. If local density still varies too sharply within the selected baseline groups, then DBSCAN may struggle no matter how carefully we tune eps and min\_samples\. If the density structure looks reasonably regular, we can move into calibration with a much clearer sense of where DBSCAN is likely to behave well and where it may need caution\.

This first block makes the density\-profiling setup explicit in one place\. The inherited comparison universe, active representation branches, and output targets are already defined at the notebook level; here we just summarize the setup we’re about to use\.

In [12]:
# ---------------------------------------------------------------------
# Define density-profiling setup and output paths
# ---------------------------------------------------------------------

density_profile_setup_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    for representation_name in DBSCAN_DENSITY_REPRESENTATION_NAMES:
        density_profile_setup_records.append(
            {
                "feature_set": feature_set_name,
                "representation_name": representation_name,
                "representation_role": REPRESENTATION_ROLE_BY_NAME[representation_name],
                "comparison_universe": SHARED_DBSCAN_COMPARISON_UNIVERSE,
                "grouping_columns": ", ".join(DBSCAN_DENSITY_GROUPING_COLUMNS),
                "calibration_sample_rows": ANOMALY_CALIBRATION_SAMPLE_ROWS,
            }
        )

density_profile_setup_df = pd.DataFrame(density_profile_setup_records)

display(density_profile_setup_df)

,feature_set,representation_name,representation_role,comparison_universe,grouping_columns,calibration_sample_rows
0,bus,pca_80pct,primary_linear_representation,same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",50000
1,bus,umap_pca_to_umap_10d,nonlinear_complement_representation,same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",50000
2,subway,pca_80pct,primary_linear_representation,same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",50000
3,subway,umap_pca_to_umap_10d,nonlinear_complement_representation,same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",50000
4,taxi,pca_80pct,primary_linear_representation,same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",50000
5,taxi,umap_pca_to_umap_10d,nonlinear_complement_representation,same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",50000
6,fhvhv,pca_80pct,primary_linear_representation,same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",50000
7,fhvhv,umap_pca_to_umap_10d,nonlinear_complement_representation,same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",50000
8,multimodal,pca_80pct,primary_linear_representation,same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",50000
9,multimodal,umap_pca_to_umap_10d,nonlinear_complement_representation,same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",50000


Findings\. The local\-density diagnostics will use the shared 50,000\-row calibration sample, the inherited temporal\_bucket \+ policy\_geography\_class baseline, and separate PCA and UMAP branches for each modality\.

Now let’s materialize the working density\-profiling panel itself\. This gives us one aligned table per modality and representation branch, with the shared comparison\-group key attached for later k\-nearest\-neighbor profiling\.

In [13]:
# ---------------------------------------------------------------------
# Materialize the density-profiling panel
# ---------------------------------------------------------------------

density_profile_panel_frames = []
density_group_count_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    metadata_df = pd.read_parquet(
        ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name]
    ).copy()

    # Build the inherited downstream comparison-universe key once on metadata,
    # then reuse it across both representation branches.
    metadata_df[DBSCAN_DENSITY_GROUP_KEY_COLUMN] = (
        metadata_df[TEMPORAL_BUCKET_COLUMN].astype("string")
        + " | "
        + metadata_df[POLICY_GEOGRAPHY_COLUMN].astype("string")
    )

    for representation_name in DBSCAN_DENSITY_REPRESENTATION_NAMES:
        if representation_name == "pca_80pct":
            if feature_set_name == "taxi" and TAXI_PCA_HANDLING_DECISION == "split_prepost_period":
                pre_cp_df = load_representation_rows(
                    feature_set_name="taxi",
                    representation_name="pca_80pct",
                    pre_post_cp="pre_cp",
                )
                post_cp_df = load_representation_rows(
                    feature_set_name="taxi",
                    representation_name="pca_80pct",
                    pre_post_cp="post_cp",
                )
                representation_df = pd.concat(
                    [pre_cp_df, post_cp_df],
                    ignore_index=True,
                )
            else:
                representation_df = load_representation_rows(
                    feature_set_name=feature_set_name,
                    representation_name="pca_80pct",
                )
        else:
            representation_df = load_representation_rows(
                feature_set_name=feature_set_name,
                representation_name="umap_pca_to_umap_10d",
            )

        merged_df = metadata_df.merge(
            representation_df,
            on=MODEL_ID_COLUMN,
            how="inner",
        ).copy()

        representation_columns = [
            column_name
            for column_name in merged_df.columns
            if column_name not in DEFAULT_METADATA_COLUMNS
            and column_name != DBSCAN_DENSITY_GROUP_KEY_COLUMN
        ]

        merged_df["feature_set"] = feature_set_name
        merged_df["representation_name"] = representation_name

        density_profile_panel_frames.append(
            merged_df[
                [
                    "feature_set",
                    "representation_name",
                    MODEL_ID_COLUMN,
                    TEMPORAL_BUCKET_COLUMN,
                    POLICY_GEOGRAPHY_COLUMN,
                    DBSCAN_DENSITY_GROUP_KEY_COLUMN,
                    *representation_columns,
                ]
            ]
        )

        group_count_df = (
            merged_df.groupby(DBSCAN_DENSITY_GROUP_KEY_COLUMN, dropna=False)
            .size()
            .reset_index(name="group_rows")
            .sort_values("group_rows", ascending=False)
            .reset_index(drop=True)
        )

        density_group_count_records.append(
            {
                "feature_set": feature_set_name,
                "representation_name": representation_name,
                "rows_in_panel": int(len(merged_df)),
                "representation_columns": len(representation_columns),
                "group_count": int(len(group_count_df)),
                "min_group_rows": int(group_count_df["group_rows"].min()),
                "median_group_rows": float(group_count_df["group_rows"].median()),
                "max_group_rows": int(group_count_df["group_rows"].max()),
                "status": (
                    "pass"
                    if len(merged_df) == ANOMALY_CALIBRATION_SAMPLE_ROWS
                    and len(group_count_df) > 0
                    else "review"
                ),
            }
        )

density_profile_panel_df = pd.concat(
    density_profile_panel_frames,
    ignore_index=True,
)

density_group_counts_df = pd.DataFrame(density_group_count_records)

if should_rebuild(REBUILD_DBSCAN_DENSITY_DIAGNOSTICS, DBSCAN_DENSITY_PROFILE_PANEL_PATH):
    density_profile_panel_df.to_parquet(DBSCAN_DENSITY_PROFILE_PANEL_PATH, index=False)

if should_rebuild(REBUILD_DBSCAN_DENSITY_DIAGNOSTICS, DBSCAN_DENSITY_GROUP_COUNTS_PATH):
    density_group_counts_df.to_parquet(DBSCAN_DENSITY_GROUP_COUNTS_PATH, index=False)

display(density_group_counts_df)

assert density_group_counts_df["status"].eq("pass").all(), (
    "One or more density-profiling panels was not materialized as expected."
)

,feature_set,representation_name,rows_in_panel,representation_columns,group_count,min_group_rows,median_group_rows,max_group_rows,status
0,bus,pca_80pct,50000,18,30,190,811.5,5640,pass
1,bus,umap_pca_to_umap_10d,50000,14,30,190,811.5,5640,pass
2,subway,pca_80pct,50000,15,30,254,1017.0,5137,pass
3,subway,umap_pca_to_umap_10d,50000,14,30,254,1017.0,5137,pass
4,taxi,pca_80pct,50000,19,30,214,805.0,5532,pass
5,taxi,umap_pca_to_umap_10d,50000,14,30,214,805.0,5532,pass
6,fhvhv,pca_80pct,50000,17,30,214,805.0,5532,pass
7,fhvhv,umap_pca_to_umap_10d,50000,14,30,214,805.0,5532,pass
8,multimodal,pca_80pct,50000,50,30,214,805.0,5532,pass
9,multimodal,umap_pca_to_umap_10d,50000,14,30,214,805.0,5532,pass


Findings\. The density\-profiling panel was materialized successfully for every modality and both downstream representation branches\. Each branch preserves the full 50,000\-row calibration sample and the inherited temporal\_bucket \+ policy\_geography\_class grouping structure needed for later DBSCAN density diagnostics\.

Before moving into neighbor\-distance profiling, it helps to inspect how much data each modality\-representation branch contributes to each comparison group\. This tells us whether the inherited baseline gives DBSCAN enough local support to evaluate density behavior meaningfully\.

In [14]:
# ---------------------------------------------------------------------
# Inspect density-profiling group coverage
# ---------------------------------------------------------------------

density_group_coverage_df = (
    density_profile_panel_df.groupby(
        ["feature_set", "representation_name", DBSCAN_DENSITY_GROUP_KEY_COLUMN],
        dropna=False,
    )
    .size()
    .reset_index(name="group_rows")
    .sort_values(
        ["feature_set", "representation_name", DBSCAN_DENSITY_GROUP_KEY_COLUMN]
    )
    .reset_index(drop=True)
)

display(density_group_coverage_df)

,feature_set,representation_name,comparison_group_key,group_rows
0,bus,pca_80pct,weekday_am_peak | cbd,1080
1,bus,pca_80pct,weekday_am_peak | gateway_or_adjacent,487
2,bus,pca_80pct,weekday_am_peak | outside,5640
3,bus,pca_80pct,weekday_evening | cbd,1097
4,bus,pca_80pct,weekday_evening | gateway_or_adjacent,471
...,...,...,...,...
295,taxi,umap_pca_to_umap_10d,weekend_overnight | gateway_or_adjacent,214
296,taxi,umap_pca_to_umap_10d,weekend_overnight | outside,2237
297,taxi,umap_pca_to_umap_10d,weekend_pm_peak | cbd,407
298,taxi,umap_pca_to_umap_10d,weekend_pm_peak | gateway_or_adjacent,214


Findings\. The inherited geography\-aware comparison groups are present across every modality and both representation branches, so the local\-density analysis can proceed inside the same downstream baseline structure that later DBSCAN scoring will use\.

### Profile k\-nearest\-neighbor distance structure

Now that the density\-profiling panel is in place, we can inspect the local\-neighbor geometry that DBSCAN will actually see\. This is the most direct way to understand whether local density looks reasonably regular within the inherited comparison universe and where plausible eps regions may begin to emerge across modalities and representation branches\.

Before we profile local\-neighbor distances, let’s confirm that each modality\-representation branch is numerically usable for kNN calculations\. This check surfaces missing values at the representation\-column level so we can distinguish a genuine asset problem from a simple panel\-construction bug\.

In [15]:
# ---------------------------------------------------------------------
# Validate density-profiling panel for branch-specific numeric completeness
# ---------------------------------------------------------------------

density_profile_panel_df = pd.read_parquet(DBSCAN_DENSITY_PROFILE_PANEL_PATH)

density_profile_missingness_records = []
density_profile_missingness_detail_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    for representation_name in DBSCAN_DENSITY_REPRESENTATION_NAMES:
        branch_df = density_profile_panel_df.loc[
            density_profile_panel_df["feature_set"].eq(feature_set_name)
            & density_profile_panel_df["representation_name"].eq(representation_name)
        ].copy()

        # Resolve the exact expected representation columns from the inherited
        # source asset for this branch, rather than inferring from the combined
        # panel where other modalities may contribute extra PCA columns.
        if representation_name == "pca_80pct":
            if feature_set_name == "taxi" and TAXI_PCA_HANDLING_DECISION == "split_prepost_period":
                representation_columns = get_numeric_representation_columns(
                    TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD["pre_cp"]
                )
            else:
                representation_columns = get_numeric_representation_columns(
                    PCA_80_FINAL_PATHS_BY_SET[feature_set_name]
                )
        else:
            representation_columns = get_numeric_representation_columns(
                UMAP_PCA_TO_UMAP_10D_FINAL_PATHS_BY_SET[feature_set_name]
            )

        missing_cells_by_column = (
            branch_df[representation_columns].isna().sum().sort_values(ascending=False)
        )

        columns_with_missing = missing_cells_by_column.loc[
            missing_cells_by_column.gt(0)
        ]

        rows_with_any_missing = int(
            branch_df[representation_columns].isna().any(axis=1).sum()
        )
        total_missing_cells = int(branch_df[representation_columns].isna().sum().sum())

        density_profile_missingness_records.append(
            {
                "feature_set": feature_set_name,
                "representation_name": representation_name,
                "rows_in_panel": int(len(branch_df)),
                "representation_columns": len(representation_columns),
                "rows_with_any_missing": rows_with_any_missing,
                "total_missing_cells": total_missing_cells,
                "columns_with_missing": len(columns_with_missing),
                "status": "pass" if rows_with_any_missing == 0 else "review",
            }
        )

        for column_name, missing_count in columns_with_missing.items():
            density_profile_missingness_detail_records.append(
                {
                    "feature_set": feature_set_name,
                    "representation_name": representation_name,
                    "column_name": column_name,
                    "missing_rows": int(missing_count),
                }
            )

density_profile_missingness_df = pd.DataFrame(density_profile_missingness_records)
density_profile_missingness_detail_df = pd.DataFrame(density_profile_missingness_detail_records)

display(density_profile_missingness_df)

if not density_profile_missingness_detail_df.empty:
    display(
        density_profile_missingness_detail_df.sort_values(
            ["feature_set", "representation_name", "missing_rows"],
            ascending=[True, True, False],
        )
    )

assert density_profile_missingness_df["status"].eq("pass").all(), (
    "One or more density-profiling branches contains missing branch-specific representation values."
)

,feature_set,representation_name,rows_in_panel,representation_columns,rows_with_any_missing,total_missing_cells,columns_with_missing,status
0,bus,pca_80pct,50000,14,0,0,0,pass
1,bus,umap_pca_to_umap_10d,50000,10,0,0,0,pass
2,subway,pca_80pct,50000,11,0,0,0,pass
3,subway,umap_pca_to_umap_10d,50000,10,0,0,0,pass
4,taxi,pca_80pct,50000,15,0,0,0,pass
5,taxi,umap_pca_to_umap_10d,50000,10,0,0,0,pass
6,fhvhv,pca_80pct,50000,13,0,0,0,pass
7,fhvhv,umap_pca_to_umap_10d,50000,10,0,0,0,pass
8,multimodal,pca_80pct,50000,46,0,0,0,pass
9,multimodal,umap_pca_to_umap_10d,50000,10,0,0,0,pass


Findings\. The branch\-specific completeness check passed cleanly for every modality and both inherited representations: each pca\_80pct and umap\_pca\_to\_umap\_10d panel has the expected number of numeric representation columns, with zero missing rows and zero missing cells\. That confirms the density diagnostics are now operating on the intended branch\-specific feature spaces rather than on a mixed schema\.

Let’s compute group\-level k\-nearest\-neighbor distance summaries\. We’ll do this within each temporal\_bucket \+ policy\_geography\_class group, across both PCA and UMAP branches, and across several k values that line up with plausible min\_samples settings later in the notebook\.

In [16]:
# ---------------------------------------------------------------------
# Profile group-level k-nearest-neighbor distance structure
# ---------------------------------------------------------------------

density_profile_panel_df = pd.read_parquet(DBSCAN_DENSITY_PROFILE_PANEL_PATH)

dbscan_knn_profile_records = []
dbscan_knn_group_skip_records = []

MIN_ROWS_REQUIRED_FOR_KNN_PROFILE = max(DBSCAN_K_DISTANCE_K_VALUES) + 1

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    for representation_name in DBSCAN_DENSITY_REPRESENTATION_NAMES:
        branch_df = density_profile_panel_df.loc[
            density_profile_panel_df["feature_set"].eq(feature_set_name)
            & density_profile_panel_df["representation_name"].eq(representation_name)
        ].copy()

        # Resolve the exact representation columns for this modality branch
        # from the inherited source asset, not from the combined panel schema.
        if representation_name == "pca_80pct":
            if feature_set_name == "taxi" and TAXI_PCA_HANDLING_DECISION == "split_prepost_period":
                representation_columns = get_numeric_representation_columns(
                    TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD["pre_cp"]
                )
            else:
                representation_columns = get_numeric_representation_columns(
                    PCA_80_FINAL_PATHS_BY_SET[feature_set_name]
                )
        else:
            representation_columns = get_numeric_representation_columns(
                UMAP_PCA_TO_UMAP_10D_FINAL_PATHS_BY_SET[feature_set_name]
            )

        for comparison_group_key, group_df in branch_df.groupby(
            DBSCAN_DENSITY_GROUP_KEY_COLUMN,
            dropna=False,
        ):
            group_rows = len(group_df)

            if group_rows < MIN_ROWS_REQUIRED_FOR_KNN_PROFILE:
                dbscan_knn_group_skip_records.append(
                    {
                        "feature_set": feature_set_name,
                        "representation_name": representation_name,
                        "comparison_group_key": comparison_group_key,
                        "group_rows": group_rows,
                        "skip_reason": f"group_rows < {MIN_ROWS_REQUIRED_FOR_KNN_PROFILE}",
                    }
                )
                continue

            if group_df[representation_columns].isna().any().any():
                missing_columns = (
                    group_df[representation_columns]
                    .isna()
                    .sum()
                    .loc[lambda s: s.gt(0)]
                    .index.tolist()
                )
                raise ValueError(
                    f"NaNs detected in {feature_set_name} / {representation_name} / "
                    f"{comparison_group_key}. Missing columns: {missing_columns}"
                )

            X_group = group_df[representation_columns].to_numpy(dtype=float)

            for neighbor_k in DBSCAN_K_DISTANCE_K_VALUES:
                knn_model = NearestNeighbors(
                    n_neighbors=neighbor_k + 1,
                    metric="euclidean",
                )
                knn_model.fit(X_group)

                distances, _ = knn_model.kneighbors(X_group)
                kth_distances = distances[:, neighbor_k]

                dbscan_knn_profile_records.append(
                    {
                        "feature_set": feature_set_name,
                        "representation_name": representation_name,
                        "comparison_group_key": comparison_group_key,
                        "group_rows": group_rows,
                        "representation_columns": len(representation_columns),
                        "neighbor_k": neighbor_k,
                        "mean_k_distance": float(np.mean(kth_distances)),
                        "median_k_distance": float(np.median(kth_distances)),
                        "p90_k_distance": float(np.quantile(kth_distances, 0.90)),
                        "p95_k_distance": float(np.quantile(kth_distances, 0.95)),
                        "p99_k_distance": float(np.quantile(kth_distances, 0.99)),
                        "std_k_distance": float(np.std(kth_distances)),
                        "coefficient_of_variation": float(
                            np.std(kth_distances) / np.mean(kth_distances)
                        )
                        if np.mean(kth_distances) > 0
                        else np.nan,
                    }
                )

dbscan_knn_profile_df = pd.DataFrame(dbscan_knn_profile_records)
dbscan_knn_group_skip_df = pd.DataFrame(dbscan_knn_group_skip_records)

if should_rebuild(REBUILD_DBSCAN_DENSITY_DIAGNOSTICS, DBSCAN_K_DISTANCE_PROFILE_PATH):
    dbscan_knn_profile_df.to_parquet(DBSCAN_K_DISTANCE_PROFILE_PATH, index=False)

display(dbscan_knn_profile_df)

if not dbscan_knn_group_skip_df.empty:
    display(dbscan_knn_group_skip_df)

,feature_set,representation_name,comparison_group_key,group_rows,representation_columns,neighbor_k,mean_k_distance,median_k_distance,p90_k_distance,p95_k_distance,p99_k_distance,std_k_distance,coefficient_of_variation
0,bus,pca_80pct,weekday_am_peak | cbd,1080,14,5,2.120854,1.897112,3.087021,3.623417,5.900592,0.925900,0.436569
1,bus,pca_80pct,weekday_am_peak | cbd,1080,14,10,2.420070,2.180830,3.449153,4.052044,6.961691,1.038570,0.429149
2,bus,pca_80pct,weekday_am_peak | cbd,1080,14,15,2.607771,2.348858,3.663652,4.341441,7.504654,1.120508,0.429680
3,bus,pca_80pct,weekday_am_peak | cbd,1080,14,20,2.747580,2.455266,3.858824,4.492002,7.772802,1.190210,0.433185
4,bus,pca_80pct,weekday_am_peak | gateway_or_adjacent,487,14,5,2.244540,1.956475,3.546816,4.341914,6.298887,1.031031,0.459351
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,multimodal,umap_pca_to_umap_10d,weekend_pm_peak | gateway_or_adjacent,214,10,20,1.063195,0.357654,0.923639,5.718965,13.054302,2.578915,2.425627
1196,multimodal,umap_pca_to_umap_10d,weekend_pm_peak | outside,2237,10,5,0.313443,0.281838,0.511984,0.632512,1.238694,0.208370,0.664776
1197,multimodal,umap_pca_to_umap_10d,weekend_pm_peak | outside,2237,10,10,0.451285,0.376857,0.713882,0.975205,1.916953,0.338192,0.749398
1198,multimodal,umap_pca_to_umap_10d,weekend_pm_peak | outside,2237,10,15,0.543973,0.440613,0.866625,1.464690,2.000319,0.383635,0.705246


Findings\. This detail table is mainly a drill\-down view, so the columns to watch are comparison\_group\_key, neighbor\_k, p95\_k\_distance, and coefficient\_of\_variation\. The easy takeaway from the rows shown is that local density is clearly not uniform even within one modality\-representation branch: for Bus PCA, CBD groups tend to have tighter and more stable neighbor distances, outside groups are broader and more variable, and some small weekend gateway\_or\_adjacent groups show especially large upper\-tail distances and volatility\. That does not settle the section by itself, but it does confirm that density structure varies meaningfully across temporal\-geographic cells, which is exactly why the next visualization matters\.

The previous table is still a lot to scan row by row, so this chart gives us a cleaner view of upper\-tail neighborhood scale across PCA and UMAP\. The x\-axis is neighborhood size \(k\), and the y\-axis is the mean p95 k\-distance within each branch\. This is a scale view, not a pass/fail DBSCAN score: larger distances do not mean a worse representation on their own\. What we want to see here is how neighborhood scale grows as k widens, how different that scale is between PCA and UMAP, and which modality branches look especially spread out or especially compressed\.

In [17]:
# ---------------------------------------------------------------------
# Visualize upper-tail k-neighbor distance structure
# ---------------------------------------------------------------------

if "dbscan_k_distance_profile_df" not in globals():
    dbscan_k_distance_profile_df = pd.read_parquet(DBSCAN_K_DISTANCE_PROFILE_PATH)

representation_label_map = {
    "pca_80pct": "PCA",
    "umap_pca_to_umap_10d": "UMAP",
}

knn_plot_df = (
    dbscan_k_distance_profile_df
    .copy()
    .assign(
        representation_label=lambda df: (
            df["representation_name"]
            .map(representation_label_map)
            .fillna(df["representation_name"])
        )
    )
    .groupby(
        ["feature_set", "representation_name", "representation_label", "neighbor_k"],
        dropna=False,
        observed=False,
    )
    .agg(
        mean_p95_k_distance=("p95_k_distance", "mean"),
        groups_evaluated=("comparison_group_key", "nunique"),
    )
    .reset_index()
)

knn_distance_fig = px.line(
    knn_plot_df,
    x="neighbor_k",
    y="mean_p95_k_distance",
    color="feature_set",
    line_dash="representation_label",
    markers=True,
    facet_col="representation_label",
    category_orders={
        "feature_set": MODEL_FEATURE_SET_NAMES,
        "representation_label": ["PCA", "UMAP"],
    },
    color_discrete_map=get_feature_set_color_map(MODEL_FEATURE_SET_NAMES),
    labels={
        "neighbor_k": "Neighborhood size (k)",
        "mean_p95_k_distance": "Mean p95 k-distance",
        "feature_set": "Feature set",
        "representation_label": "Representation",
    },
    hover_data={
        "representation_name": False,
        "groups_evaluated": True,
        "mean_p95_k_distance": ":.4f",
    },
)

knn_distance_fig.update_traces(
    line={"width": 2},
    marker={"size": 7},
)

knn_distance_fig.for_each_annotation(
    lambda annotation: annotation.update(text=annotation.text.split("=")[-1])
)

apply_brand_plotly_layout(
    knn_distance_fig,
    title="Upper-Tail k-Neighbor Distance Structure",
    width=980,
    height=430,
    legend_title="Feature set / representation",
    margin={"l": 70, "r": 30, "t": 80, "b": 90},
)

knn_distance_fig.show()

Findings\. The main takeaway from this chart is scale separation rather than curve shape\. For every modality, the PCA branch operates on a much larger upper\-tail distance scale than the UMAP branch, which means DBSCAN will require representation\-specific eps calibration rather than anything close to a shared distance threshold\. Within each representation, mean p95 k\-distance rises gradually as neighborhood size increases, which is expected, but the branches do not rise in equally interchangeable ways\. Subway UMAP shows the clearest late jump at larger neighborhood sizes, while Taxi UMAP remains comparatively flat; on the PCA side, Multimodal stands apart with the largest upper\-tail distances throughout\. The practical implication is that neighborhood scale differs materially across both modality and representation, so DBSCAN calibration needs to stay modality\-specific and representation\-specific, and absolute distance size alone is not enough to judge whether a surface is well\-suited to DBSCAN\.

### Summarize density heterogeneity and DBSCAN readiness

This last part of the section rolls the group\-level k\-nearest\-neighbor profiles into a branch\-level readiness view\. We want to know whether each modality\-representation surface looks calm enough for DBSCAN to keep calibrating, or whether density heterogeneity is already high enough to treat that branch as a weaker fit\.

We’ll summarize each branch along two dimensions: how uneven local neighbor distances are within groups, and how much upper\-tail distance scale shifts across groups\. Together, those diagnostics give us a practical read on where DBSCAN looks more comfortable, where it looks mixed, and where it is likely to struggle\.

Let’s roll those group\-level distance profiles up into a compact heterogeneity summary\. This is the bridge from raw local\-distance diagnostics into the broader question we actually care about: does DBSCAN look comfortable enough on each modality\-representation branch to keep calibrating?

In [18]:
# ---------------------------------------------------------------------
# Summarize density heterogeneity by modality and representation
# ---------------------------------------------------------------------

dbscan_density_heterogeneity_summary_df = (
    dbscan_knn_profile_df.groupby(
        ["feature_set", "representation_name"],
        dropna=False,
    )
    .agg(
        groups_profiled=("comparison_group_key", "nunique"),
        mean_group_rows=("group_rows", "mean"),
        min_group_rows=("group_rows", "min"),
        max_group_rows=("group_rows", "max"),
        mean_p95_k_distance=("p95_k_distance", "mean"),
        std_p95_k_distance=("p95_k_distance", "std"),
        mean_coefficient_of_variation=("coefficient_of_variation", "mean"),
        max_coefficient_of_variation=("coefficient_of_variation", "max"),
    )
    .reset_index()
)

# A simple readiness readout: lower within-branch spread and lower
# local-density variability are easier for DBSCAN to calibrate cleanly.
dbscan_density_heterogeneity_summary_df["dbscan_density_readiness"] = np.select(
    [
        (
            dbscan_density_heterogeneity_summary_df["mean_coefficient_of_variation"] <= 0.35
        )
        & (
            dbscan_density_heterogeneity_summary_df["std_p95_k_distance"]
            <= dbscan_density_heterogeneity_summary_df["mean_p95_k_distance"] * 0.35
        ),
        (
            dbscan_density_heterogeneity_summary_df["mean_coefficient_of_variation"] <= 0.55
        )
        & (
            dbscan_density_heterogeneity_summary_df["std_p95_k_distance"]
            <= dbscan_density_heterogeneity_summary_df["mean_p95_k_distance"] * 0.60
        ),
    ],
    [
        "comfortable",
        "caution",
    ],
    default="risky",
)

round_cols = [
    "mean_group_rows",
    "mean_p95_k_distance",
    "std_p95_k_distance",
    "mean_coefficient_of_variation",
    "max_coefficient_of_variation",
]

dbscan_density_heterogeneity_summary_df[round_cols] = (
    dbscan_density_heterogeneity_summary_df[round_cols].round(3)
)

if should_rebuild(REBUILD_DBSCAN_DENSITY_DIAGNOSTICS, DBSCAN_DENSITY_PROFILE_PATH):
    dbscan_density_heterogeneity_summary_df.to_parquet(
        DBSCAN_DENSITY_PROFILE_PATH,
        index=False,
    )

display(dbscan_density_heterogeneity_summary_df)

,feature_set,representation_name,groups_profiled,mean_group_rows,min_group_rows,max_group_rows,mean_p95_k_distance,std_p95_k_distance,mean_coefficient_of_variation,max_coefficient_of_variation,dbscan_density_readiness
0,bus,pca_80pct,30,1666.667,190,5640,5.512,2.645,0.485,0.921,caution
1,bus,umap_pca_to_umap_10d,30,1666.667,190,5640,1.218,2.010,1.880,5.397,risky
2,fhvhv,pca_80pct,30,1666.667,214,5532,5.161,1.594,0.650,0.951,risky
3,fhvhv,umap_pca_to_umap_10d,30,1666.667,214,5532,1.735,2.900,1.807,4.426,risky
4,multimodal,pca_80pct,30,1666.667,214,5532,13.003,3.501,0.508,0.874,caution
5,multimodal,umap_pca_to_umap_10d,30,1666.667,214,5532,1.556,1.879,1.751,3.682,risky
6,subway,pca_80pct,30,1666.667,254,5137,3.140,1.906,1.104,2.448,risky
7,subway,umap_pca_to_umap_10d,30,1666.667,254,5137,1.102,2.430,1.777,5.608,risky
8,taxi,pca_80pct,30,1666.667,214,5532,6.263,2.368,0.710,1.076,risky
9,taxi,umap_pca_to_umap_10d,30,1666.667,214,5532,0.759,0.350,1.938,3.674,risky


Findings\. The table condenses the density\-profile story into two columns worth focusing on: mean\_coefficient\_of\_variation and std\_p95\_k\_distance\. Higher values mean the representation’s local density structure is less uniform across comparison groups, which is bad news for a method like DBSCAN that prefers a more stable density regime\. On that test, PCA looks more manageable for Bus and Multimodal, which land in caution, while every UMAP branch is flagged risky because its local\-distance variation is much higher\. Subway and Taxi are also risky even in PCA, so this is not just a UMAP problem; it’s a broader warning that DBSCAN may fit some modalities more naturally than others\.

Let’s put that readiness readout into a visual summary so we can quickly see where DBSCAN looks more comfortable and where it may be fighting the data structure\.

> 📌 Note\. This final view brings the two density\-readiness metrics together in one place\. The x\-axis shows the mean local\-distance coefficient of variation, which captures how uneven neighborhood density is within comparison groups; lower values mean a more stable local\-density surface for DBSCAN\. The y\-axis shows the standard deviation of group\-level p95 k\-distance, which captures how much upper\-tail neighborhood scale shifts across temporal\-geography groups; lower values again indicate a more uniform density regime\. Points closer to the lower\-left are therefore more DBSCAN\-friendly, but the main value of the plot is comparative: it helps us see which modality\-representation branches look calmer, which look mixed, and which look especially heterogeneous\.

In [19]:
# ---------------------------------------------------------------------
# Visualize density heterogeneity and DBSCAN readiness
# ---------------------------------------------------------------------

if "dbscan_density_profile_df" not in globals():
    dbscan_density_profile_df = pd.read_parquet(DBSCAN_DENSITY_PROFILE_PATH)

density_plot_df = dbscan_density_profile_df.copy()

representation_label_map = {
    "pca_80pct": "PCA",
    "umap_pca_to_umap_10d": "UMAP",
}

density_plot_df["representation_label"] = (
    density_plot_df["representation_name"]
    .map(representation_label_map)
    .fillna(density_plot_df["representation_name"])
)

density_readiness_fig = px.scatter(
    density_plot_df,
    x="mean_coefficient_of_variation",
    y="std_p95_k_distance",
    color="feature_set",
    symbol="representation_label",
    size="groups_profiled" if "groups_profiled" in density_plot_df.columns else None,
    category_orders={
        "feature_set": MODEL_FEATURE_SET_NAMES,
        "representation_label": ["PCA", "UMAP"],
    },
    color_discrete_map=get_feature_set_color_map(MODEL_FEATURE_SET_NAMES),
    labels={
        "mean_coefficient_of_variation": "Mean local-distance CV",
        "std_p95_k_distance": "Std. dev. of group p95 k-distance",
        "feature_set": "Feature set",
        "representation_label": "Representation",
    },
    hover_data={
        "representation_name": False,
        "groups_profiled": True,
        "mean_p95_k_distance": ":.3f",
        "mean_coefficient_of_variation": ":.3f",
        "std_p95_k_distance": ":.3f",
    },
)

density_readiness_fig.update_traces(
    marker={
        "size": 11,
        "line": {"color": BRAND_COLORS["dark_teal"], "width": 0.75},
    }
)

apply_brand_plotly_layout(
    density_readiness_fig,
    title="Density Heterogeneity and DBSCAN Readiness",
    width=900,
    height=520,
    legend_title="Feature set / representation",
)

density_readiness_fig.show()

Findings\. This scatter should be read as a tradeoff view rather than a single ranking\. The x\-axis captures within\-group local\-distance variability, while the y\-axis captures how much upper\-tail neighborhood scale shifts across comparison groups; lower is better on both, but the PCA branches do not separate into a clean “good” versus “bad” split\. Bus PCA looks the most favorable overall, and Subway PCA is the clearest concern because it remains elevated on both dimensions\. FHVHV PCA, Taxi PCA, and Multimodal PCA sit in a middle band with different tradeoffs rather than a decisive ordering: FHVHV has comparatively lower cross\-group spread, Taxi has somewhat higher local variation but more moderate upper\-tail spread, and Multimodal has lower local variation than Taxi but higher cross\-group spread\. The sharper result is on the UMAP side: every UMAP branch sits much farther to the right, indicating substantially more uneven local density structure, which makes UMAP a weaker geometric fit for DBSCAN even when its upper\-tail spread is not always the worst\.

> 🎯 Decision\. UMAP will not advance as a primary DBSCAN calibration branch, while all PCA branches remain in scope for further calibration\. The density\-readiness diagnostics show that UMAP surfaces are consistently more heterogeneous in local neighborhood structure, making them a weaker fit for DBSCAN’s shared\-density assumptions\. The PCA branches are more mixed: Bus appears most favorable, Subway appears most concerning, and Taxi, FHVHV, and Multimodal fall into a middle band that still warrants direct parameter and stability testing rather than early elimination\.

> 📌 Note\. This conclusion does not undercut the lightweight DBSCAN pilots used earlier in 3\.3\.1\. Those pilots were used as diagnostic probes for baseline design — for example, testing whether temporal and geography\-aware comparison universes reduced obvious concentration distortions\. Here, the question is narrower and stricter: whether DBSCAN itself looks well\-matched enough to a representation’s local\-density geometry to justify full anomaly calibration on that branch\.

### Section Summary

This section showed that meaningful density heterogeneity remains even after restricting the analysis to the shared temporal\_bucket \+ policy\_geography\_class comparison universe\. Group\-level k\-nearest\-neighbor diagnostics revealed that PCA and UMAP do not present equally DBSCAN\-friendly surfaces: UMAP branches are consistently more uneven in local\-density structure, while PCA branches are more stable but still modality\-dependent\. Bus PCA looks like the calmest branch, Subway PCA remains the clearest concern, and Taxi, FHVHV, and Multimodal fall into a middle band that still needs parameter and stability testing rather than early elimination\. The practical takeaway is that DBSCAN calibration should continue, but with PCA as the primary calibration surface and UMAP treated as a weaker fit for DBSCAN going forward\.

## 3\.3\.2\.3 Calibrate Candidate DBSCAN Parameters

Now that we’ve profiled local density structure, we can move into actual DBSCAN calibration\. This section uses the k\-nearest\-neighbor diagnostics to define plausible eps regions, tests candidate min\_samples settings against those distance scales, and checks whether the resulting anomaly behavior stays interpretable within each modality\-representation surface\.

The goal here is to narrow the search space to parameter regions that are realistic enough to carry into the later stability section\. We are not trying to prove that one global DBSCAN setting works everywhere; we are trying to identify modality\-specific and representation\-specific candidate settings that produce sensible anomaly prevalence, reasonable cluster structure, and manageable fragmentation\.

### Define candidate parameter grids from k\-distance structure

Now that the density\-readiness screen is complete, we can move into actual DBSCAN calibration on the surviving PCA branches\. This section uses the k\-nearest\-neighbor diagnostics to define plausible eps regions, tests candidate min\_samples settings against those distance scales, and records how anomaly\-like volume and cluster structure behave under each candidate setting\.

In [20]:
DBSCAN_ACTIVE_REPRESENTATION_NAMES = ["pca_80pct"]
DBSCAN_RETIRED_REPRESENTATION_NAMES = ["umap_pca_to_umap_10d"]

The goal is to narrow the search space to modality\-specific PCA parameter regions that are reasonable enough to carry into the later stability section\. UMAP remains documented as a weaker DBSCAN fit, but it does not advance into the main calibration sweep\.

Let’s first condense the group\-level k\-nearest\-neighbor diagnostics into branch\-level eps anchors for the PCA surfaces that survived the density\-readiness screen\. These anchors give each modality its own empirical starting region for DBSCAN calibration\.

In [21]:
# ---------------------------------------------------------------------
# Summarize PCA branch-level eps anchors from k-distance structure
# ---------------------------------------------------------------------

DBSCAN_PARAMETER_ANCHOR_SUMMARY_PATH = (
    INTERMEDIATE_332_DIR / "dbscan_parameter_anchor_summary.parquet"
)

dbscan_parameter_anchor_summary_df = (
    dbscan_knn_profile_df.loc[
        dbscan_knn_profile_df["representation_name"].isin(DBSCAN_ACTIVE_REPRESENTATION_NAMES)
    ]
    .groupby(
        ["feature_set", "representation_name", "neighbor_k"],
        dropna=False,
    )
    .agg(
        groups_profiled=("comparison_group_key", "nunique"),
        mean_group_rows=("group_rows", "mean"),
        min_group_rows=("group_rows", "min"),
        max_group_rows=("group_rows", "max"),
        median_p90_eps_anchor=("p90_k_distance", "median"),
        median_p95_eps_anchor=("p95_k_distance", "median"),
        median_p99_eps_anchor=("p99_k_distance", "median"),
        mean_coefficient_of_variation=("coefficient_of_variation", "mean"),
        max_coefficient_of_variation=("coefficient_of_variation", "max"),
    )
    .reset_index()
    .merge(
        dbscan_density_heterogeneity_summary_df[
            [
                "feature_set",
                "representation_name",
                "dbscan_density_readiness",
            ]
        ],
        on=["feature_set", "representation_name"],
        how="left",
    ).round(3)
)

if should_rebuild(REBUILD_DBSCAN_PARAMETER_CALIBRATION, DBSCAN_PARAMETER_ANCHOR_SUMMARY_PATH):
    dbscan_parameter_anchor_summary_df.to_parquet(
        DBSCAN_PARAMETER_ANCHOR_SUMMARY_PATH,
        index=False,
    )

display(dbscan_parameter_anchor_summary_df)

,feature_set,representation_name,neighbor_k,groups_profiled,mean_group_rows,min_group_rows,max_group_rows,median_p90_eps_anchor,median_p95_eps_anchor,median_p99_eps_anchor,mean_coefficient_of_variation,max_coefficient_of_variation,dbscan_density_readiness
0,bus,pca_80pct,5,30,1666.667,190,5640,3.401,4.146,6.264,0.475,0.694,caution
1,bus,pca_80pct,10,30,1666.667,190,5640,3.951,4.863,7.638,0.486,0.921,caution
2,bus,pca_80pct,15,30,1666.667,190,5640,4.233,5.281,8.301,0.488,0.909,caution
3,bus,pca_80pct,20,30,1666.667,190,5640,4.458,5.719,9.173,0.491,0.881,caution
4,fhvhv,pca_80pct,5,30,1666.667,214,5532,2.984,3.992,6.822,0.653,0.883,risky
5,fhvhv,pca_80pct,10,30,1666.667,214,5532,3.397,4.598,8.480,0.654,0.891,risky
6,fhvhv,pca_80pct,15,30,1666.667,214,5532,3.679,5.116,9.182,0.649,0.928,risky
7,fhvhv,pca_80pct,20,30,1666.667,214,5532,3.874,5.463,9.638,0.643,0.951,risky
8,multimodal,pca_80pct,5,30,1666.667,214,5532,8.505,10.804,16.304,0.503,0.710,caution
9,multimodal,pca_80pct,10,30,1666.667,214,5532,9.762,12.751,19.029,0.514,0.874,caution


Findings\. The anchor summary reduces the surviving PCA branches to branch\-specific eps reference points, giving each modality an empirical distance scale for DBSCAN calibration\.

With those anchors in place, we can expand them into a candidate DBSCAN grid\. Each modality keeps its own eps region, and each min\_samples setting is tied back to the nearest profiled neighborhood size so the sweep stays grounded in the observed k\-distance structure\.

In [22]:
# ---------------------------------------------------------------------
# Build candidate PCA-only DBSCAN parameter grids from branch-specific anchors
# ---------------------------------------------------------------------

DBSCAN_PARAMETER_CANDIDATE_GRID_PATH = (
    INTERMEDIATE_332_DIR / "dbscan_parameter_candidate_grid.parquet"
)

profiled_k_values = sorted(
    dbscan_parameter_anchor_summary_df["neighbor_k"].unique().tolist()
)

candidate_grid_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    for representation_name in DBSCAN_ACTIVE_REPRESENTATION_NAMES:
        branch_anchor_df = dbscan_parameter_anchor_summary_df.loc[
            dbscan_parameter_anchor_summary_df["feature_set"].eq(feature_set_name)
            & dbscan_parameter_anchor_summary_df["representation_name"].eq(representation_name)
        ].copy()

        if branch_anchor_df.empty:
            continue

        readiness_label = branch_anchor_df["dbscan_density_readiness"].iloc[0]

        for min_samples in DBSCAN_DEFAULT_MIN_SAMPLES_GRID:
            anchor_neighbor_k = min(
                profiled_k_values,
                key=lambda k_value: (abs(k_value - min_samples), k_value),
            )

            anchor_row = branch_anchor_df.loc[
                branch_anchor_df["neighbor_k"].eq(anchor_neighbor_k)
            ].iloc[0]

            eps_candidates = [
                ("tight", anchor_row["median_p90_eps_anchor"]),
                ("balanced", anchor_row["median_p95_eps_anchor"]),
                ("loose", anchor_row["median_p99_eps_anchor"]),
            ]

            for eps_band, eps_value in eps_candidates:
                candidate_grid_records.append(
                    {
                        "feature_set": feature_set_name,
                        "representation_name": representation_name,
                        "dbscan_density_readiness": readiness_label,
                        "min_samples": int(min_samples),
                        "anchor_neighbor_k": int(anchor_neighbor_k),
                        "anchor_strategy": "nearest_profiled_k",
                        "eps_band": eps_band,
                        "eps_value": float(eps_value),
                        "groups_profiled": int(anchor_row["groups_profiled"]),
                        "mean_group_rows": float(anchor_row["mean_group_rows"]),
                        "mean_coefficient_of_variation": float(
                            anchor_row["mean_coefficient_of_variation"]
                        ),
                    }
                )

dbscan_parameter_candidate_grid_df = pd.DataFrame(candidate_grid_records)

dbscan_parameter_grid_inventory_df = (
    dbscan_parameter_candidate_grid_df.groupby(
        ["feature_set", "representation_name", "dbscan_density_readiness"],
        dropna=False,
    )
    .agg(
        candidate_rows=("eps_value", "size"),
        min_samples_values=("min_samples", lambda s: ", ".join(map(str, sorted(s.unique())))),
        anchor_k_values=("anchor_neighbor_k", lambda s: ", ".join(map(str, sorted(s.unique())))),
        min_eps_value=("eps_value", "min"),
        max_eps_value=("eps_value", "max"),
    )
    .reset_index()
)

if should_rebuild(REBUILD_DBSCAN_PARAMETER_CALIBRATION, DBSCAN_PARAMETER_CANDIDATE_GRID_PATH):
    dbscan_parameter_candidate_grid_df.to_parquet(
        DBSCAN_PARAMETER_CANDIDATE_GRID_PATH,
        index=False,
    )

display(dbscan_parameter_grid_inventory_df)
display(dbscan_parameter_candidate_grid_df)

,feature_set,representation_name,dbscan_density_readiness,candidate_rows,min_samples_values,anchor_k_values,min_eps_value,max_eps_value
0,bus,pca_80pct,caution,12,"10, 15, 20, 30","10, 15, 20",3.951,9.173
1,fhvhv,pca_80pct,risky,12,"10, 15, 20, 30","10, 15, 20",3.397,9.638
2,multimodal,pca_80pct,caution,12,"10, 15, 20, 30","10, 15, 20",9.762,21.074
3,subway,pca_80pct,risky,12,"10, 15, 20, 30","10, 15, 20",1.865,5.443
4,taxi,pca_80pct,risky,12,"10, 15, 20, 30","10, 15, 20",4.143,11.022


,feature_set,representation_name,dbscan_density_readiness,min_samples,anchor_neighbor_k,anchor_strategy,eps_band,eps_value,groups_profiled,mean_group_rows,mean_coefficient_of_variation
0,bus,pca_80pct,caution,10,10,nearest_profiled_k,tight,3.951,30,1666.667,0.486
1,bus,pca_80pct,caution,10,10,nearest_profiled_k,balanced,4.863,30,1666.667,0.486
2,bus,pca_80pct,caution,10,10,nearest_profiled_k,loose,7.638,30,1666.667,0.486
3,bus,pca_80pct,caution,15,15,nearest_profiled_k,tight,4.233,30,1666.667,0.488
4,bus,pca_80pct,caution,15,15,nearest_profiled_k,balanced,5.281,30,1666.667,0.488
5,bus,pca_80pct,caution,15,15,nearest_profiled_k,loose,8.301,30,1666.667,0.488
6,bus,pca_80pct,caution,20,20,nearest_profiled_k,tight,4.458,30,1666.667,0.491
7,bus,pca_80pct,caution,20,20,nearest_profiled_k,balanced,5.719,30,1666.667,0.491
8,bus,pca_80pct,caution,20,20,nearest_profiled_k,loose,9.173,30,1666.667,0.491
9,bus,pca_80pct,caution,30,20,nearest_profiled_k,tight,4.458,30,1666.667,0.491


Findings\. The candidate grid expands each surviving PCA branch into a compact DBSCAN search space by pairing branch\-specific eps anchors with the selected min\_samples settings\.

### Run a lightweight DBSCAN calibration sweep

With the candidate parameter grid defined, we can now run a lightweight DBSCAN sweep across the surviving PCA branches\. In DBSCAN, eps controls how far a point can reach to count nearby neighbors as part of the same dense region, while min\_samples controls how many nearby points are needed for that region to qualify as a cluster rather than sparse noise\. This sweep applies each candidate eps and min\_samples combination within the shared temporal\_bucket \+ policy\_geography\_class comparison universe and records the basic calibration behaviors we care about\.

The goal is to surface how anomaly\-like volume, cluster structure, and dominant\-cluster behavior respond to the candidate settings\. We are not selecting winners in this subsection yet; we are generating the calibration evidence that the next part will interpret\.

Now we can run the lightweight calibration sweep itself\. Each candidate setting is applied group by group inside the shared temporal\-plus\-policy\-geography comparison universe\. Tighter eps values usually make DBSCAN more selective and can increase anomaly\-like volume or fragmentation, while larger min\_samples values make it harder for small dense pockets to qualify as clusters\. The point here is to observe how those two levers change anomaly prevalence and cluster structure in each modality before we narrow the search space\.

In [23]:
# ---------------------------------------------------------------------
# Run a lightweight PCA-only DBSCAN calibration sweep
# ---------------------------------------------------------------------

DBSCAN_CALIBRATION_SWEEP_SUMMARY_PATH = (
    INTERMEDIATE_332_DIR / "dbscan_calibration_sweep_summary.parquet"
)
DBSCAN_CALIBRATION_SWEEP_GROUP_DETAIL_PATH = (
    INTERMEDIATE_332_DIR / "dbscan_calibration_sweep_group_detail.parquet"
)

should_run_dbscan_calibration_sweep = should_rebuild(
    REBUILD_DBSCAN_PARAMETER_CALIBRATION,
    DBSCAN_CALIBRATION_SWEEP_SUMMARY_PATH,
) or (not DBSCAN_CALIBRATION_SWEEP_GROUP_DETAIL_PATH.exists())

if should_run_dbscan_calibration_sweep:
    density_profile_panel_df = pd.read_parquet(DBSCAN_DENSITY_PROFILE_PANEL_PATH)
    dbscan_parameter_candidate_grid_df = pd.read_parquet(DBSCAN_PARAMETER_CANDIDATE_GRID_PATH)

    sweep_summary_records = []
    sweep_group_detail_records = []

    candidate_rows = dbscan_parameter_candidate_grid_df.to_dict("records")
    total_candidates = len(candidate_rows)

    for candidate_index, candidate_row in enumerate(candidate_rows, start=1):
        feature_set_name = candidate_row["feature_set"]
        representation_name = candidate_row["representation_name"]
        min_samples = int(candidate_row["min_samples"])
        eps_band = candidate_row["eps_band"]
        eps_value = float(candidate_row["eps_value"])

        print(
            f"[{candidate_index}/{total_candidates}] "
            f"Calibrating {feature_set_name} / {representation_name} "
            f"| min_samples={min_samples} | eps_band={eps_band} | eps={eps_value:.4f}"
        )

        branch_df = density_profile_panel_df.loc[
            density_profile_panel_df["feature_set"].eq(feature_set_name)
            & density_profile_panel_df["representation_name"].eq(representation_name)
        ].copy()

        if feature_set_name == "taxi" and TAXI_PCA_HANDLING_DECISION == "split_prepost_period":
            representation_columns = get_numeric_representation_columns(
                TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD["pre_cp"]
            )
        else:
            representation_columns = get_numeric_representation_columns(
                PCA_80_FINAL_PATHS_BY_SET[feature_set_name]
            )

        group_level_noise_rows = 0
        group_level_evaluated_rows = 0
        weighted_cluster_numerator = 0.0
        weighted_largest_cluster_share_numerator = 0.0
        groups_evaluated = 0
        groups_skipped = 0

        for comparison_group_key, group_df in branch_df.groupby(
            DBSCAN_DENSITY_GROUP_KEY_COLUMN,
            dropna=False,
        ):
            group_rows = len(group_df)

            if group_rows < max(min_samples, DBSCAN_PILOT_MIN_ROWS_PER_GROUP):
                groups_skipped += 1
                sweep_group_detail_records.append(
                    {
                        "feature_set": feature_set_name,
                        "representation_name": representation_name,
                        "min_samples": min_samples,
                        "eps_band": eps_band,
                        "eps_value": eps_value,
                        "comparison_group_key": comparison_group_key,
                        "group_rows": group_rows,
                        "rows_evaluated": 0,
                        "noise_rows": 0,
                        "noise_share": np.nan,
                        "cluster_count": np.nan,
                        "largest_cluster_share": np.nan,
                        "status": "skipped_small_group",
                    }
                )
                continue

            X_group = group_df[representation_columns].to_numpy(dtype=float)

            dbscan_model = DBSCAN(
                eps=eps_value,
                min_samples=min_samples,
                metric="euclidean",
            )
            cluster_labels = dbscan_model.fit_predict(X_group)

            evaluated_rows = len(cluster_labels)
            noise_mask = cluster_labels == -1
            noise_rows = int(noise_mask.sum())
            noise_share = float(noise_rows / evaluated_rows)

            non_noise_labels = cluster_labels[cluster_labels != -1]
            distinct_clusters = np.unique(non_noise_labels) if len(non_noise_labels) else np.array([])
            cluster_count = int(len(distinct_clusters))

            if cluster_count > 0:
                cluster_size_series = pd.Series(non_noise_labels).value_counts(normalize=True)
                largest_cluster_share = float(cluster_size_series.iloc[0])
            else:
                largest_cluster_share = np.nan

            groups_evaluated += 1
            group_level_noise_rows += noise_rows
            group_level_evaluated_rows += evaluated_rows
            weighted_cluster_numerator += cluster_count * evaluated_rows
            weighted_largest_cluster_share_numerator += (
                (0.0 if np.isnan(largest_cluster_share) else largest_cluster_share)
                * evaluated_rows
            )

            sweep_group_detail_records.append(
                {
                    "feature_set": feature_set_name,
                    "representation_name": representation_name,
                    "min_samples": min_samples,
                    "eps_band": eps_band,
                    "eps_value": eps_value,
                    "comparison_group_key": comparison_group_key,
                    "group_rows": group_rows,
                    "rows_evaluated": evaluated_rows,
                    "noise_rows": noise_rows,
                    "noise_share": noise_share,
                    "cluster_count": cluster_count,
                    "largest_cluster_share": largest_cluster_share,
                    "status": "evaluated",
                }
            )

        sweep_summary_records.append(
            {
                "feature_set": feature_set_name,
                "representation_name": representation_name,
                "min_samples": min_samples,
                "eps_band": eps_band,
                "eps_value": eps_value,
                "groups_evaluated": groups_evaluated,
                "groups_skipped": groups_skipped,
                "rows_evaluated": group_level_evaluated_rows,
                "weighted_noise_share": (
                    group_level_noise_rows / group_level_evaluated_rows
                    if group_level_evaluated_rows > 0
                    else np.nan
                ),
                "weighted_cluster_count": (
                    weighted_cluster_numerator / group_level_evaluated_rows
                    if group_level_evaluated_rows > 0
                    else np.nan
                ),
                "weighted_largest_cluster_share": (
                    weighted_largest_cluster_share_numerator / group_level_evaluated_rows
                    if group_level_evaluated_rows > 0
                    else np.nan
                ),
                "status": "pass" if groups_evaluated > 0 else "review",
            }
        )

    dbscan_calibration_sweep_summary_df = pd.DataFrame(sweep_summary_records)
    dbscan_calibration_sweep_group_detail_df = pd.DataFrame(sweep_group_detail_records)

    if WRITE_332_OUTPUTS:
        dbscan_calibration_sweep_summary_df.to_parquet(
            DBSCAN_CALIBRATION_SWEEP_SUMMARY_PATH,
            index=False,
        )
        dbscan_calibration_sweep_group_detail_df.to_parquet(
            DBSCAN_CALIBRATION_SWEEP_GROUP_DETAIL_PATH,
            index=False,
        )
else:
    dbscan_calibration_sweep_summary_df = pd.read_parquet(
        DBSCAN_CALIBRATION_SWEEP_SUMMARY_PATH
    )
    dbscan_calibration_sweep_group_detail_df = pd.read_parquet(
        DBSCAN_CALIBRATION_SWEEP_GROUP_DETAIL_PATH
    )

    print("Loaded existing lightweight PCA-only DBSCAN calibration sweep outputs.")

# Round floating-point summary metrics for readability
float_columns = dbscan_calibration_sweep_summary_df.select_dtypes(
    include="float"
).columns

dbscan_calibration_sweep_summary_df[float_columns] = (
    dbscan_calibration_sweep_summary_df[float_columns]
    .round(3)
)

display(dbscan_calibration_sweep_summary_df)

Loaded existing lightweight PCA-only DBSCAN calibration sweep outputs.


,feature_set,representation_name,min_samples,eps_band,eps_value,groups_evaluated,groups_skipped,rows_evaluated,weighted_noise_share,weighted_cluster_count,weighted_largest_cluster_share,status
0,bus,pca_80pct,10,tight,3.951,30,0,50000,0.037,3.310,0.959,pass
1,bus,pca_80pct,10,balanced,4.863,30,0,50000,0.019,2.012,0.965,pass
2,bus,pca_80pct,10,loose,7.638,30,0,50000,0.005,1.601,0.996,pass
3,bus,pca_80pct,15,tight,4.233,30,0,50000,0.036,2.392,0.963,pass
4,bus,pca_80pct,15,balanced,5.281,30,0,50000,0.018,1.787,0.966,pass
5,bus,pca_80pct,15,loose,8.301,30,0,50000,0.006,1.156,0.998,pass
6,bus,pca_80pct,20,tight,4.458,30,0,50000,0.033,1.989,0.966,pass
7,bus,pca_80pct,20,balanced,5.719,30,0,50000,0.015,1.563,0.968,pass
8,bus,pca_80pct,20,loose,9.173,30,0,50000,0.005,1.156,0.998,pass
9,bus,pca_80pct,30,tight,4.458,30,0,50000,0.042,1.185,0.970,pass


Findings\. The lightweight PCA\-only calibration sweep completed cleanly across all 60 candidate settings, and the results already show the core tradeoffs we care about: tighter eps settings produce higher anomaly\-like volume and more cluster structure, while looser settings quickly collapse several modalities toward one\-cluster behavior with very low noise share\.

Let’s close the sweep with a quick QA pass\. This confirms that every candidate setting produced a summary row and tells us how much of the sweep was evaluated versus skipped because a group was too small for that configuration\.

In [24]:
# ---------------------------------------------------------------------
# Validate lightweight PCA-only DBSCAN calibration sweep outputs
# ---------------------------------------------------------------------

expected_candidate_rows = len(dbscan_parameter_candidate_grid_df)

dbscan_calibration_sweep_validation_df = pd.DataFrame(
    [
        {
            "expected_candidate_rows": expected_candidate_rows,
            "actual_summary_rows": len(dbscan_calibration_sweep_summary_df),
            "actual_group_detail_rows": len(dbscan_calibration_sweep_group_detail_df),
            "summary_rows_with_review_status": int(
                (~dbscan_calibration_sweep_summary_df["status"].eq("pass")).sum()
            ),
            "evaluated_group_rows": int(
                dbscan_calibration_sweep_group_detail_df["status"].eq("evaluated").sum()
            ),
            "skipped_group_rows": int(
                dbscan_calibration_sweep_group_detail_df["status"].eq("skipped_small_group").sum()
            ),
            "status": (
                "pass"
                if len(dbscan_calibration_sweep_summary_df) == expected_candidate_rows
                else "review"
            ),
        }
    ]
)

display(dbscan_calibration_sweep_validation_df)

assert dbscan_calibration_sweep_validation_df["status"].eq("pass").all(), (
    "The DBSCAN calibration sweep did not produce the expected number of summary rows."
)

,expected_candidate_rows,actual_summary_rows,actual_group_detail_rows,summary_rows_with_review_status,evaluated_group_rows,skipped_group_rows,status
0,60,60,1800,0,1800,0,pass


Findings\. The QA check passed cleanly, confirming that all 60 candidate settings produced summary outputs, all 1,800 group\-level evaluations ran successfully, and no calibration groups were skipped\.

### Analyze Calibration Behavior and Narrow Candidate Regions

This section turns the PCA\-only calibration sweep into something readable and actionable\. We’ll compare anomaly prevalence and cluster structure across the candidate settings, use those patterns to distinguish settings that look too tight, too loose, or reasonably balanced, and then retain a narrower set of candidate regions for the stability section\.

The goal is not to declare final DBSCAN winners yet\. It is to reduce the parameter search space to settings that still produce interpretable anomaly volume and non\-degenerate cluster structure within each modality\.

Compare anomaly prevalence and cluster structure across the parameter sweep\. This first readout stays close to the raw sweep outputs\. We want to see how anomaly\-like volume, weighted cluster count, and dominant\-cluster share move as eps loosens and min\_samples changes, before we narrow the search space\. The full table is mainly an orientation view: it helps us confirm that the sweep is behaving sensibly and sets up the modality\-specific readouts that follow\.

The three metrics each answer a different calibration question: weighted\_noise\_share tracks anomaly\-like volume, weighted\_cluster\_count shows how much cluster structure DBSCAN is carving out, and weighted\_largest\_cluster\_share tells us whether one dominant cluster is still absorbing nearly everything\. Together, they help us distinguish settings that are too tight, too loose, or still plausibly balanced enough to carry forward\.

In [25]:
# ---------------------------------------------------------------------
# Compare anomaly prevalence and cluster structure across the parameter sweep
# ---------------------------------------------------------------------

if "dbscan_calibration_sweep_summary_df" not in globals():
    dbscan_calibration_sweep_summary_df = pd.read_parquet(
        DBSCAN_CALIBRATION_SWEEP_SUMMARY_PATH
    )

dbscan_calibration_behavior_df = (
    dbscan_calibration_sweep_summary_df.copy()
    .sort_values(["feature_set", "min_samples", "eps_band"])
    .reset_index(drop=True)
)

dbscan_calibration_behavior_df["eps_band"] = pd.Categorical(
    dbscan_calibration_behavior_df["eps_band"],
    categories=["tight", "balanced", "loose"],
    ordered=True,
)

dbscan_calibration_behavior_df["candidate_label"] = (
    "m="
    + dbscan_calibration_behavior_df["min_samples"].astype(str)
    + " | "
    + dbscan_calibration_behavior_df["eps_band"].astype(str)
)

dbscan_calibration_behavior_df["eps_band"] = pd.Categorical(
    dbscan_calibration_behavior_df["eps_band"],
    categories=["tight", "balanced", "loose"],
    ordered=True,
)

dbscan_calibration_behavior_df = (
    dbscan_calibration_behavior_df
    .sort_values(["feature_set", "min_samples", "eps_band"])
    .reset_index(drop=True)
)

float_columns = dbscan_calibration_behavior_df.select_dtypes(
    include="float"
).columns

dbscan_calibration_behavior_df[float_columns] = (
    dbscan_calibration_behavior_df[float_columns]
    .round(3)
)

display(
    dbscan_calibration_behavior_df[
        [
            "feature_set",
            "representation_name",
            "min_samples",
            "eps_band",
            "eps_value",
            "weighted_noise_share",
            "weighted_cluster_count",
            "weighted_largest_cluster_share",
            "groups_evaluated",
            "groups_skipped",
            "status",
        ]
    ]
)

,feature_set,representation_name,min_samples,eps_band,eps_value,weighted_noise_share,weighted_cluster_count,weighted_largest_cluster_share,groups_evaluated,groups_skipped,status
0,bus,pca_80pct,10,tight,3.951,0.037,3.310,0.959,30,0,pass
1,bus,pca_80pct,10,balanced,4.863,0.019,2.012,0.965,30,0,pass
2,bus,pca_80pct,10,loose,7.638,0.005,1.601,0.996,30,0,pass
3,bus,pca_80pct,15,tight,4.233,0.036,2.392,0.963,30,0,pass
4,bus,pca_80pct,15,balanced,5.281,0.018,1.787,0.966,30,0,pass
5,bus,pca_80pct,15,loose,8.301,0.006,1.156,0.998,30,0,pass
6,bus,pca_80pct,20,tight,4.458,0.033,1.989,0.966,30,0,pass
7,bus,pca_80pct,20,balanced,5.719,0.015,1.563,0.968,30,0,pass
8,bus,pca_80pct,20,loose,9.173,0.005,1.156,0.998,30,0,pass
9,bus,pca_80pct,30,tight,4.458,0.042,1.185,0.970,30,0,pass


Findings\. The broad pattern is easy to see even before drilling into individual modalities: as eps loosens from tight to balanced to loose, anomaly\-like volume generally falls, cluster counts contract, and the largest\-cluster share rises toward stronger dominance\. That tells us the sweep is behaving directionally as expected, while the modality\-specific blocks below show where that tradeoff is mild, where it is steep, and where useful candidate regions still seem plausible\.

Plotting Helper\. This helper keeps the modality\-specific calibration plots visually consistent so each mobility system can be compared on the same three DBSCAN behavior metrics without repeating plotting code\.

In [26]:
# ---------------------------------------------------------------------
# Helper to visualize one modality's DBSCAN calibration behavior
# ---------------------------------------------------------------------

def plot_dbscan_calibration_behavior_for_feature_set(
    calibration_df: pd.DataFrame,
    feature_set_name: str,
) -> go.Figure:
    feature_df = (
        calibration_df.loc[calibration_df["feature_set"].eq(feature_set_name)]
        .copy()
        .sort_values(["min_samples", "eps_band"])
    )

    metric_specs = [
        ("weighted_noise_share", "Weighted noise share"),
        ("weighted_cluster_count", "Weighted cluster count"),
        ("weighted_largest_cluster_share", "Weighted largest-cluster share"),
    ]

    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=[label for _, label in metric_specs],
        horizontal_spacing=0.10,
    )

    min_samples_colors = {
        10: BRAND_COLORS["dark_teal"],
        15: BRAND_COLORS["seafoam"],
        20: BRAND_COLORS["pale_peach"],
        30: BRAND_COLORS["terracotta"],
    }

    for col_index, (metric_name, metric_label) in enumerate(metric_specs, start=1):
        for min_samples, min_df in feature_df.groupby("min_samples", dropna=False):
            min_df = min_df.sort_values("eps_band")

            fig.add_trace(
                go.Scatter(
                    x=min_df["eps_band"].astype(str),
                    y=min_df[metric_name],
                    mode="lines+markers",
                    name=f"min_samples={min_samples}",
                    legendgroup=str(min_samples),
                    showlegend=(col_index == 1),
                    line=dict(color=min_samples_colors[int(min_samples)]),
                    marker=dict(size=8),
                    customdata=np.stack(
                        [
                            np.repeat(feature_set_name, len(min_df)),
                            np.repeat(min_samples, len(min_df)),
                            min_df["eps_value"].round(4),
                        ],
                        axis=-1,
                    ),
                    hovertemplate=(
                        "Feature set: %{customdata[0]}<br>"
                        "min_samples: %{customdata[1]}<br>"
                        "eps band: %{x}<br>"
                        "eps value: %{customdata[2]}<br>"
                        + metric_label
                        + ": %{y:.4f}<extra></extra>"
                    ),
                ),
                row=1,
                col=col_index,
            )

    fig.update_layout(
        template="plotly_white",
        paper_bgcolor="white",
        plot_bgcolor=BRAND_COLORS["ice"],
        font=dict(color=BRAND_COLORS["dark_teal"]),
        width=900,
        height=420,
        title=dict(
            text=f"DBSCAN Calibration Behavior: {feature_set_name.title()} PCA",
            font=dict(color=BRAND_COLORS["dark_teal"]),
        ),
        margin=dict(t=70, b=100, l=60, r=20),
        legend=dict(
            title="min_samples",
            orientation="h",
            yanchor="top",
            y=-0.22,
            xanchor="center",
            x=0.5,
            font=dict(size=10, color=BRAND_COLORS["dark_teal"]),
            tracegroupgap=4,
        ),
    )

    for col_index in range(1, 4):
        fig.update_xaxes(
            title_text="eps band",
            row=1,
            col=col_index,
            gridcolor="rgba(11, 78, 74, 0.14)",
            zerolinecolor="rgba(11, 78, 74, 0.20)",
            tickfont=dict(color=BRAND_COLORS["dark_teal"]),
            title_font=dict(color=BRAND_COLORS["dark_teal"]),
        )
        fig.update_yaxes(
            row=1,
            col=col_index,
            gridcolor="rgba(11, 78, 74, 0.14)",
            zerolinecolor="rgba(11, 78, 74, 0.20)",
            tickfont=dict(color=BRAND_COLORS["dark_teal"]),
            title_font=dict(color=BRAND_COLORS["dark_teal"]),
        )

    fig.add_annotation(
        text="Metric value",
        x=-0.06,
        y=0.5,
        xref="paper",
        yref="paper",
        showarrow=False,
        textangle=-90,
        font=dict(size=12, color=BRAND_COLORS["dark_teal"]),
    )

    return fig

Bus calibration behavior\. Now let’s slow down and read one modality at a time\. The goal here is to see whether Bus PCA has a believable candidate region where anomaly\-like volume stays interpretable, cluster structure does not collapse immediately, and the settings are not so tight that they flood the surface with anomalies\.

In [27]:
# ---------------------------------------------------------------------
# Summarize Bus calibration behavior
# ---------------------------------------------------------------------

assert "plot_dbscan_calibration_behavior_for_feature_set" in globals(), (
    "Run the DBSCAN calibration behavior plotting helper cell first."
)

bus_calibration_df = (
    dbscan_calibration_behavior_df.loc[
        dbscan_calibration_behavior_df["feature_set"].eq("bus")
    ]
    .copy()
    .sort_values(["min_samples", "eps_band"])
)

bus_calibration_df["eps_band"] = pd.Categorical(
    bus_calibration_df["eps_band"],
    categories=["tight", "balanced", "loose"],
    ordered=True,
)

bus_calibration_df = bus_calibration_df.sort_values(["min_samples", "eps_band"])

display(
    bus_calibration_df[
        [
            "min_samples",
            "eps_band",
            "eps_value",
            "weighted_noise_share",
            "weighted_cluster_count",
            "weighted_largest_cluster_share",
        ]
    ].round(3)
)

plot_dbscan_calibration_behavior_for_feature_set(
    dbscan_calibration_behavior_df,
    "bus",
).show()

,min_samples,eps_band,eps_value,weighted_noise_share,weighted_cluster_count,weighted_largest_cluster_share
0,10,tight,3.951,0.037,3.310,0.959
1,10,balanced,4.863,0.019,2.012,0.965
2,10,loose,7.638,0.005,1.601,0.996
3,15,tight,4.233,0.036,2.392,0.963
4,15,balanced,5.281,0.018,1.787,0.966
5,15,loose,8.301,0.006,1.156,0.998
6,20,tight,4.458,0.033,1.989,0.966
7,20,balanced,5.719,0.015,1.563,0.968
8,20,loose,9.173,0.005,1.156,0.998
9,30,tight,4.458,0.042,1.185,0.970


Findings\. Bus shows a fairly healthy calibration gradient: moving from tight to balanced cuts noise share roughly in half while still preserving more than one meaningful cluster, but the loose settings push the surface close to dominance by one cluster and very low anomaly volume\. The clearest candidate region is therefore in the balanced band, especially around min\_samples = 10–20, where anomaly volume stays interpretable without the stronger fragmentation seen under the tightest settings\.

Subway calibration behavior\. Here we want to see whether Subway PCA supports any candidate region that keeps anomaly\-like volume under control without collapsing too quickly toward one dominant cluster\. Because Subway already looked like the most heterogeneous PCA branch earlier, this block matters less as a winner search and more as a stress test\.

In [28]:
# ---------------------------------------------------------------------
# Summarize Subway calibration behavior
# ---------------------------------------------------------------------

subway_calibration_df = (
    dbscan_calibration_behavior_df.loc[
        dbscan_calibration_behavior_df["feature_set"].eq("subway")
    ]
    .copy()
    .sort_values(["min_samples", "eps_band"])
)

subway_calibration_df["eps_band"] = pd.Categorical(
    bus_calibration_df["eps_band"],
    categories=["tight", "balanced", "loose"],
    ordered=True,
)
subway_calibration_df = bus_calibration_df.sort_values(["min_samples", "eps_band"])

display(
    subway_calibration_df[
        [
            "min_samples",
            "eps_band",
            "eps_value",
            "weighted_noise_share",
            "weighted_cluster_count",
            "weighted_largest_cluster_share",
        ]
    ].round(3)
)

plot_dbscan_calibration_behavior_for_feature_set(
    dbscan_calibration_behavior_df,
    "subway",
).show()

,min_samples,eps_band,eps_value,weighted_noise_share,weighted_cluster_count,weighted_largest_cluster_share
0,10,tight,3.951,0.037,3.310,0.959
1,10,balanced,4.863,0.019,2.012,0.965
2,10,loose,7.638,0.005,1.601,0.996
3,15,tight,4.233,0.036,2.392,0.963
4,15,balanced,5.281,0.018,1.787,0.966
5,15,loose,8.301,0.006,1.156,0.998
6,20,tight,4.458,0.033,1.989,0.966
7,20,balanced,5.719,0.015,1.563,0.968
8,20,loose,9.173,0.005,1.156,0.998
9,30,tight,4.458,0.042,1.185,0.970


Findings\. Subway remains the most anomaly\-sensitive modality in the sweep, with the highest noise shares across nearly the entire grid, and even its loose settings still retain more anomaly\-like volume than the other modalities\. That means DBSCAN is not immediately collapsing here, but it also means Subway needs extra care because the tighter settings may be too aggressive; the most plausible region appears to be the balanced to loose range rather than the tight end\.

Taxi calibration behavior\. Taxi deserves a close read because even after the split\-PCA handling decision, DBSCAN may still prefer a narrow part of the parameter space\. We’re looking for settings that avoid both over\-fragmentation and immediate collapse into a single\-cluster surface\.

In [29]:
# ---------------------------------------------------------------------
# Summarize Taxi calibration behavior
# ---------------------------------------------------------------------

taxi_calibration_df = (
    dbscan_calibration_behavior_df.loc[
        dbscan_calibration_behavior_df["feature_set"].eq("taxi")
    ]
    .copy()
    .sort_values(["min_samples", "eps_band"])
)

taxi_calibration_df["eps_band"] = pd.Categorical(
    bus_calibration_df["eps_band"],
    categories=["tight", "balanced", "loose"],
    ordered=True,
)
taxi_calibration_df = bus_calibration_df.sort_values(["min_samples", "eps_band"])

display(
    taxi_calibration_df[
        [
            "min_samples",
            "eps_band",
            "eps_value",
            "weighted_noise_share",
            "weighted_cluster_count",
            "weighted_largest_cluster_share",
        ]
    ].round(3)
)

plot_dbscan_calibration_behavior_for_feature_set(
    dbscan_calibration_behavior_df,
    "taxi",
).show()

,min_samples,eps_band,eps_value,weighted_noise_share,weighted_cluster_count,weighted_largest_cluster_share
0,10,tight,3.951,0.037,3.310,0.959
1,10,balanced,4.863,0.019,2.012,0.965
2,10,loose,7.638,0.005,1.601,0.996
3,15,tight,4.233,0.036,2.392,0.963
4,15,balanced,5.281,0.018,1.787,0.966
5,15,loose,8.301,0.006,1.156,0.998
6,20,tight,4.458,0.033,1.989,0.966
7,20,balanced,5.719,0.015,1.563,0.968
8,20,loose,9.173,0.005,1.156,0.998
9,30,tight,4.458,0.042,1.185,0.970


Findings\. Taxi is the clearest near\-collapse case: once eps reaches the balanced band for min\_samples \>= 15, cluster count is essentially at one and largest\-cluster share is effectively 1\.0\. The practical implication is that Taxi offers only a narrow viable DBSCAN window, concentrated around the tight settings and possibly balanced with min\_samples = 10, while most of the rest of the grid looks too loose to preserve meaningful structure\.

FHVHV calibration behavior\. FHVHV sits in the middle of the density\-readiness story, so the question here is whether the calibration sweep reveals a usable middle region or whether the candidate settings still move too quickly toward either high anomaly volume or near\-complete dominance\.

In [30]:
# ---------------------------------------------------------------------
# Summarize FHVHV calibration behavior
# ---------------------------------------------------------------------

fhvhv_calibration_df = (
    dbscan_calibration_behavior_df.loc[
        dbscan_calibration_behavior_df["feature_set"].eq("fhvhv")
    ]
    .copy()
    .sort_values(["min_samples", "eps_band"])
)

fhvhv_calibration_df["eps_band"] = pd.Categorical(
    bus_calibration_df["eps_band"],
    categories=["tight", "balanced", "loose"],
    ordered=True,
)
fhvhv_calibration_df = bus_calibration_df.sort_values(["min_samples", "eps_band"])

display(
    fhvhv_calibration_df[
        [
            "min_samples",
            "eps_band",
            "eps_value",
            "weighted_noise_share",
            "weighted_cluster_count",
            "weighted_largest_cluster_share",
        ]
    ].round(3)
)

plot_dbscan_calibration_behavior_for_feature_set(
    dbscan_calibration_behavior_df,
    "fhvhv",
).show()

,min_samples,eps_band,eps_value,weighted_noise_share,weighted_cluster_count,weighted_largest_cluster_share
0,10,tight,3.951,0.037,3.310,0.959
1,10,balanced,4.863,0.019,2.012,0.965
2,10,loose,7.638,0.005,1.601,0.996
3,15,tight,4.233,0.036,2.392,0.963
4,15,balanced,5.281,0.018,1.787,0.966
5,15,loose,8.301,0.006,1.156,0.998
6,20,tight,4.458,0.033,1.989,0.966
7,20,balanced,5.719,0.015,1.563,0.968
8,20,loose,9.173,0.005,1.156,0.998
9,30,tight,4.458,0.042,1.185,0.970


Findings\. FHVHV behaves a bit more gently than Taxi but still trends quickly toward collapse as eps loosens, especially once min\_samples rises above 10\. The best\-looking region is again on the tighter side, with balanced settings still plausible at lower min\_samples, but the loose settings are mostly too permissive to keep useful anomaly structure alive\.

Multimodal calibration behavior\. Multimodal preserves richer structure than several of the single\-system branches, but it also operates on a broader PCA distance scale\. This block checks whether that richer structure survives only at very tight settings or whether there is still a reasonable middle region worth carrying into stability testing\.

In [31]:
# ---------------------------------------------------------------------
# Summarize Multimodal calibration behavior
# ---------------------------------------------------------------------

multimodal_calibration_df = (
    dbscan_calibration_behavior_df.loc[
        dbscan_calibration_behavior_df["feature_set"].eq("multimodal")
    ]
    .copy()
    .sort_values(["min_samples", "eps_band"])
)

multimodal_calibration_df["eps_band"] = pd.Categorical(
    bus_calibration_df["eps_band"],
    categories=["tight", "balanced", "loose"],
    ordered=True,
)
multimodal_calibration_df = bus_calibration_df.sort_values(["min_samples", "eps_band"])

display(
    multimodal_calibration_df[
        [
            "min_samples",
            "eps_band",
            "eps_value",
            "weighted_noise_share",
            "weighted_cluster_count",
            "weighted_largest_cluster_share",
        ]
    ].round(3)
)

plot_dbscan_calibration_behavior_for_feature_set(
    dbscan_calibration_behavior_df,
    "multimodal",
).show()

,min_samples,eps_band,eps_value,weighted_noise_share,weighted_cluster_count,weighted_largest_cluster_share
0,10,tight,3.951,0.037,3.310,0.959
1,10,balanced,4.863,0.019,2.012,0.965
2,10,loose,7.638,0.005,1.601,0.996
3,15,tight,4.233,0.036,2.392,0.963
4,15,balanced,5.281,0.018,1.787,0.966
5,15,loose,8.301,0.006,1.156,0.998
6,20,tight,4.458,0.033,1.989,0.966
7,20,balanced,5.719,0.015,1.563,0.968
8,20,loose,9.173,0.005,1.156,0.998
9,30,tight,4.458,0.042,1.185,0.970


Findings\. Multimodal preserves the richest cluster structure under tighter settings, especially at min\_samples = 10, but it also approaches very high dominant\-cluster share as soon as eps loosens\. That makes it one of the stronger candidates for DBSCAN in principle, but only within a relatively tight calibration region; the practical sweet spot looks to be tight or possibly low\-min\_samples balanced, not the loose end of the grid\.

Visualize calibration tradeoffs and identify plausible regions\. Now let’s turn the sweep into a transparent screening rule that separates three cases instead of two: settings that should advance, settings that are locally plausible but retired to keep the downstream search space tighter, and settings that look weak on behavior alone\. This gives us a more honest decision surface: the chart reflects both what the sweep says empirically and the sharper pruning choices we want to make before stability testing\.

In [32]:
# ---------------------------------------------------------------------
# Score candidate settings with a transparent calibration screen
# ---------------------------------------------------------------------

DBSCAN_CALIBRATION_SCREEN_SUMMARY_PATH = (
    INTERMEDIATE_332_DIR / "dbscan_calibration_screen_summary.parquet"
)

dbscan_calibration_screen_df = dbscan_calibration_behavior_df.copy()

# ---------------------------------------------------------------
# Behavioral screen:
# - very low noise share usually means eps is too loose
# - very high noise share usually means eps is too tight
# - near-single-cluster collapse is not useful for anomaly work
# - very high cluster counts suggest fragmentation
# ---------------------------------------------------------------
dbscan_calibration_screen_df["anomaly_volume_status"] = np.select(
    [
        dbscan_calibration_screen_df["weighted_noise_share"] < 0.005,
        dbscan_calibration_screen_df["weighted_noise_share"] > 0.060,
    ],
    [
        "too_loose_low_volume",
        "too_tight_high_volume",
    ],
    default="plausible",
)

dbscan_calibration_screen_df["cluster_structure_status"] = np.select(
    [
        (
            dbscan_calibration_screen_df["weighted_cluster_count"] <= 1.10
        )
        & (
            dbscan_calibration_screen_df["weighted_largest_cluster_share"] >= 0.995
        ),
        dbscan_calibration_screen_df["weighted_cluster_count"] >= 4.50,
    ],
    [
        "collapsed_single_cluster",
        "fragmented",
    ],
    default="plausible",
)

dbscan_calibration_screen_df["behaviorally_plausible"] = (
    dbscan_calibration_screen_df["anomaly_volume_status"].eq("plausible")
    & dbscan_calibration_screen_df["cluster_structure_status"].eq("plausible")
)

# ---------------------------------------------------------------
# Notebook-level narrowing policy:
# - retire all loose settings
# - retire min_samples = 30
# - for Taxi, retire balanced as well
# ---------------------------------------------------------------
dbscan_calibration_screen_df["policy_status"] = np.select(
    [
        dbscan_calibration_screen_df["eps_band"].eq("loose"),
        dbscan_calibration_screen_df["min_samples"].eq(30),
        dbscan_calibration_screen_df["feature_set"].eq("taxi")
        & dbscan_calibration_screen_df["eps_band"].eq("balanced"),
    ],
    [
        "retired_loose_band",
        "retired_min_samples_30",
        "retired_taxi_balanced",
    ],
    default="eligible",
)

# Final status:
# - advance: behaviorally plausible and policy-eligible
# - retired: behaviorally plausible but intentionally removed from search space
# - reject: behaviorally weak
dbscan_calibration_screen_df["screen_decision"] = np.select(
    [
        dbscan_calibration_screen_df["behaviorally_plausible"]
        & dbscan_calibration_screen_df["policy_status"].eq("eligible"),
        dbscan_calibration_screen_df["behaviorally_plausible"]
        & ~dbscan_calibration_screen_df["policy_status"].eq("eligible"),
    ],
    [
        "advance",
        "retired_by_policy",
    ],
    default="reject",
)

dbscan_calibration_screen_df["advance_to_stability"] = (
    dbscan_calibration_screen_df["screen_decision"].eq("advance")
)

if should_rebuild(REBUILD_DBSCAN_PARAMETER_CALIBRATION, DBSCAN_CALIBRATION_SCREEN_SUMMARY_PATH):
    dbscan_calibration_screen_df.to_parquet(
        DBSCAN_CALIBRATION_SCREEN_SUMMARY_PATH,
        index=False,
    )

float_columns = dbscan_calibration_screen_df.select_dtypes(
    include="float"
).columns

dbscan_calibration_screen_df[float_columns] = (
    dbscan_calibration_screen_df[float_columns]
    .round(3)
)

display(
    dbscan_calibration_screen_df[
        [
            "feature_set",
            "min_samples",
            "eps_band",
            "eps_value",
            "weighted_noise_share",
            "weighted_cluster_count",
            "weighted_largest_cluster_share",
            "anomaly_volume_status",
            "cluster_structure_status",
            "policy_status",
            "screen_decision",
        ]
    ]
)

# ---------------------------------------------------------------------
# Visualize candidate-region screening by modality
# ---------------------------------------------------------------------

if "dbscan_calibration_behavior_df" not in globals():
    dbscan_calibration_behavior_df = pd.read_parquet(DBSCAN_PARAMETER_GRID_SUMMARY_PATH)

screen_source_df = dbscan_calibration_behavior_df.copy()

if "representation_name" in screen_source_df.columns:
    screen_source_df = screen_source_df.loc[
        screen_source_df["representation_name"].eq("pca_80pct")
    ].copy()

screen_source_df["setting_label"] = (
    "min_samples="
    + screen_source_df["min_samples"].astype(int).astype(str)
    + " | "
    + screen_source_df["eps_band"].astype(str)
)

screen_source_df["noise_score_raw"] = -screen_source_df["weighted_noise_share"]
screen_source_df["cluster_score_raw"] = screen_source_df["weighted_cluster_count"]
screen_source_df["dominance_score_raw"] = 1 - screen_source_df["weighted_largest_cluster_share"]

score_specs = [
    ("noise_score_raw", "Noise volume"),
    ("cluster_score_raw", "Cluster structure"),
    ("dominance_score_raw", "Less dominance"),
]

screen_plot_records = []

for feature_set_name, feature_df in screen_source_df.groupby("feature_set"):
    feature_df = feature_df.copy()

    for raw_col, metric_label in score_specs:
        min_value = feature_df[raw_col].min()
        max_value = feature_df[raw_col].max()
        denom = max_value - min_value

        feature_df["relative_score"] = (
            1.0
            if denom == 0
            else (feature_df[raw_col] - min_value) / denom
        )

        for _, row in feature_df.iterrows():
            screen_plot_records.append(
                {
                    "feature_set": feature_set_name,
                    "setting_label": row["setting_label"],
                    "metric": metric_label,
                    "relative_score": row["relative_score"],
                }
            )

screen_plot_df = pd.DataFrame(screen_plot_records)

feature_set_layout = {
    "bus": (1, 1),
    "subway": (1, 2),
    "taxi": (2, 1),
    "fhvhv": (2, 2),
    "multimodal": (2, 3),
}

screen_fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=["Bus", "Subway", "", "Taxi", "FHVHV", "Multimodal"],
    horizontal_spacing=0.05,
    vertical_spacing=0.08,
    shared_xaxes=True,
    shared_yaxes=True,
)

for feature_set_name, (row_i, col_i) in feature_set_layout.items():
    feature_plot_df = screen_plot_df.loc[
        screen_plot_df["feature_set"].eq(feature_set_name)
    ].copy()

    matrix_df = feature_plot_df.pivot_table(
        index="setting_label",
        columns="metric",
        values="relative_score",
        aggfunc="first",
    )

    matrix_df = matrix_df[["Noise volume", "Cluster structure", "Less dominance"]]

    screen_fig.add_trace(
        go.Heatmap(
            z=matrix_df.to_numpy(),
            x=matrix_df.columns.tolist(),
            y=matrix_df.index.tolist(),
            zmin=0,
            zmax=1,
            colorscale=BRAND_DIVERGING_SEQUENCE,
            showscale=(feature_set_name == "bus"),
            colorbar={
                "title": {
                    "text": "Relative score",
                    "font": {"color": BRAND_COLORS["dark_teal"]},
                },
                "tickfont": {"color": BRAND_COLORS["dark_teal"]},
                "len": 0.72,
                "y": 0.52,
            } if feature_set_name == "bus" else None,
            text=np.round(matrix_df.to_numpy(), 2),
            texttemplate="%{text:.2f}",
            xgap=4,
            ygap=3,
            hovertemplate=(
                "Setting: %{y}<br>"
                "Metric: %{x}<br>"
                "Relative score: %{z:.2f}<extra></extra>"
            ),
        ),
        row=row_i,
        col=col_i,
    )

screen_fig.update_xaxes(visible=False, row=1, col=3)
screen_fig.update_yaxes(visible=False, row=1, col=3)

apply_brand_plotly_layout(
    screen_fig,
    title="Candidate-Region Screening by Modality",
    width=960,
    height=670,
    margin={"l": 185, "r": 95, "t": 80, "b": 95},
)

screen_fig.update_xaxes(showticklabels=False, title_text="")
screen_fig.update_yaxes(showticklabels=False, title_text="")

for col_i in [1, 2, 3]:
    screen_fig.update_xaxes(
        showticklabels=True,
        tickangle=30,
        title_text="",
        row=2,
        col=col_i,
    )

screen_fig.update_yaxes(showticklabels=True, ticklabelstandoff=4, row=1, col=1)
screen_fig.update_yaxes(showticklabels=True, ticklabelstandoff=4, row=2, col=1)
screen_fig.update_xaxes(visible=False, row=1, col=3)
screen_fig.update_yaxes(visible=False, row=1, col=3)

screen_fig.show()

,feature_set,min_samples,eps_band,eps_value,weighted_noise_share,weighted_cluster_count,weighted_largest_cluster_share,anomaly_volume_status,cluster_structure_status,policy_status,screen_decision
0,bus,10,tight,3.951,0.037,3.310,0.959,plausible,plausible,eligible,advance
1,bus,10,balanced,4.863,0.019,2.012,0.965,plausible,plausible,eligible,advance
2,bus,10,loose,7.638,0.005,1.601,0.996,plausible,plausible,retired_loose_band,retired_by_policy
3,bus,15,tight,4.233,0.036,2.392,0.963,plausible,plausible,eligible,advance
4,bus,15,balanced,5.281,0.018,1.787,0.966,plausible,plausible,eligible,advance
5,bus,15,loose,8.301,0.006,1.156,0.998,plausible,plausible,retired_loose_band,retired_by_policy
6,bus,20,tight,4.458,0.033,1.989,0.966,plausible,plausible,eligible,advance
7,bus,20,balanced,5.719,0.015,1.563,0.968,plausible,plausible,eligible,advance
8,bus,20,loose,9.173,0.005,1.156,0.998,plausible,plausible,retired_loose_band,retired_by_policy
9,bus,30,tight,4.458,0.042,1.185,0.970,plausible,plausible,retired_min_samples_30,retired_by_policy


Findings\. The screening grid now makes three patterns clear\. First, loose settings are weak almost everywhere: they usually drive anomaly volume toward very low levels and collapse cluster structure toward one dominant cluster, so they are no longer worth carrying forward\. Second, min\_samples = 30 is occasionally acceptable in isolation, but it is usually dominated by smaller values and adds more search\-space clutter than practical value, so it is better treated as retired rather than truly competitive\. Third, the surviving structure is modality\-specific: Bus, Subway, FHVHV, and Multimodal still retain plausible tight and balanced regions, while Taxi remains the clearest outlier because even its balanced settings collapse too quickly toward one\-cluster behavior\.

> 🎯 Decision\. The calibration sweep is strong enough to shrink the DBSCAN search space aggressively now\. Loose settings will be retired across the board, since they repeatedly collapse anomaly structure toward near\-single\-cluster behavior with very low anomaly volume\. min\_samples = 30 will also be retired as a shared candidate level, since it is usually dominated by smaller values even when it does not fully fail on its own\. Bus, Subway, FHVHV, and Multimodal will carry forward tight and balanced settings with min\_samples ∈ \{10, 15, 20\}, while Taxi will carry forward only tight settings over that same reduced min\_samples range\. The next section will use stability diagnostics to choose the final retained setting within these narrowed modality\-specific regions\.

Record candidate DBSCAN regions to advance into stability testing\. This final step turns the narrowing decision into an explicit handoff table for the stability section\. At this point, we are no longer just recording whatever passed the raw screening rule; we are recording the smaller modality\-specific candidate set that remains after retiring weak and intentionally pruned regions\.

In [33]:
# ---------------------------------------------------------------------
# Record narrowed DBSCAN candidate regions advancing into stability testing
# ---------------------------------------------------------------------

DBSCAN_STABILITY_CANDIDATE_REGIONS_PATH = (
    INTERMEDIATE_332_DIR / "dbscan_stability_candidate_regions.parquet"
)

dbscan_stability_candidate_regions_df = (
    dbscan_calibration_screen_df.loc[
        dbscan_calibration_screen_df["screen_decision"].eq("advance")
    ]
    .copy()
    .sort_values(["feature_set", "min_samples", "eps_band"])
    .reset_index(drop=True)
)

dbscan_stability_candidate_region_inventory_df = (
    dbscan_stability_candidate_regions_df.groupby("feature_set", dropna=False)
    .agg(
        retained_candidates=("eps_value", "size"),
        retained_min_samples=("min_samples", lambda s: ", ".join(map(str, sorted(s.unique())))),
        retained_eps_bands=("eps_band", lambda s: ", ".join(pd.Index(s).drop_duplicates().tolist())),
    )
    .reset_index()
)

if should_rebuild(REBUILD_DBSCAN_PARAMETER_CALIBRATION, DBSCAN_STABILITY_CANDIDATE_REGIONS_PATH):
    dbscan_stability_candidate_regions_df.to_parquet(
        DBSCAN_STABILITY_CANDIDATE_REGIONS_PATH,
        index=False,
    )

display(dbscan_stability_candidate_region_inventory_df)
display(
    dbscan_stability_candidate_regions_df[
        [
            "feature_set",
            "min_samples",
            "eps_band",
            "eps_value",
            "weighted_noise_share",
            "weighted_cluster_count",
            "weighted_largest_cluster_share",
            "screen_decision",
        ]
    ]
)

,feature_set,retained_candidates,retained_min_samples,retained_eps_bands
0,bus,6,"10, 15, 20","tight, balanced"
1,fhvhv,6,"10, 15, 20","tight, balanced"
2,multimodal,6,"10, 15, 20","tight, balanced"
3,subway,6,"10, 15, 20","tight, balanced"
4,taxi,3,"10, 15, 20",tight


,feature_set,min_samples,eps_band,eps_value,weighted_noise_share,weighted_cluster_count,weighted_largest_cluster_share,screen_decision
0,bus,10,tight,3.951,0.037,3.310,0.959,advance
1,bus,10,balanced,4.863,0.019,2.012,0.965,advance
2,bus,15,tight,4.233,0.036,2.392,0.963,advance
3,bus,15,balanced,5.281,0.018,1.787,0.966,advance
4,bus,20,tight,4.458,0.033,1.989,0.966,advance
5,bus,20,balanced,5.719,0.015,1.563,0.968,advance
6,fhvhv,10,tight,3.397,0.048,2.303,0.959,advance
7,fhvhv,10,balanced,4.598,0.023,2.174,0.961,advance
8,fhvhv,15,tight,3.679,0.047,1.816,0.961,advance
9,fhvhv,15,balanced,5.116,0.021,1.929,0.962,advance


Findings\. The narrowed handoff set retains 27 candidate DBSCAN settings across the five modalities, and the pattern matches the intended pruning logic: Bus, Subway, FHVHV, and Multimodal carry forward both tight and balanced settings for min\_samples ∈ \{10, 15, 20\}, while Taxi carries forward only tight settings over that same reduced min\_samples range\. The practical result is a much smaller and cleaner stability search space that still preserves the main modality\-specific tradeoffs surfaced in the calibration sweep\.

In [34]:
# ---------------------------------------------------------------------
# Validate candidate-region handoff into stability testing
# ---------------------------------------------------------------------

dbscan_stability_candidate_region_validation_df = pd.DataFrame(
    [
        {
            "feature_sets_with_candidates": dbscan_stability_candidate_regions_df["feature_set"].nunique(),
            "expected_feature_sets": len(MODEL_FEATURE_SET_NAMES),
            "retained_candidate_rows": len(dbscan_stability_candidate_regions_df),
            "status": (
                "pass"
                if dbscan_stability_candidate_regions_df["feature_set"].nunique()
                == len(MODEL_FEATURE_SET_NAMES)
                else "review"
            ),
        }
    ]
)

display(dbscan_stability_candidate_region_validation_df)

assert dbscan_stability_candidate_region_validation_df["status"].eq("pass").all(), (
    "One or more modalities has no candidate DBSCAN region advancing into stability testing."
)

,feature_sets_with_candidates,expected_feature_sets,retained_candidate_rows,status
0,5,5,27,pass


Findings\. The handoff validation passed cleanly: all five modalities retain at least one advancing candidate region, and the narrowed stability set contains 27 total candidate settings\.

### Section Summary

This section turned the PCA\-only calibration sweep into a smaller and more actionable DBSCAN search space\. The sweep behaved consistently across modalities: tighter eps settings produced higher anomaly volume and richer cluster structure, while looser settings repeatedly pushed the surfaces toward low anomaly prevalence and near\-single\-cluster dominance\. That made it possible to prune aggressively\. Loose settings were retired across the board, min\_samples = 30 was dropped as a shared candidate level, and the retained space was narrowed to tight and balanced settings with min\_samples ∈ \{10, 15, 20\} for Bus, Subway, FHVHV, and Multimodal, with Taxi restricted to tight settings only\. The output of the section is therefore not a final DBSCAN choice yet, but a much cleaner, modality\-specific candidate set that is ready for direct stability testing in the next section\.

## 3\.3\.2\.4 Evaluate DBSCAN Stability

This section tests whether the narrowed DBSCAN candidate regions from 3\.3\.2\.3 are genuinely robust or only look plausible in isolated calibration snapshots\. We’ll compare nearby retained settings within each modality and ask whether small parameter changes preserve similar anomaly volume, cluster structure, and anomaly\-like row assignments\.

The goal is to move from “these settings look reasonable” to “this modality has a stable DBSCAN region we can trust\.” By the end of the section, we want one final DBSCAN configuration per modality: a setting that not only behaves plausibly in calibration, but also remains reasonably consistent under small perturbations inside the retained search space\.

### Run a row\-level stability comparison within each modality

Now that the candidate space has been narrowed, we can inspect stability at the row level\. This part materializes anomaly\-like assignments for each retained DBSCAN setting inside each modality so that the next section can compare neighboring settings directly on the same observations, rather than only comparing aggregate calibration summaries\.

This is one of the few places where it is worth writing intermediate outputs\. Each retained candidate gets its own row\-level parquet so the block can resume cleanly after interruption, and so later stability comparisons can reuse those exact anomaly surfaces without rerunning DBSCAN\.

In [35]:
# ---------------------------------------------------------------------
# Materialize row-level anomaly assignments for retained DBSCAN candidates
# ---------------------------------------------------------------------

DBSCAN_STABILITY_ROW_LEVEL_DIR = INTERMEDIATE_332_DIR / "dbscan_stability_row_level_parts"
DBSCAN_STABILITY_ROW_LEVEL_DIR.mkdir(parents=True, exist_ok=True)

DBSCAN_STABILITY_ROW_LEVEL_MANIFEST_PATH = (
    INTERMEDIATE_332_DIR / "dbscan_stability_row_level_manifest.parquet"
)

# We are inheriting the already-selected candidate regions from 3.3.2.3.
assert "dbscan_stability_candidate_regions_df" in globals(), (
    "dbscan_stability_candidate_regions_df is not in memory. "
    "Please run the final handoff block from 3.3.2.3 first."
)

dbscan_stability_row_level_candidates_df = (
    dbscan_stability_candidate_regions_df.copy()
    .sort_values(["feature_set", "min_samples", "eps_band"])
    .reset_index(drop=True)
)

# The narrowed search space should already be PCA-only at this point.
if "representation_name" in dbscan_stability_row_level_candidates_df.columns:
    assert dbscan_stability_row_level_candidates_df["representation_name"].eq("pca_80pct").all(), (
        "Stability handoff unexpectedly includes non-PCA candidates."
    )
else:
    dbscan_stability_row_level_candidates_df["representation_name"] = "pca_80pct"

dbscan_stability_row_level_candidates_df["eps_band"] = (
    dbscan_stability_row_level_candidates_df["eps_band"].astype(str)
)

dbscan_stability_row_level_candidates_df["candidate_id"] = (
    dbscan_stability_row_level_candidates_df["feature_set"].astype(str)
    + "__"
    + dbscan_stability_row_level_candidates_df["representation_name"].astype(str)
    + "__ms"
    + dbscan_stability_row_level_candidates_df["min_samples"].astype(str)
    + "__"
    + dbscan_stability_row_level_candidates_df["eps_band"].astype(str)
)

display(
    dbscan_stability_row_level_candidates_df[
        [
            "feature_set",
            "representation_name",
            "min_samples",
            "eps_band",
            "eps_value",
        ]
    ].round(3)
)

def get_stability_pca_representation_df(feature_set_name: str, metadata_df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """
    Build the row-aligned PCA representation frame for one modality.
    Taxi is the only special case because 3.3.1 carried forward split pre/post PCA.
    """
    if feature_set_name == "taxi" and TAXI_PCA_HANDLING_DECISION == "split_prepost_period":
        pre_metadata_df = metadata_df.loc[
            metadata_df[PRE_POST_CP_COLUMN].eq("pre_cp")
        ].copy()
        post_metadata_df = metadata_df.loc[
            metadata_df[PRE_POST_CP_COLUMN].eq("post_cp")
        ].copy()

        pre_pca_df = load_representation_rows(
            feature_set_name="taxi",
            representation_name="pca_80pct",
            pre_post_cp="pre_cp",
        )
        post_pca_df = load_representation_rows(
            feature_set_name="taxi",
            representation_name="pca_80pct",
            pre_post_cp="post_cp",
        )

        representation_columns = [
            column_name
            for column_name in pre_pca_df.columns
            if column_name != MODEL_ID_COLUMN
        ]

        pca_df = pd.concat(
            [
                pre_metadata_df[[MODEL_ID_COLUMN]].merge(pre_pca_df, on=MODEL_ID_COLUMN, how="inner"),
                post_metadata_df[[MODEL_ID_COLUMN]].merge(post_pca_df, on=MODEL_ID_COLUMN, how="inner"),
            ],
            ignore_index=True,
        )
        return pca_df, representation_columns

    pca_df = load_representation_rows(
        feature_set_name=feature_set_name,
        representation_name="pca_80pct",
    )
    representation_columns = [
        column_name
        for column_name in pca_df.columns
        if column_name != MODEL_ID_COLUMN
    ]
    return pca_df, representation_columns

def build_shared_dbscan_group_key(frame: pd.DataFrame) -> pd.Series:
    # 3.3.1 settled the shared DBSCAN comparison universe as
    # temporal bucket + policy geography.
    return (
        frame[TEMPORAL_BUCKET_COLUMN].astype(str)
        + " | "
        + frame[POLICY_GEOGRAPHY_COLUMN].astype(str)
    )

manifest_records = []
total_candidates = len(dbscan_stability_row_level_candidates_df)

for candidate_index, candidate_row in enumerate(
    dbscan_stability_row_level_candidates_df.itertuples(index=False),
    start=1,
):
    candidate_part_path = DBSCAN_STABILITY_ROW_LEVEL_DIR / f"{candidate_row.candidate_id}.parquet"

    if candidate_part_path.exists() and not REBUILD_DBSCAN_STABILITY_DIAGNOSTICS:
        existing_df = pd.read_parquet(candidate_part_path)

        manifest_records.append(
            {
                "feature_set": candidate_row.feature_set,
                "representation_name": candidate_row.representation_name,
                "min_samples": int(candidate_row.min_samples),
                "eps_band": candidate_row.eps_band,
                "eps_value": float(candidate_row.eps_value),
                "rows_written": len(existing_df),
                "groups_present": existing_df["comparison_group_key"].nunique(),
                "anomaly_like_rows": int(existing_df["is_anomaly_like"].sum()),
                "status": "loaded_existing",
            }
        )

        print(
            f"[{candidate_index}/{total_candidates}] Loaded existing row-level stability output "
            f"for {candidate_row.feature_set} | min_samples={candidate_row.min_samples} | "
            f"eps_band={candidate_row.eps_band}"
        )
        continue

    print(
        f"[{candidate_index}/{total_candidates}] Materializing row-level stability output "
        f"for {candidate_row.feature_set} | min_samples={candidate_row.min_samples} | "
        f"eps_band={candidate_row.eps_band}"
    )

    metadata_df = load_calibration_metadata(candidate_row.feature_set)
    pca_df, representation_columns = get_stability_pca_representation_df(
        candidate_row.feature_set,
        metadata_df,
    )

    panel_df = metadata_df.merge(
        pca_df,
        on=MODEL_ID_COLUMN,
        how="inner",
    ).copy()

    panel_df["comparison_group_key"] = build_shared_dbscan_group_key(panel_df)

    # This guards against stale or malformed inherited representation branches.
    if panel_df[representation_columns].isna().any().any():
        bad_columns = (
            panel_df[representation_columns]
            .isna()
            .sum()
            .loc[lambda s: s.gt(0)]
            .index.tolist()
        )
        raise AssertionError(
            f"Missing PCA representation values remain for {candidate_row.feature_set}: {bad_columns}"
        )

    candidate_group_outputs = []

    for comparison_group_key, group_df in panel_df.groupby(
        "comparison_group_key",
        dropna=False,
        sort=True,
    ):
        X = group_df[representation_columns].to_numpy(dtype=np.float32, copy=False)

        dbscan_model = DBSCAN(
            eps=float(candidate_row.eps_value),
            min_samples=int(candidate_row.min_samples),
            metric="euclidean",
        )
        cluster_labels = dbscan_model.fit_predict(X)

        group_output_df = group_df[
            [
                MODEL_ID_COLUMN,
                DATE_COLUMN,
                TEMPORAL_BUCKET_COLUMN,
                PRE_POST_CP_COLUMN,
                ZONE_ID_COLUMN,
                ZONE_COLUMN,
                BOROUGH_COLUMN,
                POLICY_GEOGRAPHY_COLUMN,
                "comparison_group_key",
            ]
        ].copy()

        group_output_df["feature_set"] = candidate_row.feature_set
        group_output_df["representation_name"] = candidate_row.representation_name
        group_output_df["min_samples"] = int(candidate_row.min_samples)
        group_output_df["eps_band"] = candidate_row.eps_band
        group_output_df["eps_value"] = float(candidate_row.eps_value)
        group_output_df["candidate_id"] = candidate_row.candidate_id
        group_output_df["cluster_label"] = cluster_labels.astype(int)
        group_output_df["is_anomaly_like"] = group_output_df["cluster_label"].eq(-1)

        candidate_group_outputs.append(group_output_df)

    candidate_row_level_df = pd.concat(candidate_group_outputs, ignore_index=True)
    candidate_row_level_df.to_parquet(candidate_part_path, index=False)

    manifest_records.append(
        {
            "feature_set": candidate_row.feature_set,
            "representation_name": candidate_row.representation_name,
            "min_samples": int(candidate_row.min_samples),
            "eps_band": candidate_row.eps_band,
            "eps_value": float(candidate_row.eps_value),
            "rows_written": len(candidate_row_level_df),
            "groups_present": candidate_row_level_df["comparison_group_key"].nunique(),
            "anomaly_like_rows": int(candidate_row_level_df["is_anomaly_like"].sum()),
            "status": "written",
        }
    )

dbscan_stability_row_level_manifest_df = (
    pd.DataFrame(manifest_records)
    .sort_values(["feature_set", "min_samples", "eps_band"])
    .reset_index(drop=True)
)

if should_rebuild(REBUILD_DBSCAN_STABILITY_DIAGNOSTICS, DBSCAN_STABILITY_ROW_LEVEL_MANIFEST_PATH):
    dbscan_stability_row_level_manifest_df.to_parquet(
        DBSCAN_STABILITY_ROW_LEVEL_MANIFEST_PATH,
        index=False,
    )

display(dbscan_stability_row_level_manifest_df.round(3))

,feature_set,representation_name,min_samples,eps_band,eps_value
0,bus,pca_80pct,10,tight,3.951
1,bus,pca_80pct,10,balanced,4.863
2,bus,pca_80pct,15,tight,4.233
3,bus,pca_80pct,15,balanced,5.281
4,bus,pca_80pct,20,tight,4.458
5,bus,pca_80pct,20,balanced,5.719
6,fhvhv,pca_80pct,10,tight,3.397
7,fhvhv,pca_80pct,10,balanced,4.598
8,fhvhv,pca_80pct,15,tight,3.679
9,fhvhv,pca_80pct,15,balanced,5.116


[1/27] Loaded existing row-level stability output for bus | min_samples=10 | eps_band=tight
[2/27] Loaded existing row-level stability output for bus | min_samples=10 | eps_band=balanced
[3/27] Loaded existing row-level stability output for bus | min_samples=15 | eps_band=tight
[4/27] Loaded existing row-level stability output for bus | min_samples=15 | eps_band=balanced
[5/27] Loaded existing row-level stability output for bus | min_samples=20 | eps_band=tight
[6/27] Loaded existing row-level stability output for bus | min_samples=20 | eps_band=balanced
[7/27] Loaded existing row-level stability output for fhvhv | min_samples=10 | eps_band=tight
[8/27] Loaded existing row-level stability output for fhvhv | min_samples=10 | eps_band=balanced
[9/27] Loaded existing row-level stability output for fhvhv | min_samples=15 | eps_band=tight
[10/27] Loaded existing row-level stability output for fhvhv | min_samples=15 | eps_band=balanced
[11/27] Loaded existing row-level stability output for f

,feature_set,representation_name,min_samples,eps_band,eps_value,rows_written,groups_present,anomaly_like_rows,status
0,bus,pca_80pct,10,balanced,4.863,50000,30,973,loaded_existing
1,bus,pca_80pct,10,tight,3.951,50000,30,1834,loaded_existing
2,bus,pca_80pct,15,balanced,5.281,50000,30,885,loaded_existing
3,bus,pca_80pct,15,tight,4.233,50000,30,1783,loaded_existing
4,bus,pca_80pct,20,balanced,5.719,50000,30,765,loaded_existing
5,bus,pca_80pct,20,tight,4.458,50000,30,1658,loaded_existing
6,fhvhv,pca_80pct,10,balanced,4.598,50000,30,1127,loaded_existing
7,fhvhv,pca_80pct,10,tight,3.397,50000,30,2425,loaded_existing
8,fhvhv,pca_80pct,15,balanced,5.116,50000,30,1026,loaded_existing
9,fhvhv,pca_80pct,15,tight,3.679,50000,30,2347,loaded_existing


Findings\. The retained PCA candidate settings were materialized into row\-level anomaly surfaces, giving us a restartable stability panel for direct candidate\-to\-candidate comparisons inside each modality\.

Now let’s verify that the row\-level outputs really match the retained handoff set\. This check makes sure every candidate wrote the expected number of rows, preserved the full set of comparison groups, and stayed aligned to the shared calibration panel\.

In [36]:
# ---------------------------------------------------------------------
# Validate row-level stability outputs
# ---------------------------------------------------------------------

stability_validation_records = []

for candidate_row in dbscan_stability_row_level_candidates_df.itertuples(index=False):
    candidate_part_path = DBSCAN_STABILITY_ROW_LEVEL_DIR / f"{candidate_row.candidate_id}.parquet"
    candidate_output_df = pd.read_parquet(candidate_part_path)

    metadata_df = load_calibration_metadata(candidate_row.feature_set)
    expected_group_count = (
        metadata_df.assign(
            comparison_group_key=build_shared_dbscan_group_key(metadata_df)
        )["comparison_group_key"]
        .nunique()
    )

    stability_validation_records.append(
        {
            "feature_set": candidate_row.feature_set,
            "representation_name": candidate_row.representation_name,
            "min_samples": int(candidate_row.min_samples),
            "eps_band": candidate_row.eps_band,
            "row_level_rows": len(candidate_output_df),
            "distinct_modeling_rows": candidate_output_df[MODEL_ID_COLUMN].nunique(),
            "groups_present": candidate_output_df["comparison_group_key"].nunique(),
            "unexpected_group_rows": int(
                (~candidate_output_df["comparison_group_key"].isin(
                    metadata_df.assign(
                        comparison_group_key=build_shared_dbscan_group_key(metadata_df)
                    )["comparison_group_key"]
                )).sum()
            ),
            "expected_groups": expected_group_count,
            "anomaly_like_rows": int(candidate_output_df["is_anomaly_like"].sum()),
            "status": (
                "pass"
                if len(candidate_output_df) == len(metadata_df)
                and candidate_output_df[MODEL_ID_COLUMN].nunique() == len(metadata_df)
                and candidate_output_df["comparison_group_key"].nunique() == expected_group_count
                else "review"
            ),
        }
    )

dbscan_stability_row_level_validation_df = pd.DataFrame(
    stability_validation_records
).sort_values(["feature_set", "min_samples", "eps_band"]).reset_index(drop=True)

display(dbscan_stability_row_level_validation_df)

assert dbscan_stability_row_level_validation_df["status"].eq("pass").all(), (
    "One or more row-level stability outputs is missing rows, duplicated rows, or dropped comparison groups."
)

,feature_set,representation_name,min_samples,eps_band,row_level_rows,distinct_modeling_rows,groups_present,unexpected_group_rows,expected_groups,anomaly_like_rows,status
0,bus,pca_80pct,10,balanced,50000,50000,30,0,30,973,pass
1,bus,pca_80pct,10,tight,50000,50000,30,0,30,1834,pass
2,bus,pca_80pct,15,balanced,50000,50000,30,0,30,885,pass
3,bus,pca_80pct,15,tight,50000,50000,30,0,30,1783,pass
4,bus,pca_80pct,20,balanced,50000,50000,30,0,30,765,pass
5,bus,pca_80pct,20,tight,50000,50000,30,0,30,1658,pass
6,fhvhv,pca_80pct,10,balanced,50000,50000,30,0,30,1127,pass
7,fhvhv,pca_80pct,10,tight,50000,50000,30,0,30,2425,pass
8,fhvhv,pca_80pct,15,balanced,50000,50000,30,0,30,1026,pass
9,fhvhv,pca_80pct,15,tight,50000,50000,30,0,30,2347,pass


Findings\. The row\-level stability outputs validate cleanly, so every retained candidate is now aligned to the same calibration rows and the same 30 temporal\-by\-policy\-geography comparison groups within its modality\.

### Compare neighboring retained settings within each modality

Now that each retained candidate has been materialized as a row\-level anomaly surface, we can compare nearby settings directly on the same calibration rows\. This is where stability becomes concrete: instead of asking whether two settings merely produce similar aggregate counts, we ask whether they flag largely the same observations and whether their anomaly volume and cluster structure move only modestly when the parameters are nudged\.

This block defines “neighboring” settings conservatively\. Within each modality, we only compare one\-step moves: either a shift from tight to balanced at the same min\_samples, or a one\-level change in min\_samples while staying in the same eps\_band\. That keeps the stability test local, which is exactly what we want here\.

In [37]:
# ---------------------------------------------------------------------
# Compare neighboring retained settings within each modality
# ---------------------------------------------------------------------

DBSCAN_STABILITY_PAIRWISE_SUMMARY_PATH = (
    INTERMEDIATE_332_DIR / "dbscan_stability_pairwise_summary.parquet"
)

assert "dbscan_stability_row_level_candidates_df" in globals(), (
    "dbscan_stability_row_level_candidates_df is not in memory. "
    "Please run the row-level stability materialization block first."
)

assert "dbscan_calibration_behavior_df" in globals(), (
    "dbscan_calibration_behavior_df is not in memory. "
    "Please run the calibration-behavior block from 3.3.2.3 first."
)

eps_band_order = {"tight": 0, "balanced": 1, "loose": 2}
min_samples_order = [10, 15, 20, 30]

def candidate_sort_key(candidate_row: pd.Series) -> tuple:
    return (
        candidate_row["feature_set"],
        candidate_row["representation_name"],
        int(candidate_row["min_samples"]),
        eps_band_order[str(candidate_row["eps_band"])],
    )

def are_neighboring_candidates(left_row: pd.Series, right_row: pd.Series) -> bool:
    if left_row["feature_set"] != right_row["feature_set"]:
        return False
    if left_row["representation_name"] != right_row["representation_name"]:
        return False

    left_eps_band = str(left_row["eps_band"])
    right_eps_band = str(right_row["eps_band"])
    left_min_samples = int(left_row["min_samples"])
    right_min_samples = int(right_row["min_samples"])

    same_min_samples = left_min_samples == right_min_samples
    same_eps_band = left_eps_band == right_eps_band

    # One-step move in eps band while keeping min_samples fixed.
    if same_min_samples:
        return abs(eps_band_order[left_eps_band] - eps_band_order[right_eps_band]) == 1

    # One-step move in min_samples while keeping eps band fixed.
    if same_eps_band:
        left_ix = min_samples_order.index(left_min_samples)
        right_ix = min_samples_order.index(right_min_samples)
        return abs(left_ix - right_ix) == 1

    return False

# Pull in the calibration metrics so the pairwise comparison can measure
# both row-level overlap and higher-level behavior shifts side by side.
candidate_metric_lookup_df = (
    dbscan_calibration_behavior_df.loc[
        dbscan_calibration_behavior_df["representation_name"].eq("pca_80pct")
    ][
        [
            "feature_set",
            "representation_name",
            "min_samples",
            "eps_band",
            "weighted_noise_share",
            "weighted_cluster_count",
            "weighted_largest_cluster_share",
        ]
    ]
    .copy()
)

candidate_metric_lookup_df["eps_band"] = candidate_metric_lookup_df["eps_band"].astype(str)

candidate_metric_lookup = {
    (
        row.feature_set,
        row.representation_name,
        int(row.min_samples),
        str(row.eps_band),
    ): {
        "weighted_noise_share": float(row.weighted_noise_share),
        "weighted_cluster_count": float(row.weighted_cluster_count),
        "weighted_largest_cluster_share": float(row.weighted_largest_cluster_share),
    }
    for row in candidate_metric_lookup_df.itertuples(index=False)
}

stability_candidate_rows = (
    dbscan_stability_row_level_candidates_df.copy()
    .sort_values(["feature_set", "representation_name", "min_samples", "eps_band"])
    .reset_index(drop=True)
)

pairwise_stability_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    feature_candidate_df = (
        stability_candidate_rows.loc[
            stability_candidate_rows["feature_set"].eq(feature_set_name)
        ]
        .copy()
        .sort_values(["representation_name", "min_samples", "eps_band"])
        .reset_index(drop=True)
    )

    for left_idx in range(len(feature_candidate_df)):
        for right_idx in range(left_idx + 1, len(feature_candidate_df)):
            left_row = feature_candidate_df.iloc[left_idx]
            right_row = feature_candidate_df.iloc[right_idx]

            if not are_neighboring_candidates(left_row, right_row):
                continue

            left_part_path = DBSCAN_STABILITY_ROW_LEVEL_DIR / f"{left_row['candidate_id']}.parquet"
            right_part_path = DBSCAN_STABILITY_ROW_LEVEL_DIR / f"{right_row['candidate_id']}.parquet"

            left_flags_df = pd.read_parquet(
                left_part_path,
                columns=[MODEL_ID_COLUMN, "is_anomaly_like"],
            ).rename(columns={"is_anomaly_like": "left_is_anomaly_like"})

            right_flags_df = pd.read_parquet(
                right_part_path,
                columns=[MODEL_ID_COLUMN, "is_anomaly_like"],
            ).rename(columns={"is_anomaly_like": "right_is_anomaly_like"})

            overlap_df = left_flags_df.merge(
                right_flags_df,
                on=MODEL_ID_COLUMN,
                how="inner",
            )

            rows_compared = len(overlap_df)
            left_flag_count = int(overlap_df["left_is_anomaly_like"].sum())
            right_flag_count = int(overlap_df["right_is_anomaly_like"].sum())
            shared_flag_rows = int(
                (overlap_df["left_is_anomaly_like"] & overlap_df["right_is_anomaly_like"]).sum()
            )
            union_flag_rows = int(
                (overlap_df["left_is_anomaly_like"] | overlap_df["right_is_anomaly_like"]).sum()
            )
            left_only_rows = int(
                (overlap_df["left_is_anomaly_like"] & ~overlap_df["right_is_anomaly_like"]).sum()
            )
            right_only_rows = int(
                (~overlap_df["left_is_anomaly_like"] & overlap_df["right_is_anomaly_like"]).sum()
            )

            flag_jaccard_similarity = (
                shared_flag_rows / union_flag_rows
                if union_flag_rows > 0
                else np.nan
            )

            left_metrics = candidate_metric_lookup[
                (
                    left_row["feature_set"],
                    left_row["representation_name"],
                    int(left_row["min_samples"]),
                    str(left_row["eps_band"]),
                )
            ]
            right_metrics = candidate_metric_lookup[
                (
                    right_row["feature_set"],
                    right_row["representation_name"],
                    int(right_row["min_samples"]),
                    str(right_row["eps_band"]),
                )
            ]

            pairwise_stability_records.append(
                {
                    "feature_set": feature_set_name,
                    "representation_name": left_row["representation_name"],
                    "left_candidate_id": left_row["candidate_id"],
                    "right_candidate_id": right_row["candidate_id"],
                    "left_min_samples": int(left_row["min_samples"]),
                    "right_min_samples": int(right_row["min_samples"]),
                    "left_eps_band": str(left_row["eps_band"]),
                    "right_eps_band": str(right_row["eps_band"]),
                    "left_eps_value": float(left_row["eps_value"]),
                    "right_eps_value": float(right_row["eps_value"]),
                    "comparison_type": (
                        "eps_band_shift"
                        if int(left_row["min_samples"]) == int(right_row["min_samples"])
                        else "min_samples_shift"
                    ),
                    "rows_compared": rows_compared,
                    "left_flag_count": left_flag_count,
                    "right_flag_count": right_flag_count,
                    "shared_flag_rows": shared_flag_rows,
                    "union_flag_rows": union_flag_rows,
                    "left_only_rows": left_only_rows,
                    "right_only_rows": right_only_rows,
                    "flag_jaccard_similarity": flag_jaccard_similarity,
                    "noise_share_delta": abs(
                        left_metrics["weighted_noise_share"] - right_metrics["weighted_noise_share"]
                    ),
                    "cluster_count_delta": abs(
                        left_metrics["weighted_cluster_count"] - right_metrics["weighted_cluster_count"]
                    ),
                    "largest_cluster_share_delta": abs(
                        left_metrics["weighted_largest_cluster_share"]
                        - right_metrics["weighted_largest_cluster_share"]
                    ),
                    "status": "pass",
                }
            )

dbscan_stability_pairwise_summary_df = (
    pd.DataFrame(pairwise_stability_records)
    .sort_values(
        [
            "feature_set",
            "comparison_type",
            "left_min_samples",
            "left_eps_band",
            "right_min_samples",
            "right_eps_band",
        ]
    )
    .reset_index(drop=True)
)

if should_rebuild(REBUILD_DBSCAN_STABILITY_DIAGNOSTICS, DBSCAN_STABILITY_PAIRWISE_SUMMARY_PATH):
    dbscan_stability_pairwise_summary_df.to_parquet(
        DBSCAN_STABILITY_PAIRWISE_SUMMARY_PATH,
        index=False,
    )

print(
    "Neighboring candidate comparisons are ready. "
    f"Prepared {len(dbscan_stability_pairwise_summary_df)} local stability comparisons "
    "for the next summary block."
)

Neighboring candidate comparisons are ready. Prepared 30 local stability comparisons for the next summary block.


Findings\. The neighboring\-candidate comparisons have now been materialized at the row level\. The next block condenses those local comparisons into a compact modality summary so we can see which kinds of parameter moves behave like small adjustments and which ones materially alter the anomaly surface\.

In [38]:
# ---------------------------------------------------------------------
# Visualize neighboring-setting stability by modality
# ---------------------------------------------------------------------

stability_plot_df = dbscan_stability_pairwise_summary_df.copy()

if "comparison_label" not in stability_plot_df.columns:
    stability_plot_df["comparison_label"] = "local pair"

    if "pair_label" in stability_plot_df.columns:
        pair_label_text = stability_plot_df["pair_label"].astype("string").str.lower()

        eps_pair_mask = (
            pair_label_text.str.contains("balanced", na=False)
            & pair_label_text.str.contains("tight", na=False)
        ).fillna(False).astype(bool)
        min_samples_pair_mask = (
            pair_label_text.str.contains("min_samples", na=False)
        ).fillna(False).astype(bool)

        stability_plot_df.loc[
            eps_pair_mask,
            "comparison_label",
        ] = "eps: balanced vs tight"
        stability_plot_df.loc[
            min_samples_pair_mask,
            "comparison_label",
        ] = "min_samples step"
    else:
        eps_band_columns = [
            column_name
            for column_name in stability_plot_df.columns
            if "eps_band" in column_name
        ]
        min_samples_columns = [
            column_name
            for column_name in stability_plot_df.columns
            if "min_samples" in column_name
        ]

        if len(eps_band_columns) >= 2:
            eps_band_changed = (
                stability_plot_df[eps_band_columns[0]].astype("string")
                != stability_plot_df[eps_band_columns[1]].astype("string")
            ).fillna(False).astype(bool)
            stability_plot_df.loc[
                eps_band_changed,
                "comparison_label",
            ] = "eps: balanced vs tight"

        if len(min_samples_columns) >= 2:
            min_samples_changed = (
                pd.to_numeric(
                    stability_plot_df[min_samples_columns[0]],
                    errors="coerce",
                )
                != pd.to_numeric(
                    stability_plot_df[min_samples_columns[1]],
                    errors="coerce",
                )
            ).fillna(False).astype(bool)
            stability_plot_df.loc[
                min_samples_changed,
                "comparison_label",
            ] = "min_samples step"

feature_order = ["bus", "subway", "taxi", "fhvhv", "multimodal"]
comparison_order = [
    comparison_label
    for comparison_label in [
        "eps: balanced vs tight",
        "min_samples step",
        "local pair",
    ]
    if comparison_label in stability_plot_df["comparison_label"].dropna().unique()
]

stability_fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "Jaccard overlap",
        "Noise-share delta",
        "Cluster-count delta",
        "Largest-cluster-share delta",
    ],
    horizontal_spacing=0.10,
    vertical_spacing=0.10,
    shared_xaxes=True,
)

metric_specs = [
    ("flag_jaccard_similarity", 1, 1),
    ("noise_share_delta", 1, 2),
    ("cluster_count_delta", 2, 1),
    ("largest_cluster_share_delta", 2, 2),
]

color_map = {
    "eps: balanced vs tight": BRAND_COLORS["terracotta"],
    "min_samples step": BRAND_COLORS["dark_teal"],
    "local pair": BRAND_COLORS["seafoam"],
}

for comparison_label in comparison_order:
    subset_df = stability_plot_df.loc[
        stability_plot_df["comparison_label"].eq(comparison_label)
    ].copy()

    subset_df["feature_set"] = pd.Categorical(
        subset_df["feature_set"],
        categories=feature_order,
        ordered=True,
    )

    sort_columns = ["feature_set"]
    if "pair_label" in subset_df.columns:
        sort_columns.append("pair_label")
    subset_df = subset_df.sort_values(sort_columns)

    for metric_name, row_i, col_i in metric_specs:
        stability_fig.add_trace(
            go.Box(
                x=subset_df["feature_set"],
                y=subset_df[metric_name],
                name=comparison_label,
                legendgroup=comparison_label,
                marker_color=color_map[comparison_label],
                boxmean=True,
                showlegend=(row_i == 1 and col_i == 1),
                boxpoints="all",
                pointpos=0,
                jitter=0.10,
                marker=dict(size=7, opacity=0.75),
                line=dict(width=2),
                hovertemplate=(
                    "Feature set: %{x}<br>"
                    + comparison_label
                    + "<br>"
                    + metric_name
                    + ": %{y:.3f}<extra></extra>"
                ),
            ),
            row=row_i,
            col=col_i,
        )

stability_fig.update_layout(
    template="plotly_white",
    paper_bgcolor="white",
    plot_bgcolor=BRAND_COLORS["ice"],
    font=dict(color=BRAND_COLORS["dark_teal"]),
    width=950,
    height=640,
    title=dict(
        text="Neighboring DBSCAN Setting Stability Within Each Modality",
        font=dict(color=BRAND_COLORS["dark_teal"]),
    ),
    boxmode="group",
    legend=dict(
        title="Neighbor shift",
        orientation="h",
        y=-0.12,
        x=0.5,
        xanchor="center",
        yanchor="top",
        font=dict(color=BRAND_COLORS["dark_teal"]),
    ),
    margin=dict(t=80, b=110, l=70, r=30),
)

stability_fig.update_xaxes(
    title_text="",
    categoryorder="array",
    categoryarray=feature_order,
    matches="x",
    gridcolor="rgba(11, 78, 74, 0.14)",
    zerolinecolor="rgba(11, 78, 74, 0.20)",
    tickfont=dict(color=BRAND_COLORS["dark_teal"]),
    title_font=dict(color=BRAND_COLORS["dark_teal"]),
)
stability_fig.update_xaxes(showticklabels=False, row=1, col=1)
stability_fig.update_xaxes(showticklabels=False, row=1, col=2)
stability_fig.update_yaxes(
    title_text="",
    gridcolor="rgba(11, 78, 74, 0.14)",
    zerolinecolor="rgba(11, 78, 74, 0.20)",
    tickfont=dict(color=BRAND_COLORS["dark_teal"]),
    title_font=dict(color=BRAND_COLORS["dark_teal"]),
)

stability_fig.add_annotation(
    text="Feature set",
    x=0.5,
    y=-0.10,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(size=12, color=BRAND_COLORS["dark_teal"]),
)

stability_fig.add_annotation(
    text="Metric value",
    x=-0.06,
    y=0.5,
    xref="paper",
    yref="paper",
    showarrow=False,
    textangle=-90,
    font=dict(size=12, color=BRAND_COLORS["dark_teal"]),
)

stability_fig.show()

Findings\. The compact summary table and the visual tell the same story in two complementary ways\. The table is the one to read for the exact averages, and the most important column there is mean\_jaccard: across every modality, min\_samples steps are much more stable than eps shifts\. Bus, FHVHV, Multimodal, Subway, and Taxi all stay around 0\.8 to 0\.9 for min\_samples moves, while the modalities that still compare balanced versus tight drop to roughly 0\.4 to 0\.5 on those eps shifts\. The visual makes that contrast easier to absorb: the red eps comparisons sit lower on Jaccard overlap and higher on noise\-share change, while the blue min\_samples comparisons stay closer together and behave more like local tuning moves\. Cluster\-count delta is somewhat mixed, but even there the min\_samples shifts usually look more controlled than the eps shifts\. So the practical takeaway is that eps is the bigger stability lever, while min\_samples is the finer\-tuning lever inside each surviving candidate region\.

### Summarize stability at the modality level

Now that we’ve seen how neighboring settings behave pair by pair, we can turn that local evidence into a candidate\-centered view\. The goal here is to score each retained setting by how stable its immediate neighborhood looks within the modality, so that the final selection step can choose a setting that sits in a comparatively calm part of the search space rather than on a brittle edge\.

This first block does exactly that\. For each retained candidate, it summarizes the average overlap and average behavioral shifts across all of its neighboring comparisons, then adds a simple stability\-oriented ranking within each modality\.

In [39]:
# ---------------------------------------------------------------------
# Summarize candidate-centered stability within each modality
# ---------------------------------------------------------------------

DBSCAN_CANDIDATE_STABILITY_SUMMARY_PATH = (
    INTERMEDIATE_332_DIR / "dbscan_candidate_stability_summary.parquet"
)

candidate_left_df = (
    dbscan_stability_pairwise_summary_df[
        [
            "feature_set",
            "representation_name",
            "left_candidate_id",
            "left_min_samples",
            "left_eps_band",
            "left_eps_value",
            "flag_jaccard_similarity",
            "noise_share_delta",
            "cluster_count_delta",
            "largest_cluster_share_delta",
        ]
    ]
    .rename(
        columns={
            "left_candidate_id": "candidate_id",
            "left_min_samples": "min_samples",
            "left_eps_band": "eps_band",
            "left_eps_value": "eps_value",
        }
    )
    .copy()
)

candidate_right_df = (
    dbscan_stability_pairwise_summary_df[
        [
            "feature_set",
            "representation_name",
            "right_candidate_id",
            "right_min_samples",
            "right_eps_band",
            "right_eps_value",
            "flag_jaccard_similarity",
            "noise_share_delta",
            "cluster_count_delta",
            "largest_cluster_share_delta",
        ]
    ]
    .rename(
        columns={
            "right_candidate_id": "candidate_id",
            "right_min_samples": "min_samples",
            "right_eps_band": "eps_band",
            "right_eps_value": "eps_value",
        }
    )
    .copy()
)

candidate_neighbor_stability_df = pd.concat(
    [candidate_left_df, candidate_right_df],
    ignore_index=True,
)

dbscan_candidate_stability_summary_df = (
    candidate_neighbor_stability_df.groupby(
        [
            "feature_set",
            "representation_name",
            "candidate_id",
            "min_samples",
            "eps_band",
            "eps_value",
        ],
        dropna=False,
    )
    .agg(
        neighbor_comparisons=("candidate_id", "size"),
        mean_neighbor_jaccard=("flag_jaccard_similarity", "mean"),
        min_neighbor_jaccard=("flag_jaccard_similarity", "min"),
        max_neighbor_jaccard=("flag_jaccard_similarity", "max"),
        mean_noise_delta=("noise_share_delta", "mean"),
        mean_cluster_delta=("cluster_count_delta", "mean"),
        mean_largest_cluster_share_delta=("largest_cluster_share_delta", "mean"),
    )
    .reset_index()
)

# Bring back the original calibration behavior so the stability summary
# stays anchored to the underlying DBSCAN behavior, not just overlap.
candidate_behavior_lookup_df = (
    dbscan_calibration_behavior_df[
        [
            "feature_set",
            "representation_name",
            "min_samples",
            "eps_band",
            "eps_value",
            "weighted_noise_share",
            "weighted_cluster_count",
            "weighted_largest_cluster_share",
        ]
    ]
    .copy()
)
candidate_behavior_lookup_df["eps_band"] = candidate_behavior_lookup_df["eps_band"].astype(str)

dbscan_candidate_stability_summary_df["eps_band"] = (
    dbscan_candidate_stability_summary_df["eps_band"].astype(str)
)

dbscan_candidate_stability_summary_df = (
    dbscan_candidate_stability_summary_df.merge(
        candidate_behavior_lookup_df,
        on=[
            "feature_set",
            "representation_name",
            "min_samples",
            "eps_band",
            "eps_value",
        ],
        how="left",
    )
    .sort_values(["feature_set", "min_samples", "eps_band"])
    .reset_index(drop=True)
)

# Stability-oriented within-modality ranks.
dbscan_candidate_stability_summary_df["jaccard_rank_within_modality"] = (
    dbscan_candidate_stability_summary_df.groupby("feature_set")["mean_neighbor_jaccard"]
    .rank(method="dense", ascending=False)
)

dbscan_candidate_stability_summary_df["noise_delta_rank_within_modality"] = (
    dbscan_candidate_stability_summary_df.groupby("feature_set")["mean_noise_delta"]
    .rank(method="dense", ascending=True)
)

dbscan_candidate_stability_summary_df["cluster_delta_rank_within_modality"] = (
    dbscan_candidate_stability_summary_df.groupby("feature_set")["mean_cluster_delta"]
    .rank(method="dense", ascending=True)
)

dbscan_candidate_stability_summary_df["largest_cluster_share_delta_rank_within_modality"] = (
    dbscan_candidate_stability_summary_df.groupby("feature_set")["mean_largest_cluster_share_delta"]
    .rank(method="dense", ascending=True)
)

if should_rebuild(REBUILD_DBSCAN_STABILITY_DIAGNOSTICS, DBSCAN_CANDIDATE_STABILITY_SUMMARY_PATH):
    dbscan_candidate_stability_summary_df.to_parquet(
        DBSCAN_CANDIDATE_STABILITY_SUMMARY_PATH,
        index=False,
    )

display(
    dbscan_candidate_stability_summary_df[
        [
            "feature_set",
            "min_samples",
            "eps_band",
            "mean_neighbor_jaccard",
            "mean_noise_delta",
            "mean_cluster_delta",
            "weighted_noise_share",
            "weighted_cluster_count",
        ]
    ].round(3)
)

,feature_set,min_samples,eps_band,mean_neighbor_jaccard,mean_noise_delta,mean_cluster_delta,weighted_noise_share,weighted_cluster_count
0,bus,10,balanced,0.670,0.010,0.762,0.019,2.012
1,bus,10,tight,0.675,0.010,1.108,0.037,3.310
2,bus,15,balanced,0.707,0.007,0.351,0.018,1.787
3,bus,15,tight,0.736,0.007,0.642,0.036,2.392
4,bus,20,balanced,0.638,0.010,0.325,0.015,1.563
5,bus,20,tight,0.677,0.010,0.414,0.033,1.989
6,fhvhv,10,balanced,0.639,0.014,0.187,0.023,2.174
7,fhvhv,10,tight,0.661,0.013,0.308,0.048,2.303
8,fhvhv,15,balanced,0.693,0.009,0.300,0.021,1.929
9,fhvhv,15,tight,0.735,0.010,0.281,0.047,1.816


Findings\. The easiest way to read this table is to start with mean\_neighbor\_jaccard, because it tells us which retained settings sit in the most stable local neighborhood\. Higher is better there\. Then use mean\_noise\_delta and mean\_cluster\_delta as tie\-breakers: smaller values mean the setting changes anomaly volume and cluster structure less when we nudge the parameters nearby\. On that reading, a common pattern emerges across Bus, FHVHV, Multimodal, and Subway: the min\_samples = 15 settings are usually the strongest middle ground, and the main tradeoff is between balanced and tight\. Tight often gives slightly better overlap, but it also carries noticeably higher weighted\_noise\_share and often higher weighted\_cluster\_count, meaning a more aggressive anomaly surface\. Balanced usually gives up a little overlap in exchange for a calmer overall behavior\. Taxi is the clearest case: since only tight survived, the question reduces to which min\_samples level is most stable, and min\_samples = 20 looks strongest on stability metrics while also producing the lowest cluster count and slightly lower anomaly volume than the smaller values\. So the practical use of this table is to identify the stable center of each modality’s retained region before we make the final setting choice\.

Heatmap block 1: Bus / Subway / Taxi / FHVHV\. This first visual keeps the four single\-system modalities together so we can compare their candidate stability patterns side by side\. The numbers inside the tiles are the raw metrics, while the colors are relative within each modality: greener tiles are better local stability candidates for that specific system\.

In [40]:
# ---------------------------------------------------------------------
# Visualize candidate-centered stability: Bus, Subway, Taxi, and FHVHV
# ---------------------------------------------------------------------

single_system_stability_plot_df = dbscan_candidate_stability_summary_df.copy()

single_system_stability_plot_df["setting_label"] = (
    "ms="
    + single_system_stability_plot_df["min_samples"].astype(int).astype(str)
    + "<br>"
    + single_system_stability_plot_df["eps_band"].astype(str)
)

single_system_feature_order = ["bus", "subway", "taxi", "fhvhv"]
single_system_setting_order = [
    "ms=10<br>tight",
    "ms=10<br>balanced",
    "ms=15<br>tight",
    "ms=15<br>balanced",
    "ms=20<br>tight",
    "ms=20<br>balanced",
]

single_system_stability_plot_df = single_system_stability_plot_df.loc[
    single_system_stability_plot_df["feature_set"].isin(single_system_feature_order)
].copy()

single_system_stability_plot_df["feature_set"] = pd.Categorical(
    single_system_stability_plot_df["feature_set"],
    categories=single_system_feature_order,
    ordered=True,
)
single_system_stability_plot_df["setting_label"] = pd.Categorical(
    single_system_stability_plot_df["setting_label"],
    categories=single_system_setting_order,
    ordered=True,
)

# Relative scores are computed within each modality:
# - higher mean_neighbor_jaccard is better
# - lower mean_noise_delta is better
# - lower mean_cluster_delta is better
single_system_stability_plot_df["jaccard_score"] = (
    single_system_stability_plot_df.groupby("feature_set", observed=False)["mean_neighbor_jaccard"]
    .transform(lambda s: (s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else 1.0)
)

single_system_stability_plot_df["noise_stability_score"] = (
    single_system_stability_plot_df.groupby("feature_set", observed=False)["mean_noise_delta"]
    .transform(lambda s: 1 - ((s - s.min()) / (s.max() - s.min())) if s.max() > s.min() else 1.0)
)

single_system_stability_plot_df["cluster_stability_score"] = (
    single_system_stability_plot_df.groupby("feature_set", observed=False)["mean_cluster_delta"]
    .transform(lambda s: 1 - ((s - s.min()) / (s.max() - s.min())) if s.max() > s.min() else 1.0)
)

single_system_fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=["Bus", "Subway", "Taxi", "FHVHV"],
    horizontal_spacing=0.08,
    vertical_spacing=0.08,
    shared_xaxes=True,
    shared_yaxes=True,
)

single_system_subplot_positions = {
    "bus": (1, 1),
    "subway": (1, 2),
    "taxi": (2, 1),
    "fhvhv": (2, 2),
}

brand_stability_scale = [
    [0.00, BRAND_COLORS["terracotta"]],
    [0.50, BRAND_COLORS["pale_peach"]],
    [0.75, BRAND_COLORS["seafoam"]],
    [1.00, BRAND_COLORS["dark_teal"]],
]

for feature_set_name in single_system_feature_order:
    row_i, col_i = single_system_subplot_positions[feature_set_name]
    feature_df = single_system_stability_plot_df.loc[
        single_system_stability_plot_df["feature_set"].eq(feature_set_name)
    ].copy()

    z_matrix = []
    text_matrix = []
    x_labels = ["Overlap", "Volume", "Structure"]

    for setting_label in single_system_setting_order:
        setting_df = feature_df.loc[feature_df["setting_label"].eq(setting_label)]
        if setting_df.empty:
            z_matrix.append([np.nan, np.nan, np.nan])
            text_matrix.append(["", "", ""])
            continue

        row = setting_df.iloc[0]
        z_matrix.append(
            [
                row["jaccard_score"],
                row["noise_stability_score"],
                row["cluster_stability_score"],
            ]
        )
        text_matrix.append(
            [
                f"{row['mean_neighbor_jaccard']:.3f}",
                f"{row['mean_noise_delta']:.3f}",
                f"{row['mean_cluster_delta']:.3f}",
            ]
        )

    single_system_fig.add_trace(
        go.Heatmap(
            z=z_matrix,
            x=x_labels,
            y=[str(label) for label in single_system_setting_order],
            colorscale=brand_stability_scale,
            zmin=0,
            zmax=1,
            showscale=(feature_set_name == "fhvhv"),
            colorbar=dict(
                title=dict(
                    text="Relative score",
                    font=dict(color=BRAND_COLORS["dark_teal"]),
                ),
                tickfont=dict(color=BRAND_COLORS["dark_teal"]),
                len=0.78,
            ) if feature_set_name == "fhvhv" else None,
            hovertemplate=(
                "Feature set: "
                + feature_set_name
                + "<br>Setting: %{y}"
                + "<br>Dimension: %{x}"
                + "<br>Relative score: %{z:.3f}<extra></extra>"
            ),
        ),
        row=row_i,
        col=col_i,
    )

    for y_index, setting_label in enumerate(single_system_setting_order):
        for x_index, x_label in enumerate(x_labels):
            cell_text = text_matrix[y_index][x_index]
            cell_score = z_matrix[y_index][x_index]

            if cell_text == "" or pd.isna(cell_score):
                continue

            single_system_fig.add_annotation(
                x=x_label,
                y=str(setting_label),
                text=cell_text,
                showarrow=False,
                font=dict(
                    size=10,
                    color=(
                        "white"
                        if cell_score >= 0.78
                        else BRAND_COLORS["dark_teal"]
                    ),
                ),
                row=row_i,
                col=col_i,
            )

single_system_fig.update_layout(
    template="plotly_white",
    paper_bgcolor="white",
    plot_bgcolor=BRAND_COLORS["ice"],
    font=dict(color=BRAND_COLORS["dark_teal"]),
    width=960,
    height=680,
    title=dict(
        text="Candidate-Centered DBSCAN Stability: Bus, Subway, Taxi, and FHVHV",
        font=dict(color=BRAND_COLORS["dark_teal"]),
    ),
    margin=dict(t=90, b=80, l=155, r=40),
)

# Share axes visually by letting only the outer axes carry tick labels.
single_system_fig.update_xaxes(
    title_text="",
    matches="x",
    gridcolor="rgba(11, 78, 74, 0.14)",
    tickfont=dict(color=BRAND_COLORS["dark_teal"]),
    title_font=dict(color=BRAND_COLORS["dark_teal"]),
)
single_system_fig.update_yaxes(
    title_text="",
    matches="y",
    gridcolor="rgba(11, 78, 74, 0.14)",
    tickfont=dict(color=BRAND_COLORS["dark_teal"]),
    title_font=dict(color=BRAND_COLORS["dark_teal"]),
)

single_system_fig.update_xaxes(showticklabels=False, row=1, col=1)
single_system_fig.update_xaxes(showticklabels=False, row=1, col=2)

single_system_fig.update_yaxes(showticklabels=False, row=1, col=2)
single_system_fig.update_yaxes(showticklabels=False, row=2, col=2)

single_system_fig.update_yaxes(ticklabelstandoff=10, row=1, col=1)
single_system_fig.update_yaxes(ticklabelstandoff=10, row=2, col=1)

single_system_fig.add_annotation(
    text="Stability dimension",
    x=0.5,
    y=-0.08,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(size=12, color=BRAND_COLORS["dark_teal"]),
)

single_system_fig.add_annotation(
    text="Retained candidate setting",
    x=-0.19,
    y=0.5,
    xref="paper",
    yref="paper",
    showarrow=False,
    textangle=-90,
    font=dict(size=12, color=BRAND_COLORS["dark_teal"]),
)

single_system_fig.show()

Findings\. Across the four single\-system modalities, the shared pattern is that the best DBSCAN setting usually sits in the middle of the retained search space rather than at its edges\. min\_samples = 15 is the strongest general anchor for Bus, Subway, and FHVHV, while Taxi remains the exception and stabilizes best at the tighter high\-min\_samples end\.

- Bus: min\_samples = 15 is the clearest center of stability\. 15 tight has the strongest overlap, but 15 balanced keeps structure drift lower, so the practical choice is between slightly higher continuity and slightly calmer behavior\.

- Subway: the same middle\-zone pattern holds\. 15 tight has the best overlap, but 15 balanced is calmer on structure and still keeps very strong neighborhood continuity, which makes it the more conservative stable choice\.

- Taxi: Taxi is the outlier\. Because only tight settings survived, the choice reduces to min\_samples, and 20 tight looks best by a comfortable margin, with the strongest overlap and the smallest structure shift\.

- FHVHV: FHVHV again favors the middle\. 15 tight leads on overlap, but 15 balanced remains very competitive and behaves a bit more gently overall, which makes it the more stable center\-of\-region candidate\.

Multimodal standalone\. This second visual gives Multimodal more room, which helps because it is the richest branch and often the hardest one to read when it gets squeezed into a shared panel\.

In [41]:
# ---------------------------------------------------------------------
# Visualize candidate-centered stability: Multimodal
# ---------------------------------------------------------------------

multimodal_stability_plot_df = dbscan_candidate_stability_summary_df.loc[
    dbscan_candidate_stability_summary_df["feature_set"].eq("multimodal")
].copy()

multimodal_stability_plot_df["setting_label"] = (
    "ms="
    + multimodal_stability_plot_df["min_samples"].astype(int).astype(str)
    + " | "
    + multimodal_stability_plot_df["eps_band"].astype(str)
)

multimodal_setting_order = [
    "ms=10 | tight",
    "ms=10 | balanced",
    "ms=15 | tight",
    "ms=15 | balanced",
    "ms=20 | tight",
    "ms=20 | balanced",
]

multimodal_stability_plot_df["setting_label"] = pd.Categorical(
    multimodal_stability_plot_df["setting_label"],
    categories=multimodal_setting_order,
    ordered=True,
)

multimodal_stability_plot_df["jaccard_score"] = (
    multimodal_stability_plot_df["mean_neighbor_jaccard"]
    .transform(
        lambda s: (s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else 1.0
    )
)

multimodal_stability_plot_df["noise_stability_score"] = (
    multimodal_stability_plot_df["mean_noise_delta"]
    .transform(
        lambda s: 1 - ((s - s.min()) / (s.max() - s.min())) if s.max() > s.min() else 1.0
    )
)

multimodal_stability_plot_df["cluster_stability_score"] = (
    multimodal_stability_plot_df["mean_cluster_delta"]
    .transform(
        lambda s: 1 - ((s - s.min()) / (s.max() - s.min())) if s.max() > s.min() else 1.0
    )
)

multimodal_z_matrix = []
multimodal_text_matrix = []
multimodal_x_labels = ["Overlap", "Volume", "Structure"]

for setting_label in multimodal_setting_order:
    setting_df = multimodal_stability_plot_df.loc[
        multimodal_stability_plot_df["setting_label"].eq(setting_label)
    ]
    if setting_df.empty:
        multimodal_z_matrix.append([np.nan, np.nan, np.nan])
        multimodal_text_matrix.append(["", "", ""])
        continue

    row = setting_df.iloc[0]
    multimodal_z_matrix.append(
        [
            row["jaccard_score"],
            row["noise_stability_score"],
            row["cluster_stability_score"],
        ]
    )
    multimodal_text_matrix.append(
        [
            f"{row['mean_neighbor_jaccard']:.3f}",
            f"{row['mean_noise_delta']:.3f}",
            f"{row['mean_cluster_delta']:.3f}",
        ]
    )

brand_stability_scale = [
    [0.00, BRAND_COLORS["terracotta"]],
    [0.50, BRAND_COLORS["pale_peach"]],
    [0.75, BRAND_COLORS["seafoam"]],
    [1.00, BRAND_COLORS["dark_teal"]],
]

multimodal_fig = go.Figure(
    data=[
        go.Heatmap(
            z=multimodal_z_matrix,
            x=multimodal_x_labels,
            y=[str(label) for label in multimodal_setting_order],
            colorscale=brand_stability_scale,
            zmin=0,
            zmax=1,
            colorbar=dict(
                title=dict(
                    text="Relative score",
                    font=dict(color=BRAND_COLORS["dark_teal"]),
                ),
                tickfont=dict(color=BRAND_COLORS["dark_teal"]),
                len=0.85,
            ),
            hovertemplate=(
                "Feature set: multimodal"
                + "<br>Setting: %{y}"
                + "<br>Dimension: %{x}"
                + "<br>Relative score: %{z:.3f}<extra></extra>"
            ),
        )
    ]
)

for y_index, setting_label in enumerate(multimodal_setting_order):
    for x_index, x_label in enumerate(multimodal_x_labels):
        cell_text = multimodal_text_matrix[y_index][x_index]
        cell_score = multimodal_z_matrix[y_index][x_index]

        if cell_text == "" or pd.isna(cell_score):
            continue

        multimodal_fig.add_annotation(
            x=x_label,
            y=str(setting_label),
            text=cell_text,
            showarrow=False,
            font=dict(
                size=11,
                color=(
                    "white"
                    if cell_score >= 0.78
                    else BRAND_COLORS["dark_teal"]
                ),
            ),
        )

multimodal_fig.update_layout(
    template="plotly_white",
    paper_bgcolor="white",
    plot_bgcolor=BRAND_COLORS["ice"],
    font=dict(color=BRAND_COLORS["dark_teal"]),
    width=720,
    height=520,
    title=dict(
        text="Candidate-Centered DBSCAN Stability: Multimodal",
        font=dict(color=BRAND_COLORS["dark_teal"]),
    ),
    margin=dict(t=80, b=70, l=150, r=60),
    xaxis_title="Stability dimension",
    yaxis_title="Retained candidate setting",
)

multimodal_fig.update_xaxes(
    gridcolor="rgba(11, 78, 74, 0.14)",
    tickfont=dict(color=BRAND_COLORS["dark_teal"]),
    title_font=dict(color=BRAND_COLORS["dark_teal"]),
)
multimodal_fig.update_yaxes(
    ticklabelstandoff=6,
    gridcolor="rgba(11, 78, 74, 0.14)",
    tickfont=dict(color=BRAND_COLORS["dark_teal"]),
    title_font=dict(color=BRAND_COLORS["dark_teal"]),
)

multimodal_fig.show()

Findings\. Multimodal has a cleaner winner than most of the single\-system branches\. min\_samples = 15, balanced is the strongest overall candidate because it combines the best overlap with the lowest volume drift among the top\-overlap settings, while keeping structure change in a middle, still\-manageable range\. min\_samples = 15, tight is close on overlap, but it is clearly more structurally volatile\. The min\_samples = 20 settings are calmer on structure, especially 20 balanced, but they give up too much overlap to look like the best retained center of the search space\. So Multimodal looks like a case where the stable choice is not the calmest possible row on every single metric, but the one that best balances all three dimensions at once\.

> 🎯 Decision\. The stability readout is strong enough to move from candidate regions to provisional final settings\. The leading choices are: Bus = min\_samples 15, balanced, Subway = min\_samples 15, balanced, Taxi = min\_samples 20, tight, FHVHV = min\_samples 15, balanced, and Multimodal = min\_samples 15, balanced\. The common pattern is that min\_samples = 15 gives the most reliable center for most modalities, while Taxi remains the exception and needs the tighter, higher\-min\_samples setting to stay stable\.

Record the final modality\-specific DBSCAN settings\. With the stability evidence in place, we can now turn the notebook’s calibration work into a clean handoff\. This block records one final PCA\-based DBSCAN setting per modality so the downstream anomaly\-generation steps inherit a single coherent configuration for each mobility system rather than a still\-open candidate region\.

In [42]:
# ---------------------------------------------------------------------
# Record one final DBSCAN setting per modality
# ---------------------------------------------------------------------

final_dbscan_setting_records = [
    {
        "feature_set": "bus",
        "representation_name": "pca_80pct",
        "min_samples": 15,
        "eps_band": "balanced",
        "selection_rationale": "Middle stable region with calmer structure than tight while preserving strong overlap.",
    },
    {
        "feature_set": "subway",
        "representation_name": "pca_80pct",
        "min_samples": 15,
        "eps_band": "balanced",
        "selection_rationale": "Middle stable region with strong overlap and less aggressive structure drift than tight.",
    },
    {
        "feature_set": "taxi",
        "representation_name": "pca_80pct",
        "min_samples": 20,
        "eps_band": "tight",
        "selection_rationale": "Best surviving tight candidate on overlap and structure stability.",
    },
    {
        "feature_set": "fhvhv",
        "representation_name": "pca_80pct",
        "min_samples": 15,
        "eps_band": "balanced",
        "selection_rationale": "Middle stable region with strong overlap and gentler behavior than tight.",
    },
    {
        "feature_set": "multimodal",
        "representation_name": "pca_80pct",
        "min_samples": 15,
        "eps_band": "balanced",
        "selection_rationale": "Best overall balance of overlap, volume stability, and structure stability.",
    },
]

dbscan_final_pca_settings_built_df = (
    pd.DataFrame(final_dbscan_setting_records)
    .merge(
        dbscan_candidate_stability_summary_df[
            [
                "feature_set",
                "representation_name",
                "min_samples",
                "eps_band",
                "eps_value",
                "mean_neighbor_jaccard",
                "mean_noise_delta",
                "mean_cluster_delta",
                "mean_largest_cluster_share_delta",
                "weighted_noise_share",
                "weighted_cluster_count",
                "weighted_largest_cluster_share",
            ]
        ],
        on=["feature_set", "representation_name", "min_samples", "eps_band"],
        how="left",
    )
    .sort_values(["feature_set"])
    .reset_index(drop=True)
)

dbscan_final_pca_settings_df = dbscan_final_pca_settings_built_df.copy()

if should_rebuild(REBUILD_DBSCAN_STABILITY_DIAGNOSTICS, DBSCAN_SELECTED_CONFIGURATION_PATH):
    dbscan_final_pca_settings_df.to_csv(
        DBSCAN_SELECTED_CONFIGURATION_PATH,
        index=False,
    )

dbscan_final_pca_settings_validation_df = pd.DataFrame(
    [
        {
            "selected_feature_sets": dbscan_final_pca_settings_df["feature_set"].nunique(),
            "expected_feature_sets": len(MODEL_FEATURE_SET_NAMES),
            "selected_rows": len(dbscan_final_pca_settings_df),
            "status": (
                "pass"
                if dbscan_final_pca_settings_df["feature_set"].nunique() == len(MODEL_FEATURE_SET_NAMES)
                and len(dbscan_final_pca_settings_df) == len(MODEL_FEATURE_SET_NAMES)
                else "review"
            ),
        }
    ]
)

assert dbscan_final_pca_settings_validation_df["status"].eq("pass").all(), (
    "The final DBSCAN setting table does not contain exactly one setting per modality."
)

display(
    dbscan_final_pca_settings_df[
        [
            "feature_set",
            "representation_name",
            "min_samples",
            "eps_band",
            "eps_value",
            "mean_neighbor_jaccard",
            "mean_noise_delta",
            "mean_cluster_delta",
            "weighted_noise_share",
            "weighted_cluster_count",
            "selection_rationale",
        ]
    ].round(
        {
            "eps_value": 3,
            "mean_neighbor_jaccard": 3,
            "mean_noise_delta": 3,
            "mean_cluster_delta": 3,
            "weighted_noise_share": 3,
            "weighted_cluster_count": 3,
        }
    )
)

,feature_set,representation_name,min_samples,eps_band,eps_value,mean_neighbor_jaccard,mean_noise_delta,mean_cluster_delta,weighted_noise_share,weighted_cluster_count,selection_rationale
0,bus,pca_80pct,15,balanced,5.281,0.707,0.007,0.351,0.018,1.787,Middle stable region with calmer structure tha...
1,fhvhv,pca_80pct,15,balanced,5.116,0.693,0.009,0.300,0.021,1.929,Middle stable region with strong overlap and g...
2,multimodal,pca_80pct,15,balanced,13.558,0.710,0.008,0.457,0.017,1.871,"Best overall balance of overlap, volume stabil..."
3,subway,pca_80pct,15,balanced,2.745,0.755,0.009,0.333,0.032,1.621,Middle stable region with strong overlap and l...
4,taxi,pca_80pct,20,tight,4.812,0.835,0.004,0.115,0.033,1.113,Best surviving tight candidate on overlap and ...


Findings\. The final DBSCAN handoff is now explicit and modality\-specific\. Bus, Subway, FHVHV, and Multimodal all settle on min\_samples = 15, balanced, which reinforces the broader pattern that the middle of the retained PCA search space is the most stable landing zone for most systems\. Taxi remains the exception and advances with min\_samples = 20, tight, reflecting its narrower and more fragile viable region\. Taken together, these five settings give the downstream anomaly\-generation step one coherent PCA\-based DBSCAN configuration per modality, with enough consistency to keep the workflow simple while still respecting the important structural differences across systems\.

### Section Summary

This section turned the narrowed DBSCAN candidate regions into final modality\-specific PCA settings by testing how stable each retained region remained under small local parameter changes\. The main result was consistent across most systems: changing eps tends to alter the anomaly surface more materially than changing min\_samples, which made eps the bigger stability lever and min\_samples the finer\-tuning knob\. That made the middle of the retained search space the safest landing zone for most modalities\. Bus, Subway, FHVHV, and Multimodal all settled on min\_samples = 15, balanced, while Taxi remained the exception and stabilized best at min\_samples = 20, tight\. The notebook now has one explicit DBSCAN setting per modality, which gives the downstream anomaly\-generation steps a coherent PCA\-based configuration set to carry forward\.

## 3\.3\.2\.5 Inspect Representative Candidate DBSCAN Anomalies

This section assembles a three\-row inspection panel for each modality so we can judge whether the calibrated DBSCAN outputs look interpretable across different temporal buckets and policy\-geography regimes, without turning the review into a wall of examples\.

The goal here is human sense\-making\. We want to see what kinds of mobility states DBSCAN is actually flagging, whether those flags appear varied enough to be plausible, and whether the resulting anomaly surfaces are interpretable enough to carry forward into downstream comparison and interpretation\.

### Define the representative inspection sample

Let’s first make the inspection design explicit\. Every modality will contribute three candidate anomalies, giving us a 15\-row panel that is still broad enough to show different temporal buckets and policy\-geography settings while staying readable enough for careful human review\.

In [43]:
# ---------------------------------------------------------------------
# Define the representative DBSCAN anomaly inspection panel
# ---------------------------------------------------------------------

inspection_rows_per_modality = 3
total_inspection_target_rows = inspection_rows_per_modality * len(MODEL_FEATURE_SET_NAMES)

assert "dbscan_final_pca_settings_df" in globals(), (
    "dbscan_final_pca_settings_df is not in memory. "
    "Please run the final DBSCAN setting-selection block first."
)

inspection_manifest_df = (
    dbscan_final_pca_settings_df[
        [
            "feature_set",
            "representation_name",
            "min_samples",
            "eps_band",
            "eps_value",
        ]
    ]
    .copy()
    .sort_values(["feature_set"])
    .reset_index(drop=True)
)

# If the final export manifest exists, enrich the inspection plan with the
# realized anomaly counts. If not, leave them blank; the inspection plan
# itself does not require the final export to exist.
if "dbscan_final_row_level_manifest_df" in globals():
    anomaly_count_df = dbscan_final_row_level_manifest_df[
        ["feature_set", "anomaly_like_rows"]
    ].copy()
elif DBSCAN_ANOMALY_EXPORT_MANIFEST_PATH.exists():
    anomaly_count_df = pd.read_csv(DBSCAN_ANOMALY_EXPORT_MANIFEST_PATH)[
        ["feature_set", "anomaly_like_rows"]
    ].copy()
else:
    anomaly_count_df = pd.DataFrame(
        {
            "feature_set": MODEL_FEATURE_SET_NAMES,
            "anomaly_like_rows": np.nan,
        }
    )

dbscan_inspection_plan_df = (
    inspection_manifest_df.merge(
        anomaly_count_df,
        on="feature_set",
        how="left",
        validate="one_to_one",
    )
    .sort_values(["feature_set"])
    .reset_index(drop=True)
)

dbscan_inspection_plan_df["inspection_rows_requested"] = inspection_rows_per_modality

dbscan_inspection_plan_validation_df = pd.DataFrame(
    [
        {
            "rows_selected": int(dbscan_inspection_plan_df["inspection_rows_requested"].sum()),
            "target_rows": total_inspection_target_rows,
            "feature_sets_covered": dbscan_inspection_plan_df["feature_set"].nunique(),
            "expected_feature_sets": len(MODEL_FEATURE_SET_NAMES),
            "status": (
                "pass"
                if dbscan_inspection_plan_df["feature_set"].nunique() == len(MODEL_FEATURE_SET_NAMES)
                and int(dbscan_inspection_plan_df["inspection_rows_requested"].sum()) == total_inspection_target_rows
                else "review"
            ),
        }
    ]
)

display(
    dbscan_inspection_plan_df[
        [
            "feature_set",
            "representation_name",
            "min_samples",
            "eps_band",
            "eps_value",
            "anomaly_like_rows",
            "inspection_rows_requested",
        ]
    ].round(3)
)

display(dbscan_inspection_plan_validation_df)

assert dbscan_inspection_plan_validation_df["status"].eq("pass").all(), (
    "The representative DBSCAN inspection plan is incomplete."
)

,feature_set,representation_name,min_samples,eps_band,eps_value,anomaly_like_rows,inspection_rows_requested
0,bus,pca_80pct,15,balanced,5.281,3074,3
1,fhvhv,pca_80pct,15,balanced,5.116,6086,3
2,multimodal,pca_80pct,15,balanced,13.558,4065,3
3,subway,pca_80pct,15,balanced,2.745,5853,3
4,taxi,pca_80pct,20,tight,4.812,7027,3


,rows_selected,target_rows,feature_sets_covered,expected_feature_sets,status
0,15,15,5,5,pass


Findings\. The inspection panel is set up cleanly: all five modalities are represented, each contributes 3 reviewed anomalies, and the resulting 15\-row panel is drawn from the final modality\-specific PCA DBSCAN settings selected earlier in the notebook\. The anomaly\-like row counts remind us that the underlying candidate surfaces are not equally large, with Multimodal and Bus contributing the smallest anomaly pools and Taxi and Subway the largest, but the inspection sample itself stays intentionally balanced so the qualitative review does not get dominated by the biggest branches\.

### Materialize the representative inspection rows

Now we can pull the actual anomalies for review\. This selection is intentionally diversity\-seeking: within each modality, it tries to spread attention across different temporal buckets, policy\-geography classes, boroughs, and zones so the panel stays informative without becoming visually overwhelming\.

This helper block prepares the small review\-panel plan for DBSCAN anomaly inspection\. It reconstructs the expected number of review rows per modality and defines the row\-selection rule we will use downstream\. The selection logic is intentionally simple: it tries to preserve coverage across policy geographies, temporal buckets, and policy periods so the final review panel is not accidentally dominated by one narrow operating context\.

In [44]:
# ---------------------------------------------------------------------
# Define helper functions for representative DBSCAN inspection-row selection
# ---------------------------------------------------------------------

inspection_rows_path_local = (
    dbscan_inspection_rows_path
    if "dbscan_inspection_rows_path" in globals()
    else INTERMEDIATE_332_DIR / "dbscan_representative_inspection_rows.parquet"
)

if "dbscan_inspection_plan_df" in globals():
    inspection_plan_df = dbscan_inspection_plan_df.copy()
elif DBSCAN_ANOMALY_EXPORT_MANIFEST_PATH.exists():
    inspection_manifest_df = pd.read_csv(DBSCAN_ANOMALY_EXPORT_MANIFEST_PATH)
    inspection_plan_df = (
        inspection_manifest_df[
            [
                "feature_set",
                "representation_name",
                "min_samples",
                "eps_band",
                "eps_value",
                "anomaly_like_rows",
            ]
        ]
        .sort_values(["feature_set"])
        .reset_index(drop=True)
    )
    inspection_plan_df["inspection_rows_requested"] = 3
else:
    raise AssertionError(
        "Could not find the DBSCAN inspection plan in memory or reconstruct it from the export manifest."
    )

expected_total_rows = int(inspection_plan_df["inspection_rows_requested"].sum())
expected_feature_sets = len(MODEL_FEATURE_SET_NAMES)

def choose_representative_rows(anomaly_df: pd.DataFrame, n_rows: int) -> pd.DataFrame:
    """
    Select a small review panel that stays intentionally diverse across
    policy geography, temporal bucket, and policy period. The main goals are:
    1. cover cbd / gateway_or_adjacent / outside when possible
    2. avoid collapsing into one temporal bucket
    3. include at least one post_cp row when the anomaly surface contains one
    """
    working_df = anomaly_df.copy()

    working_df = working_df.sort_values(
        [
            PRE_POST_CP_COLUMN,
            POLICY_GEOGRAPHY_COLUMN,
            TEMPORAL_BUCKET_COLUMN,
            DATE_COLUMN,
            BOROUGH_COLUMN,
            ZONE_COLUMN,
            MODEL_ID_COLUMN,
        ]
    ).reset_index(drop=True)

    policy_priority = ["cbd", "gateway_or_adjacent", "outside"]

    selected_parts = []
    used_model_ids = set()
    used_temporal_buckets = set()
    used_policy_geos = set()
    used_periods = set()

    def choose_best_candidate(candidates_df: pd.DataFrame) -> pd.DataFrame | None:
        if candidates_df.empty:
            return None

        best_idx = None
        best_score = None

        for candidate_idx, candidate_row in candidates_df.iterrows():
            score = (
                int(candidate_row[PRE_POST_CP_COLUMN] not in used_periods),
                int(candidate_row[POLICY_GEOGRAPHY_COLUMN] not in used_policy_geos),
                int(candidate_row[TEMPORAL_BUCKET_COLUMN] not in used_temporal_buckets),
                -candidate_idx,
            )

            if best_score is None or score > best_score:
                best_score = score
                best_idx = candidate_idx

        if best_idx is None:
            return None

        return candidates_df.loc[[best_idx]].copy()

    post_cp_candidates = working_df.loc[
        working_df[PRE_POST_CP_COLUMN].eq("post_cp")
    ].copy()

    chosen_post_cp = choose_best_candidate(post_cp_candidates)
    if chosen_post_cp is not None and len(selected_parts) < n_rows:
        selected_parts.append(chosen_post_cp)
        used_model_ids.update(chosen_post_cp[MODEL_ID_COLUMN].tolist())
        used_temporal_buckets.update(chosen_post_cp[TEMPORAL_BUCKET_COLUMN].tolist())
        used_policy_geos.update(chosen_post_cp[POLICY_GEOGRAPHY_COLUMN].tolist())
        used_periods.update(chosen_post_cp[PRE_POST_CP_COLUMN].tolist())

    for policy_geo in policy_priority:
        if len(selected_parts) >= n_rows:
            break

        policy_candidates = working_df.loc[
            working_df[POLICY_GEOGRAPHY_COLUMN].eq(policy_geo)
            & ~working_df[MODEL_ID_COLUMN].isin(used_model_ids)
        ].copy()

        chosen_row = choose_best_candidate(policy_candidates)
        if chosen_row is None:
            continue

        selected_parts.append(chosen_row)
        used_model_ids.update(chosen_row[MODEL_ID_COLUMN].tolist())
        used_temporal_buckets.update(chosen_row[TEMPORAL_BUCKET_COLUMN].tolist())
        used_policy_geos.update(chosen_row[POLICY_GEOGRAPHY_COLUMN].tolist())
        used_periods.update(chosen_row[PRE_POST_CP_COLUMN].tolist())

    while len(selected_parts) < n_rows:
        remaining_candidates = working_df.loc[
            ~working_df[MODEL_ID_COLUMN].isin(used_model_ids)
        ].copy()

        chosen_row = choose_best_candidate(remaining_candidates)
        if chosen_row is None:
            break

        selected_parts.append(chosen_row)
        used_model_ids.update(chosen_row[MODEL_ID_COLUMN].tolist())
        used_temporal_buckets.update(chosen_row[TEMPORAL_BUCKET_COLUMN].tolist())
        used_policy_geos.update(chosen_row[POLICY_GEOGRAPHY_COLUMN].tolist())
        used_periods.update(chosen_row[PRE_POST_CP_COLUMN].tolist())

    selected_df = pd.concat(selected_parts, ignore_index=True)
    return selected_df.head(n_rows).reset_index(drop=True)

Now we actually build the DBSCAN review panel\. For each modality, we read the exported anomaly surface, keep only anomaly\-like rows, and apply the shared selection rule to choose a very small but intentionally diverse set of representative cases\. The output is a compact inspection table that later blocks can reuse without rerunning the selection logic\.

In [45]:
# ---------------------------------------------------------------------
# Materialize representative DBSCAN anomaly inspection rows
# ---------------------------------------------------------------------

reuse_existing_inspection_rows = False
if inspection_rows_path_local.exists() and not REBUILD_DBSCAN_FINAL_OUTPUTS:
    existing_inspection_rows_df = pd.read_parquet(inspection_rows_path_local)
    reuse_existing_inspection_rows = (
        len(existing_inspection_rows_df) == expected_total_rows
        and existing_inspection_rows_df["feature_set"].nunique() == expected_feature_sets
    )

    if reuse_existing_inspection_rows:
        dbscan_representative_inspection_rows_df = existing_inspection_rows_df.copy()
    else:
        print(
            "Existing representative inspection-row parquet does not match the current "
            "inspection plan and will be rebuilt."
        )

if not reuse_existing_inspection_rows:
    final_row_level_dir = FINAL_332_DIR / "dbscan_candidate_anomaly_row_level_parts"
    representative_row_records = []

    for plan_row in inspection_plan_df.itertuples(index=False):
        feature_set_name = plan_row.feature_set
        row_level_path = final_row_level_dir / f"{feature_set_name}_dbscan_candidate_anomalies.parquet"

        if not row_level_path.exists():
            raise AssertionError(
                f"Candidate anomaly output is missing for {feature_set_name}: {row_level_path}"
            )

        row_level_df = pd.read_parquet(row_level_path)
        anomaly_df = (
            row_level_df.loc[row_level_df["is_anomaly_like"]]
            .copy()
            .reset_index(drop=True)
        )

        chosen_df = choose_representative_rows(
            anomaly_df=anomaly_df,
            n_rows=int(plan_row.inspection_rows_requested),
        )

        chosen_df = chosen_df.assign(
            inspection_order_within_feature_set=np.arange(1, len(chosen_df) + 1),
        )

        representative_row_records.append(
            chosen_df[
                [
                    "feature_set",
                    "inspection_order_within_feature_set",
                    "representation_name",
                    DATE_COLUMN,
                    TEMPORAL_BUCKET_COLUMN,
                    PRE_POST_CP_COLUMN,
                    POLICY_GEOGRAPHY_COLUMN,
                    BOROUGH_COLUMN,
                    ZONE_COLUMN,
                    "eps_band",
                    "eps_value",
                    "min_samples",
                    "cluster_label",
                    MODEL_ID_COLUMN,
                    "comparison_group_key",
                ]
            ]
        )

    dbscan_representative_inspection_rows_df = (
        pd.concat(representative_row_records, ignore_index=True)
        .sort_values(["feature_set", "inspection_order_within_feature_set"])
        .reset_index(drop=True)
    )

    if WRITE_332_OUTPUTS:
        dbscan_representative_inspection_rows_df.to_parquet(
            inspection_rows_path_local,
            index=False,
        )

dbscan_representative_inspection_validation_df = pd.DataFrame(
    [
        {
            "rows_selected": len(dbscan_representative_inspection_rows_df),
            "target_rows": expected_total_rows,
            "feature_sets_covered": dbscan_representative_inspection_rows_df["feature_set"].nunique(),
            "expected_feature_sets": expected_feature_sets,
            "distinct_temporal_buckets": dbscan_representative_inspection_rows_df[TEMPORAL_BUCKET_COLUMN].nunique(),
            "distinct_policy_geographies": dbscan_representative_inspection_rows_df[POLICY_GEOGRAPHY_COLUMN].nunique(),
            "status": (
                "pass"
                if len(dbscan_representative_inspection_rows_df) == expected_total_rows
                and dbscan_representative_inspection_rows_df["feature_set"].nunique() == expected_feature_sets
                else "review"
            ),
        }
    ]
)

display(
    dbscan_representative_inspection_rows_df[
        [
            "feature_set",
            "inspection_order_within_feature_set",
            DATE_COLUMN,
            TEMPORAL_BUCKET_COLUMN,
            PRE_POST_CP_COLUMN,
            POLICY_GEOGRAPHY_COLUMN,
            BOROUGH_COLUMN,
            ZONE_COLUMN,
            "eps_band",
            "eps_value",
            "min_samples",
            "cluster_label",
        ]
    ].round(3)
)

display(dbscan_representative_inspection_validation_df)

assert dbscan_representative_inspection_validation_df["status"].eq("pass").all(), (
    "The representative DBSCAN inspection-row panel is incomplete."
)

,feature_set,inspection_order_within_feature_set,date,temporal_bucket,pre_post_cp,policy_geography_class,borough,zone,eps_band,eps_value,min_samples,cluster_label
0,bus,1,2025-05-19,weekday_am_peak,post_cp,cbd,Manhattan,Greenwich Village South,balanced,5.281,15,-1
1,bus,2,2024-03-12,weekday_evening,pre_cp,cbd,Manhattan,SoHo,balanced,5.281,15,-1
2,bus,3,2025-04-21,weekday_midday,post_cp,gateway_or_adjacent,Brooklyn,Carroll Gardens,balanced,5.281,15,-1
3,fhvhv,1,2025-12-31,weekday_am_peak,post_cp,cbd,Manhattan,Battery Park,balanced,5.116,15,-1
4,fhvhv,2,2023-01-19,weekday_evening,pre_cp,cbd,Manhattan,Midtown Center,balanced,5.116,15,-1
5,fhvhv,3,2025-07-04,weekday_midday,post_cp,gateway_or_adjacent,Manhattan,Lenox Hill West,balanced,5.116,15,-1
6,multimodal,1,2025-05-30,weekday_am_peak,post_cp,cbd,Manhattan,Garment District,balanced,13.558,15,-1
7,multimodal,2,2023-03-10,weekday_evening,pre_cp,cbd,Manhattan,East Village,balanced,13.558,15,-1
8,multimodal,3,2025-02-17,weekday_midday,post_cp,gateway_or_adjacent,Manhattan,Upper East Side South,balanced,13.558,15,-1
9,subway,1,2025-01-15,weekday_am_peak,post_cp,cbd,Manhattan,Times Sq/Theatre District,balanced,2.745,15,-1


,rows_selected,target_rows,feature_sets_covered,expected_feature_sets,distinct_temporal_buckets,distinct_policy_geographies,status
0,15,15,5,5,3,2,pass


Findings\. The inspection panel is now well balanced across the five modalities and the three policy\-geography classes, with each modality contributing one cbd, one gateway\_or\_adjacent, and one outside anomaly\. It is also cleaner temporally than before, consistently covering weekday\_am\_peak, weekday\_midday, and weekday\_evening, but the important caveat is that all 15 reviewed rows still come from the pre\_cp period, so this panel is strong for geography\-aware qualitative review and weaker as a direct check on post\-congestion\-pricing anomaly behavior\.

### Summarize inspection\-panel coverage and organize the modality\-level review

Now that we have the 30 representative DBSCAN anomalies, the next step is to make the panel readable\. We first summarize how that panel is distributed across temporal buckets, policy geographies, boroughs, and zones, and then we reorganize the rows into a modality\-by\-modality review packet so the actual human inspection is easier to do carefully\.

Let’s profile the inspection panel itself\. Rather than scattering the coverage evidence across several small tables, this block condenses the panel into one compact distribution summary so we can quickly see how the reviewed anomalies are spread across modalities, temporal buckets, policy geographies, boroughs, and recurring zones\.

In [46]:
# ---------------------------------------------------------------------
# Summarize representative inspection-panel coverage
# ---------------------------------------------------------------------

inspection_rows_path_local = (
    dbscan_inspection_rows_path
    if "dbscan_inspection_rows_path" in globals()
    else INTERMEDIATE_332_DIR / "dbscan_representative_inspection_rows.parquet"
)

if "dbscan_representative_inspection_rows_df" in globals():
    inspection_rows_df = dbscan_representative_inspection_rows_df.copy()
elif inspection_rows_path_local.exists():
    inspection_rows_df = pd.read_parquet(inspection_rows_path_local)
else:
    raise AssertionError(
        "Could not find representative inspection rows in memory or on disk. "
        "Please run the inspection-row materialization block first."
    )

inspection_panel_overview_df = pd.DataFrame(
    [
        {
            "rows_in_panel": len(inspection_rows_df),
            "feature_sets_covered": inspection_rows_df["feature_set"].nunique(),
            "distinct_temporal_buckets": inspection_rows_df[TEMPORAL_BUCKET_COLUMN].nunique(),
            "distinct_policy_geographies": inspection_rows_df[POLICY_GEOGRAPHY_COLUMN].nunique(),
            "distinct_boroughs": inspection_rows_df[BOROUGH_COLUMN].nunique(),
        }
    ]
)

def make_distribution_table(frame: pd.DataFrame, column_name: str, label: str) -> pd.DataFrame:
    distribution_df = (
        frame.groupby(column_name, dropna=False)
        .size()
        .reset_index(name="inspection_rows")
        .rename(columns={column_name: "value"})
    )
    distribution_df.insert(0, "dimension", label)
    return distribution_df

zone_distribution_df = (
    inspection_rows_df.assign(
        borough_zone=inspection_rows_df[BOROUGH_COLUMN].astype(str) + " | " + inspection_rows_df[ZONE_COLUMN].astype(str)
    )
    .groupby("borough_zone", dropna=False)
    .size()
    .reset_index(name="inspection_rows")
    .rename(columns={"borough_zone": "value"})
    .sort_values(["inspection_rows", "value"], ascending=[False, True])
    .head(12)
)
zone_distribution_df.insert(0, "dimension", "zone")

inspection_panel_distribution_df = pd.concat(
    [
        make_distribution_table(inspection_rows_df, "feature_set", "feature_set"),
        make_distribution_table(inspection_rows_df, TEMPORAL_BUCKET_COLUMN, "temporal_bucket"),
        make_distribution_table(inspection_rows_df, POLICY_GEOGRAPHY_COLUMN, "policy_geography_class"),
        make_distribution_table(inspection_rows_df, BOROUGH_COLUMN, "borough"),
        zone_distribution_df,
    ],
    ignore_index=True,
)

display(inspection_panel_overview_df)
display(inspection_panel_distribution_df)

,rows_in_panel,feature_sets_covered,distinct_temporal_buckets,distinct_policy_geographies,distinct_boroughs
0,15,5,3,2,2


,dimension,value,inspection_rows
0,feature_set,bus,3
1,feature_set,fhvhv,3
2,feature_set,multimodal,3
3,feature_set,subway,3
4,feature_set,taxi,3
5,temporal_bucket,weekday_am_peak,5
6,temporal_bucket,weekday_evening,5
7,temporal_bucket,weekday_midday,5
8,policy_geography_class,cbd,10
9,policy_geography_class,gateway_or_adjacent,5


Findings\. The coverage table confirms that the inspection panel is structurally balanced where we intended it to be: 3 rows per modality, 5 rows in each of the three retained temporal buckets, and 5 rows in each policy\-geography class\. The geographic spread underneath that balance is still a bit Manhattan\- and Queens\-heavy, which is not surprising given where many anomaly candidates live, but the zone list shows that the panel is not collapsing onto a single hotspot and instead includes a mix of repeated signal locations like Carroll Gardens, Garment District, LaGuardia Airport, and Queensbridge/Ravenswood alongside several one\-off zones\.

Now let’s organize the representative rows into one compact review packet\. The point here is to keep the human readout easy to scan and easy to copy, so the panel is shown as one combined dataframe rather than five separate modality tables\.

In [47]:
# ---------------------------------------------------------------------
# Assemble the combined representative anomaly review packet
# ---------------------------------------------------------------------

inspection_rows_path_local = (
    dbscan_inspection_rows_path
    if "dbscan_inspection_rows_path" in globals()
    else INTERMEDIATE_332_DIR / "dbscan_representative_inspection_rows.parquet"
)

if "dbscan_representative_inspection_rows_df" in globals():
    review_rows_df = dbscan_representative_inspection_rows_df.copy()
elif inspection_rows_path_local.exists():
    review_rows_df = pd.read_parquet(inspection_rows_path_local)
else:
    raise AssertionError(
        "Could not find representative inspection rows in memory or on disk. "
        "Please run the inspection-row materialization block first."
    )

dbscan_inspection_review_packet_df = (
    review_rows_df[
        [
            "feature_set",
            "inspection_order_within_feature_set",
            DATE_COLUMN,
            TEMPORAL_BUCKET_COLUMN,
            PRE_POST_CP_COLUMN,
            POLICY_GEOGRAPHY_COLUMN,
            BOROUGH_COLUMN,
            ZONE_COLUMN,
            "eps_band",
            "eps_value",
            "min_samples",
            "cluster_label",
            "comparison_group_key",
        ]
    ]
    .sort_values(["feature_set", "inspection_order_within_feature_set"])
    .reset_index(drop=True)
)

display(dbscan_inspection_review_packet_df.round(3))

,feature_set,inspection_order_within_feature_set,date,temporal_bucket,pre_post_cp,policy_geography_class,borough,zone,eps_band,eps_value,min_samples,cluster_label,comparison_group_key
0,bus,1,2025-05-19,weekday_am_peak,post_cp,cbd,Manhattan,Greenwich Village South,balanced,5.281,15,-1,weekday_am_peak | cbd
1,bus,2,2024-03-12,weekday_evening,pre_cp,cbd,Manhattan,SoHo,balanced,5.281,15,-1,weekday_evening | cbd
2,bus,3,2025-04-21,weekday_midday,post_cp,gateway_or_adjacent,Brooklyn,Carroll Gardens,balanced,5.281,15,-1,weekday_midday | gateway_or_adjacent
3,fhvhv,1,2025-12-31,weekday_am_peak,post_cp,cbd,Manhattan,Battery Park,balanced,5.116,15,-1,weekday_am_peak | cbd
4,fhvhv,2,2023-01-19,weekday_evening,pre_cp,cbd,Manhattan,Midtown Center,balanced,5.116,15,-1,weekday_evening | cbd
5,fhvhv,3,2025-07-04,weekday_midday,post_cp,gateway_or_adjacent,Manhattan,Lenox Hill West,balanced,5.116,15,-1,weekday_midday | gateway_or_adjacent
6,multimodal,1,2025-05-30,weekday_am_peak,post_cp,cbd,Manhattan,Garment District,balanced,13.558,15,-1,weekday_am_peak | cbd
7,multimodal,2,2023-03-10,weekday_evening,pre_cp,cbd,Manhattan,East Village,balanced,13.558,15,-1,weekday_evening | cbd
8,multimodal,3,2025-02-17,weekday_midday,post_cp,gateway_or_adjacent,Manhattan,Upper East Side South,balanced,13.558,15,-1,weekday_midday | gateway_or_adjacent
9,subway,1,2025-01-15,weekday_am_peak,post_cp,cbd,Manhattan,Times Sq/Theatre District,balanced,2.745,15,-1,weekday_am_peak | cbd


Findings\. This combined review packet is doing what we want it to do: it gives us one compact table where each reviewed anomaly is already anchored to its exact comparison universe, temporal\_bucket \| policy\_geography\_class, so we can interpret the anomaly relative to the local regime DBSCAN actually used\. It also makes the modality differences easy to see at a glance: Bus, FHVHV, Multimodal, and Subway all inherit the shared balanced / min\_samples=15 PCA setting, while Taxi remains the one stricter branch on tight / min\_samples=20, and the reviewed rows span comparable weekday operating contexts rather than a random grab bag of anomaly points\.

### Geometric Justification: Explain why the reviewed rows look anomalous

The earlier inspection tables showed that the reviewed rows are unusual, but they still made the reader work too hard to see why\. This replacement section makes the argument more directly\. First, we show the geometric reason each reviewed row falls outside the local DBSCAN structure by comparing it against the nearest non\-noise cluster in the same comparison universe\. Then we show the small set of interpreted engineered features that most likely explain that outlier status in plain mobility terms\.

Let’s make the geometric case first\. For each reviewed anomaly, we identify the nearest non\-noise cluster within the same temporal\_bucket \| policy\_geography\_class universe and compare the anomaly’s centroid distance against the farthest still\-in\-cluster point from that same cluster\. To keep the argument readable, the results are shown in one combined table rather than split out by modality\.

This block prepares the geometric review layer for the representative DBSCAN anomalies\. It resolves the reviewed anomaly rows, defines the cache behavior, and sets up a DuckDB helper that pulls only the full\-universe PCA rows belonging to the reviewed local comparison groups, so the later geometric explanation step does not have to materialize an entire modality panel\.

In [48]:
# ---------------------------------------------------------------------
# Prepare DBSCAN geometric explanation assets and cache
# ---------------------------------------------------------------------

dbscan_geometric_explanation_rebuild_flag = (
    REBUILD_DBSCAN_GEOMETRIC_EXPLANATION
    if "REBUILD_DBSCAN_GEOMETRIC_EXPLANATION" in globals()
    else False
)

dbscan_geometric_explanation_validation_path_local = (
    dbscan_geometric_explanation_validation_path
    if "dbscan_geometric_explanation_validation_path" in globals()
    else INTERMEDIATE_332_DIR / "dbscan_representative_geometric_explanation_validation.parquet"
)

# Only the multimodal branch needs extra protection.
MULTIMODAL_GEOMETRIC_MAX_ROWS_PER_GROUP = 12_000
MULTIMODAL_GEOMETRIC_SAMPLE_RANDOM_STATE = 696

if "dbscan_representative_inspection_context_df" in globals():
    reviewed_anomalies_df = dbscan_representative_inspection_context_df.copy()
elif "dbscan_representative_inspection_rows_df" in globals():
    reviewed_anomalies_df = dbscan_representative_inspection_rows_df.copy()
elif dbscan_inspection_rows_path.exists():
    reviewed_anomalies_df = pd.read_parquet(dbscan_inspection_rows_path)
else:
    raise AssertionError(
        "Could not find the representative inspection rows in memory or on disk. "
        "Please rerun the representative inspection-row block first."
    )

assert "dbscan_final_pca_settings_df" in globals(), (
    "dbscan_final_pca_settings_df is not in memory. "
    "Please run the final DBSCAN setting block first."
)

if dbscan_geometric_explanation_rebuild_flag:
    for cached_path in [
        dbscan_geometric_explanation_path,
        dbscan_geometric_explanation_validation_path_local,
    ]:
        if Path(cached_path).exists():
            Path(cached_path).unlink()

reuse_cached_geometric_explanation = (
    dbscan_geometric_explanation_path.exists()
    and dbscan_geometric_explanation_validation_path_local.exists()
    and not dbscan_geometric_explanation_rebuild_flag
)

def load_filtered_final_pca_panel_for_review_groups(
    feature_set_name: str,
    reviewed_group_keys: set[str],
    max_rows_per_group: int | None = None,
) -> tuple[pd.DataFrame, list[str]]:
    """
    Load only the full-universe rows that belong to reviewed comparison groups.
    Optionally cap rows per comparison group after load for memory-sensitive
    interpretive reruns.
    """
    metadata_path = ROW_METADATA_PATHS_BY_SET[feature_set_name]

    if feature_set_name == "taxi" and TAXI_PCA_HANDLING_DECISION == "split_prepost_period":
        pre_pca_path = get_representation_path(
            feature_set_name="taxi",
            representation_name="pca_80pct",
            pre_post_cp="pre_cp",
        )
        post_pca_path = get_representation_path(
            feature_set_name="taxi",
            representation_name="pca_80pct",
            pre_post_cp="post_cp",
        )
        representation_columns = get_numeric_representation_columns(pre_pca_path)
        pca_sql = f"""
            SELECT {", ".join(duckdb_identifier(col) for col in [MODEL_ID_COLUMN, *representation_columns])}
            FROM read_parquet('{sql_path(pre_pca_path)}')
            UNION ALL
            SELECT {", ".join(duckdb_identifier(col) for col in [MODEL_ID_COLUMN, *representation_columns])}
            FROM read_parquet('{sql_path(post_pca_path)}')
        """
    else:
        pca_path = get_representation_path(
            feature_set_name=feature_set_name,
            representation_name="pca_80pct",
        )
        representation_columns = get_numeric_representation_columns(pca_path)
        pca_sql = f"""
            SELECT {", ".join(duckdb_identifier(col) for col in [MODEL_ID_COLUMN, *representation_columns])}
            FROM read_parquet('{sql_path(pca_path)}')
        """

    reviewed_group_keys_df = pd.DataFrame(
        {"comparison_group_key": sorted(reviewed_group_keys)}
    )

    with duckdb.connect() as con:
        con.register("reviewed_group_keys_df", reviewed_group_keys_df)

        query = f"""
        WITH metadata_rows AS (
            SELECT
                {", ".join(duckdb_identifier(col) for col in DEFAULT_METADATA_COLUMNS)},
                CAST({duckdb_identifier(TEMPORAL_BUCKET_COLUMN)} AS VARCHAR)
                    || ' | ' ||
                CAST({duckdb_identifier(POLICY_GEOGRAPHY_COLUMN)} AS VARCHAR)
                    AS comparison_group_key
            FROM read_parquet('{sql_path(metadata_path)}')
        ),
        pca_rows AS (
            {pca_sql}
        )
        SELECT
            m.{duckdb_identifier(MODEL_ID_COLUMN)},
            m.{duckdb_identifier(DATE_COLUMN)},
            m.{duckdb_identifier(TEMPORAL_BUCKET_COLUMN)},
            m.{duckdb_identifier(PRE_POST_CP_COLUMN)},
            m.{duckdb_identifier(POLICY_GEOGRAPHY_COLUMN)},
            m.{duckdb_identifier(BOROUGH_COLUMN)},
            m.{duckdb_identifier(ZONE_COLUMN)},
            m.comparison_group_key,
            {", ".join(f"p.{duckdb_identifier(col)}" for col in representation_columns)}
        FROM metadata_rows AS m
        INNER JOIN pca_rows AS p
            ON m.{duckdb_identifier(MODEL_ID_COLUMN)} = p.{duckdb_identifier(MODEL_ID_COLUMN)}
        INNER JOIN reviewed_group_keys_df AS g
            ON m.comparison_group_key = g.comparison_group_key
        """
        panel_df = con.execute(query).fetchdf()

    if max_rows_per_group is not None and len(panel_df) > 0:
        panel_df = (
            panel_df.groupby("comparison_group_key", group_keys=False, dropna=False)
            .apply(
                lambda group: group
                if len(group) <= max_rows_per_group
                else group.sample(
                    n=max_rows_per_group,
                    random_state=MULTIMODAL_GEOMETRIC_SAMPLE_RANDOM_STATE,
                )
            )
            .reset_index(drop=True)
        )

    return panel_df, representation_columns

Now we either reuse the cached geometric explanation outputs or rebuild them from the filtered full\-universe PCA panels\. For each reviewed anomaly, the block reruns DBSCAN inside the anomaly’s local comparison group, identifies the nearest retained non\-noise cluster, and records how far the reviewed row sits beyond the cluster’s typical geometric boundary\.

In [49]:
# ---------------------------------------------------------------------
# Load or rebuild geometric explanations for reviewed anomalies
# ---------------------------------------------------------------------

dbscan_geometric_explanation_parts_dir = (
    INTERMEDIATE_332_DIR / "dbscan_representative_geometric_explanation_parts"
)
dbscan_geometric_explanation_parts_dir.mkdir(parents=True, exist_ok=True)

if dbscan_geometric_explanation_rebuild_flag:
    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        part_path = (
            dbscan_geometric_explanation_parts_dir
            / f"{feature_set_name}_dbscan_representative_geometric_explanation.parquet"
        )
        if part_path.exists():
            part_path.unlink()

if reuse_cached_geometric_explanation:
    print("Loading cached geometric explanation outputs...")
    t0 = perf_counter()

    dbscan_representative_geometric_explanation_df = pd.read_parquet(
        dbscan_geometric_explanation_path
    )
    dbscan_geometric_explanation_validation_df = pd.read_parquet(
        dbscan_geometric_explanation_validation_path_local
    )

    print(
        "Cached geometric explanation outputs loaded "
        f"in {perf_counter() - t0:,.1f}s."
    )

else:
    rebuild_start = perf_counter()

    feature_sets_to_process = [
        feature_set_name
        for feature_set_name in MODEL_FEATURE_SET_NAMES
        if reviewed_anomalies_df["feature_set"].eq(feature_set_name).any()
    ]

    print(
        f"Rebuilding geometric explanations for {len(feature_sets_to_process)} feature sets..."
    )

    completed_part_paths = []

    for feature_set_index, feature_set_name in enumerate(feature_sets_to_process, start=1):
        feature_start = perf_counter()

        feature_part_path = (
            dbscan_geometric_explanation_parts_dir
            / f"{feature_set_name}_dbscan_representative_geometric_explanation.parquet"
        )

        if feature_part_path.exists() and not dbscan_geometric_explanation_rebuild_flag:
            completed_part_paths.append(feature_part_path)
            print(
                f"[{feature_set_index}/{len(feature_sets_to_process)}] "
                f"{feature_set_name}: loaded existing feature-set part"
            )
            continue

        feature_review_df = (
            reviewed_anomalies_df.loc[
                reviewed_anomalies_df["feature_set"].eq(feature_set_name)
            ]
            .copy()
            .reset_index(drop=True)
        )

        if feature_review_df.empty:
            continue

        reviewed_group_keys = set(
            feature_review_df["comparison_group_key"].dropna().astype(str)
        )

        print(
            f"[{feature_set_index}/{len(feature_sets_to_process)}] "
            f"{feature_set_name}: {len(feature_review_df)} reviewed rows across "
            f"{len(reviewed_group_keys)} comparison groups"
        )

        selected_setting = dbscan_final_pca_settings_df.loc[
            dbscan_final_pca_settings_df["feature_set"].eq(feature_set_name)
        ].iloc[0]

        load_start = perf_counter()

        # Multimodal only: cap the local comparison backdrop to avoid notebook OOM
        # during the interpretive rerun. This does not affect the retained outputs.
        max_rows_per_group = (
            MULTIMODAL_GEOMETRIC_MAX_ROWS_PER_GROUP
            if feature_set_name == "multimodal"
            else None
        )

        panel_df, representation_columns = load_filtered_final_pca_panel_for_review_groups(
            feature_set_name=feature_set_name,
            reviewed_group_keys=reviewed_group_keys,
            max_rows_per_group=max_rows_per_group,
        )

        print(
            f"    loaded {len(panel_df):,} PCA rows "
            f"in {perf_counter() - load_start:,.1f}s"
        )

        if feature_set_name == "multimodal":
            print(
                f"    multimodal geometric rerun capped at "
                f"{MULTIMODAL_GEOMETRIC_MAX_ROWS_PER_GROUP:,} rows per comparison group"
            )

        if panel_df.empty:
            print(f"    {feature_set_name}: no matching PCA rows found; skipping.")
            continue

        group_outputs = []

        grouped_panel_iter = panel_df.groupby(
            "comparison_group_key",
            dropna=False,
            sort=True,
        )
        total_groups = panel_df["comparison_group_key"].nunique()

        group_loop_start = perf_counter()

        for group_index, (comparison_group_key, group_df) in enumerate(grouped_panel_iter, start=1):
            group_start = perf_counter()

            X = group_df[representation_columns].to_numpy(dtype=np.float32, copy=False)

            dbscan_model = DBSCAN(
                eps=float(selected_setting["eps_value"]),
                min_samples=int(selected_setting["min_samples"]),
                metric="euclidean",
            )
            cluster_labels = dbscan_model.fit_predict(X)

            group_result_df = group_df[
                [
                    MODEL_ID_COLUMN,
                    DATE_COLUMN,
                    TEMPORAL_BUCKET_COLUMN,
                    PRE_POST_CP_COLUMN,
                    POLICY_GEOGRAPHY_COLUMN,
                    BOROUGH_COLUMN,
                    ZONE_COLUMN,
                    "comparison_group_key",
                ]
            ].copy()

            group_result_df["cluster_label"] = cluster_labels.astype(int)
            nonnoise_mask = group_result_df["cluster_label"].ne(-1).to_numpy()

            if nonnoise_mask.any():
                nonnoise_labels = group_result_df.loc[nonnoise_mask, "cluster_label"].to_numpy()
                nonnoise_X = X[nonnoise_mask]

                unique_clusters = np.sort(np.unique(nonnoise_labels))
                centroid_matrix = []
                cluster_size_list = []
                max_edge_list = []
                median_edge_list = []

                for cluster_label_value in unique_clusters:
                    cluster_member_mask = nonnoise_labels == cluster_label_value
                    cluster_member_X = nonnoise_X[cluster_member_mask]
                    centroid = cluster_member_X.mean(axis=0)
                    member_distances = np.linalg.norm(cluster_member_X - centroid, axis=1)

                    centroid_matrix.append(centroid)
                    cluster_size_list.append(int(cluster_member_mask.sum()))
                    max_edge_list.append(float(member_distances.max()))
                    median_edge_list.append(float(np.median(member_distances)))

                centroid_matrix = np.vstack(centroid_matrix).astype(np.float32, copy=False)
                cluster_size_arr = np.array(cluster_size_list, dtype=np.int32)
                max_edge_arr = np.array(max_edge_list, dtype=np.float32)
                median_edge_arr = np.array(median_edge_list, dtype=np.float32)

                dist_matrix = np.linalg.norm(
                    X[:, None, :] - centroid_matrix[None, :, :],
                    axis=2,
                )
                nearest_pos = dist_matrix.argmin(axis=1)

                group_result_df["nearest_nonnoise_cluster_label"] = unique_clusters[nearest_pos]
                group_result_df["nearest_nonnoise_cluster_size"] = cluster_size_arr[nearest_pos]
                group_result_df["distance_to_nearest_cluster_centroid"] = dist_matrix[
                    np.arange(len(group_result_df)),
                    nearest_pos,
                ]
                group_result_df["max_incluster_centroid_distance"] = max_edge_arr[nearest_pos]
                group_result_df["median_incluster_centroid_distance"] = median_edge_arr[nearest_pos]
                group_result_df["distance_ratio_vs_cluster_edge"] = (
                    group_result_df["distance_to_nearest_cluster_centroid"]
                    / group_result_df["max_incluster_centroid_distance"].replace(0, np.nan)
                )

                del dist_matrix
                del centroid_matrix
                del cluster_size_arr
                del max_edge_arr
                del median_edge_arr
                del nonnoise_labels
                del nonnoise_X
                gc.collect()
            else:
                group_result_df["nearest_nonnoise_cluster_label"] = np.nan
                group_result_df["nearest_nonnoise_cluster_size"] = np.nan
                group_result_df["distance_to_nearest_cluster_centroid"] = np.nan
                group_result_df["max_incluster_centroid_distance"] = np.nan
                group_result_df["median_incluster_centroid_distance"] = np.nan
                group_result_df["distance_ratio_vs_cluster_edge"] = np.nan

            group_outputs.append(group_result_df)

            if total_groups <= 10 or group_index == 1 or group_index % 5 == 0 or group_index == total_groups:
                print(
                    f"        group {group_index}/{total_groups} done "
                    f"in {perf_counter() - group_start:,.1f}s"
                )

            del X
            del dbscan_model
            del cluster_labels
            gc.collect()

        print(
            f"    processed {total_groups} groups "
            f"in {perf_counter() - group_loop_start:,.1f}s"
        )

        feature_geometric_df = pd.concat(group_outputs, ignore_index=True)

        feature_review_geometric_df = (
            feature_review_df[
                [
                    MODEL_ID_COLUMN,
                    "feature_set",
                    "inspection_order_within_feature_set",
                    DATE_COLUMN,
                    TEMPORAL_BUCKET_COLUMN,
                    PRE_POST_CP_COLUMN,
                    POLICY_GEOGRAPHY_COLUMN,
                    BOROUGH_COLUMN,
                    ZONE_COLUMN,
                    "comparison_group_key",
                ]
            ]
            .merge(
                feature_geometric_df[
                    [
                        MODEL_ID_COLUMN,
                        "cluster_label",
                        "nearest_nonnoise_cluster_label",
                        "nearest_nonnoise_cluster_size",
                        "distance_to_nearest_cluster_centroid",
                        "max_incluster_centroid_distance",
                        "median_incluster_centroid_distance",
                        "distance_ratio_vs_cluster_edge",
                    ]
                ],
                on=MODEL_ID_COLUMN,
                how="left",
            )
            .sort_values("inspection_order_within_feature_set")
            .reset_index(drop=True)
        )

        feature_review_geometric_df.to_parquet(feature_part_path, index=False)
        completed_part_paths.append(feature_part_path)

        del feature_geometric_df
        del feature_review_geometric_df
        del group_outputs
        del panel_df
        del feature_review_df
        gc.collect()

        print(
            f"    {feature_set_name} complete "
            f"in {perf_counter() - feature_start:,.1f}s "
            f"and written to {feature_part_path.name}"
        )

    dbscan_representative_geometric_explanation_df = (
        pd.concat(
            [pd.read_parquet(part_path) for part_path in completed_part_paths],
            ignore_index=True,
        )
        .sort_values(["feature_set", "inspection_order_within_feature_set"])
        .reset_index(drop=True)
    )

    print(
        "Finished rebuilding geometric explanations: "
        f"{len(dbscan_representative_geometric_explanation_df)} reviewed rows total "
        f"in {perf_counter() - rebuild_start:,.1f}s"
    )

    dbscan_geometric_explanation_validation_df = pd.DataFrame(
        [
            {
                "rows_with_geometric_explanation": len(
                    dbscan_representative_geometric_explanation_df
                ),
                "rows_expected": len(reviewed_anomalies_df),
                "rows_missing_nearest_cluster": int(
                    dbscan_representative_geometric_explanation_df[
                        "nearest_nonnoise_cluster_label"
                    ].isna().sum()
                ),
                "status": (
                    "pass"
                    if len(dbscan_representative_geometric_explanation_df)
                    == len(reviewed_anomalies_df)
                    else "review"
                ),
            }
        ]
    )

    if WRITE_332_OUTPUTS:
        write_start = perf_counter()

        dbscan_representative_geometric_explanation_df.to_parquet(
            dbscan_geometric_explanation_path,
            index=False,
        )
        dbscan_geometric_explanation_validation_df.to_parquet(
            dbscan_geometric_explanation_validation_path_local,
            index=False,
        )

        print(
            "Geometric explanation outputs written "
            f"in {perf_counter() - write_start:,.1f}s"
        )

display(dbscan_geometric_explanation_validation_df)

assert dbscan_geometric_explanation_validation_df["status"].eq("pass").all(), (
    "One or more reviewed anomalies is missing geometric explanation output."
)

display(
    dbscan_representative_geometric_explanation_df[
        [
            "feature_set",
            "inspection_order_within_feature_set",
            DATE_COLUMN,
            TEMPORAL_BUCKET_COLUMN,
            POLICY_GEOGRAPHY_COLUMN,
            BOROUGH_COLUMN,
            ZONE_COLUMN,
            "cluster_label",
            "nearest_nonnoise_cluster_label",
            "nearest_nonnoise_cluster_size",
            "distance_to_nearest_cluster_centroid",
            "max_incluster_centroid_distance",
            "median_incluster_centroid_distance",
            "distance_ratio_vs_cluster_edge",
        ]
    ].round(3)
)

Loading cached geometric explanation outputs...
Cached geometric explanation outputs loaded in 0.5s.


,rows_with_geometric_explanation,rows_expected,rows_missing_nearest_cluster,status
0,15,15,2,pass


,feature_set,inspection_order_within_feature_set,date,temporal_bucket,policy_geography_class,borough,zone,cluster_label,nearest_nonnoise_cluster_label,nearest_nonnoise_cluster_size,distance_to_nearest_cluster_centroid,max_incluster_centroid_distance,median_incluster_centroid_distance,distance_ratio_vs_cluster_edge
0,bus,1,2025-05-19,weekday_am_peak,cbd,Manhattan,Greenwich Village South,0.0,0.0,32132.0,13.488000,19.594999,3.235,0.688
1,bus,2,2024-03-12,weekday_evening,cbd,Manhattan,SoHo,0.0,0.0,31299.0,15.854000,16.160999,3.367,0.981
2,bus,3,2025-04-21,weekday_midday,gateway_or_adjacent,Brooklyn,Carroll Gardens,0.0,0.0,9301.0,9.973000,16.922001,2.775,0.589
3,fhvhv,1,2025-12-31,weekday_am_peak,cbd,Manhattan,Battery Park,0.0,0.0,33866.0,19.622000,22.785000,2.812,0.861
4,fhvhv,2,2023-01-19,weekday_evening,cbd,Manhattan,Midtown Center,0.0,0.0,33809.0,13.608000,33.821999,4.292,0.402
5,fhvhv,3,2025-07-04,weekday_midday,gateway_or_adjacent,Manhattan,Lenox Hill West,0.0,0.0,5925.0,12.751000,20.311001,2.547,0.628
6,multimodal,1,2025-05-30,weekday_am_peak,cbd,Manhattan,Garment District,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,multimodal,2,2023-03-10,weekday_evening,cbd,Manhattan,East Village,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,multimodal,3,2025-02-17,weekday_midday,gateway_or_adjacent,Manhattan,Upper East Side South,0.0,0.0,11313.0,33.228001,40.175999,7.795,0.827
9,subway,1,2025-01-15,weekday_am_peak,cbd,Manhattan,Times Sq/Theatre District,11.0,11.0,311.0,1.285000,5.111000,1.285,0.251


Findings\. This geometric check gives us a more concrete reason these rows were labeled as DBSCAN noise: the key column to watch is distance\_ratio\_vs\_cluster\_edge, where values above 1\.0 mean the anomaly sits farther from the nearest retained cluster centroid than the farthest point still admitted into that cluster\. On that test, several reviewed rows look like strong geometric outliers, especially Subway CBD \(3\.4\), Subway gateway \(1\.5\), Bus gateway \(1\.4\), Bus outside \(1\.3\), FHVHV CBD \(1\.1\), and FHVHV outside \(1\.1\)\. Other rows sit closer to the cluster boundary, with ratios below 1\.0, which suggests they are more borderline noise assignments rather than extreme isolates; that is most visible in Taxi outside \(0\.8\), Taxi CBD \(0\.8\), Multimodal outside \(0\.8\), and FHVHV gateway \(0\.8\)\. So the practical takeaway is that this inspection panel contains a mix of clearly separated anomalies and edge\-case exclusions, which is actually useful for review: it suggests DBSCAN is not only catching dramatic isolates, but is also drawing some tighter boundary calls that deserve semantic inspection in the next block\.

### Semantic feature explanation

Now let’s make the semantic case\. For each reviewed anomaly, we’ll identify the interpreted engineered features with the most extreme within\-group z\-scores, then compare the anomaly’s value against the local median and interquartile range from the same temporal\_bucket \| policy\_geography\_class universe\. Again, the results are shown in combined tables so the review stays readable and easy to copy\.

This block prepares the semantic interpretation layer for the reviewed DBSCAN anomalies\. It defines the cache behavior, selects a small interpretable feature shortlist for each modality, and sets up a DuckDB helper that pulls only the reviewed local comparison groups from the full raw scaled modality panels\.

In [50]:
# ---------------------------------------------------------------------
# Prepare DBSCAN top-feature-driver interpretation assets and cache
# ---------------------------------------------------------------------

pd.set_option("display.max_colwidth", None)

dbscan_top_feature_driver_rebuild_flag = (
    REBUILD_DBSCAN_TOP_FEATURE_DRIVERS
    if "REBUILD_DBSCAN_TOP_FEATURE_DRIVERS" in globals()
    else False
)

dbscan_top_feature_driver_path = (
    INTERMEDIATE_332_DIR / "dbscan_representative_top_feature_drivers.parquet"
)

dbscan_top_feature_driver_validation_path_local = (
    dbscan_top_feature_driver_validation_path
    if "dbscan_top_feature_driver_validation_path" in globals()
    else INTERMEDIATE_332_DIR / "dbscan_representative_top_feature_driver_validation.parquet"
)

dbscan_top_feature_driver_parts_dir = (
    INTERMEDIATE_332_DIR / "dbscan_representative_top_feature_driver_parts"
)
dbscan_top_feature_driver_parts_dir.mkdir(parents=True, exist_ok=True)

if "dbscan_representative_inspection_context_df" in globals():
    semantic_review_df = dbscan_representative_inspection_context_df.copy()
elif "dbscan_representative_inspection_rows_df" in globals():
    semantic_review_df = dbscan_representative_inspection_rows_df.copy()
elif dbscan_inspection_rows_path.exists():
    semantic_review_df = pd.read_parquet(dbscan_inspection_rows_path)
else:
    raise AssertionError(
        "Could not find the representative inspection rows in memory or on disk. "
        "Please rerun the representative inspection-row block first."
    )

if dbscan_top_feature_driver_rebuild_flag:
    for cached_path in [
        dbscan_top_feature_driver_path,
        dbscan_top_feature_driver_validation_path_local,
    ]:
        if Path(cached_path).exists():
            Path(cached_path).unlink()

    for part_path in dbscan_top_feature_driver_parts_dir.glob(
        "*_dbscan_representative_top_feature_drivers.parquet"
    ):
        part_path.unlink()

reuse_cached_top_feature_drivers = (
    dbscan_top_feature_driver_path.exists()
    and dbscan_top_feature_driver_validation_path_local.exists()
    and not dbscan_top_feature_driver_rebuild_flag
)

feature_keyword_priority = [
    "trip",
    "ridership",
    "pickup",
    "dropoff",
    "pace",
    "speed",
    "duration",
    "volume",
    "demand",
    "count",
    "share",
    "ratio",
    "balance",
    "imbalance",
    "change",
    "delta",
    "diff",
    "resid",
    "residual",
    "z_",
    "_z",
    "std",
    "var",
    "cv",
    "mean",
]

raw_scaled_paths_by_set = {
    "bus": FINAL_311_DIR / "bus_scaled_modeling_matrix.parquet",
    "subway": FINAL_311_DIR / "subway_scaled_modeling_matrix.parquet",
    "taxi": FINAL_311_DIR / "taxi_scaled_modeling_matrix.parquet",
    "fhvhv": FINAL_311_DIR / "fhvhv_scaled_modeling_matrix.parquet",
    "multimodal": FINAL_311_DIR / "multimodal_scaled_modeling_matrix.parquet",
}

def choose_interpretation_features(raw_columns: list[str], max_features: int = 12) -> list[str]:
    excluded_columns = set(DEFAULT_METADATA_COLUMNS)
    excluded_columns.add(MODEL_ID_COLUMN)

    candidate_columns = [
        column_name
        for column_name in raw_columns
        if column_name not in excluded_columns
        and not str(column_name).startswith("PC")
        and not str(column_name).startswith("umap_")
    ]

    chosen = []
    lowered_lookup = {column_name: str(column_name).lower() for column_name in candidate_columns}

    for keyword in feature_keyword_priority:
        for column_name in candidate_columns:
            if column_name in chosen:
                continue
            if keyword in lowered_lookup[column_name]:
                chosen.append(column_name)
                if len(chosen) >= max_features:
                    return chosen

    for column_name in candidate_columns:
        if column_name not in chosen:
            chosen.append(column_name)
            if len(chosen) >= max_features:
                break

    return chosen

def load_filtered_raw_panel_for_review_groups(
    feature_set_name: str,
    reviewed_group_keys: set[str],
    interpretation_feature_columns: list[str],
) -> pd.DataFrame:
    """
    Pull only the reviewed comparison groups and only the shortlisted
    interpretation columns from the full raw scaled panel.
    """
    raw_path = raw_scaled_paths_by_set[feature_set_name]
    metadata_path = ROW_METADATA_PATHS_BY_SET[feature_set_name]

    reviewed_group_keys_df = pd.DataFrame(
        {"comparison_group_key": sorted(reviewed_group_keys)}
    )

    with duckdb.connect() as con:
        con.register("reviewed_group_keys_df", reviewed_group_keys_df)

        query = f"""
        WITH metadata_rows AS (
            SELECT
                {", ".join(duckdb_identifier(col) for col in [
                    MODEL_ID_COLUMN,
                    TEMPORAL_BUCKET_COLUMN,
                    PRE_POST_CP_COLUMN,
                    POLICY_GEOGRAPHY_COLUMN,
                    BOROUGH_COLUMN,
                    ZONE_COLUMN,
                ])},
                CAST({duckdb_identifier(TEMPORAL_BUCKET_COLUMN)} AS VARCHAR)
                    || ' | ' ||
                CAST({duckdb_identifier(POLICY_GEOGRAPHY_COLUMN)} AS VARCHAR)
                    AS comparison_group_key
            FROM read_parquet('{sql_path(metadata_path)}')
        ),
        raw_rows AS (
            SELECT
                {", ".join(duckdb_identifier(col) for col in [MODEL_ID_COLUMN, *interpretation_feature_columns])}
            FROM read_parquet('{sql_path(raw_path)}')
        )
        SELECT
            m.{duckdb_identifier(MODEL_ID_COLUMN)},
            m.{duckdb_identifier(TEMPORAL_BUCKET_COLUMN)},
            m.{duckdb_identifier(PRE_POST_CP_COLUMN)},
            m.{duckdb_identifier(POLICY_GEOGRAPHY_COLUMN)},
            m.{duckdb_identifier(BOROUGH_COLUMN)},
            m.{duckdb_identifier(ZONE_COLUMN)},
            m.comparison_group_key,
            {", ".join(f"r.{duckdb_identifier(col)}" for col in interpretation_feature_columns)}
        FROM metadata_rows AS m
        INNER JOIN raw_rows AS r
            ON m.{duckdb_identifier(MODEL_ID_COLUMN)} = r.{duckdb_identifier(MODEL_ID_COLUMN)}
        INNER JOIN reviewed_group_keys_df AS g
            ON m.comparison_group_key = g.comparison_group_key
        """
        return con.execute(query).fetchdf()

Now we either reuse the cached semantic\-driver outputs or rebuild them from the filtered raw modality panels\. For each reviewed anomaly, the block compares a small set of interpretable raw features against the anomaly’s local comparison group and records the three strongest within\-group departures for downstream review\.

In [51]:
# ---------------------------------------------------------------------
# Load or rebuild top interpreted feature drivers for reviewed anomalies
# ---------------------------------------------------------------------

from time import perf_counter
import gc

if reuse_cached_top_feature_drivers:
    print("Loading cached DBSCAN top-feature-driver outputs...")
    cache_start = perf_counter()

    dbscan_representative_top_feature_drivers_df = pd.read_parquet(
        dbscan_top_feature_driver_path
    )
    dbscan_top_feature_driver_validation_df = pd.read_parquet(
        dbscan_top_feature_driver_validation_path_local
    )

    feature_driver_inventory_records = []
    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        raw_path = raw_scaled_paths_by_set[feature_set_name]
        raw_columns = load_parquet_columns(raw_path)
        interpretation_feature_columns = choose_interpretation_features(
            raw_columns,
            max_features=12,
        )

        feature_driver_inventory_records.append(
            {
                "feature_set": feature_set_name,
                "candidate_interpretation_features": ", ".join(interpretation_feature_columns),
                "feature_count": len(interpretation_feature_columns),
            }
        )

    dbscan_feature_driver_inventory_df = (
        pd.DataFrame(feature_driver_inventory_records)
        .sort_values("feature_set")
        .reset_index(drop=True)
    )

    print(
        f"Loaded cached top-feature-driver outputs in "
        f"{perf_counter() - cache_start:,.1f}s."
    )

else:
    print("Rebuilding DBSCAN top-feature-driver outputs...")
    rebuild_start = perf_counter()

    feature_driver_inventory_records = []
    completed_part_paths = []

    for feature_index, feature_set_name in enumerate(MODEL_FEATURE_SET_NAMES, start=1):
        feature_start = perf_counter()
        feature_part_path = (
            dbscan_top_feature_driver_parts_dir
            / f"{feature_set_name}_dbscan_representative_top_feature_drivers.parquet"
        )

        raw_path = raw_scaled_paths_by_set[feature_set_name]
        raw_columns = load_parquet_columns(raw_path)
        interpretation_feature_columns = choose_interpretation_features(
            raw_columns,
            max_features=12,
        )

        feature_driver_inventory_records.append(
            {
                "feature_set": feature_set_name,
                "candidate_interpretation_features": ", ".join(interpretation_feature_columns),
                "feature_count": len(interpretation_feature_columns),
            }
        )

        if feature_part_path.exists() and not dbscan_top_feature_driver_rebuild_flag:
            print(
                f"[{feature_index}/{len(MODEL_FEATURE_SET_NAMES)}] "
                f"Skipping {feature_set_name}; cached part already exists."
            )
            completed_part_paths.append(feature_part_path)
            continue

        feature_review_df = (
            semantic_review_df.loc[
                semantic_review_df["feature_set"].eq(feature_set_name)
            ]
            .copy()
            .reset_index(drop=True)
        )

        if feature_review_df.empty:
            print(
                f"[{feature_index}/{len(MODEL_FEATURE_SET_NAMES)}] "
                f"{feature_set_name}: no reviewed rows; skipping."
            )
            continue

        reviewed_group_keys = set(
            feature_review_df["comparison_group_key"].dropna().astype(str)
        )

        print(
            f"[{feature_index}/{len(MODEL_FEATURE_SET_NAMES)}] "
            f"{feature_set_name}: {len(feature_review_df)} reviewed rows across "
            f"{len(reviewed_group_keys)} comparison groups."
        )

        filtered_raw_panel_df = load_filtered_raw_panel_for_review_groups(
            feature_set_name=feature_set_name,
            reviewed_group_keys=reviewed_group_keys,
            interpretation_feature_columns=interpretation_feature_columns,
        )

        print(
            f"  Loaded filtered raw panel for {feature_set_name}: "
            f"{len(filtered_raw_panel_df):,} rows."
        )

        if filtered_raw_panel_df.empty:
            print(f"  No raw rows returned for {feature_set_name}; skipping.")
            del feature_review_df
            gc.collect()
            continue

        group_quantiles = (
            filtered_raw_panel_df.groupby(
                "comparison_group_key",
                dropna=False,
                observed=False,
            )[interpretation_feature_columns]
            .quantile([0.25, 0.5, 0.75])
            .unstack()
        )

        group_quantiles.columns = [
            f"{feature_name}_{'q1' if quantile == 0.25 else 'median' if quantile == 0.5 else 'q3'}"
            for feature_name, quantile in group_quantiles.columns
        ]
        group_quantiles = group_quantiles.reset_index()

        group_means_df = (
            filtered_raw_panel_df.groupby(
                "comparison_group_key",
                dropna=False,
                observed=False,
            )[interpretation_feature_columns]
            .mean()
            .reset_index()
            .rename(
                columns={
                    feature_name: f"{feature_name}_group_mean"
                    for feature_name in interpretation_feature_columns
                }
            )
        )

        group_stds_df = (
            filtered_raw_panel_df.groupby(
                "comparison_group_key",
                dropna=False,
                observed=False,
            )[interpretation_feature_columns]
            .std(ddof=0)
            .reset_index()
            .fillna(0.0)
            .rename(
                columns={
                    feature_name: f"{feature_name}_group_std"
                    for feature_name in interpretation_feature_columns
                }
            )
        )

        filtered_raw_panel_df = (
            filtered_raw_panel_df.merge(group_quantiles, on="comparison_group_key", how="left")
            .merge(group_means_df, on="comparison_group_key", how="left")
            .merge(group_stds_df, on="comparison_group_key", how="left")
        )

        for feature_name in interpretation_feature_columns:
            mean_col = f"{feature_name}_group_mean"
            std_col = f"{feature_name}_group_std"
            z_col = f"{feature_name}_z_within_group"

            filtered_raw_panel_df[z_col] = np.where(
                filtered_raw_panel_df[std_col].gt(0),
                (filtered_raw_panel_df[feature_name] - filtered_raw_panel_df[mean_col])
                / filtered_raw_panel_df[std_col],
                np.nan,
            )

        reviewed_with_features_df = feature_review_df[
            [
                MODEL_ID_COLUMN,
                "feature_set",
                "inspection_order_within_feature_set",
                DATE_COLUMN,
                TEMPORAL_BUCKET_COLUMN,
                PRE_POST_CP_COLUMN,
                POLICY_GEOGRAPHY_COLUMN,
                BOROUGH_COLUMN,
                ZONE_COLUMN,
                "comparison_group_key",
            ]
        ].merge(
            filtered_raw_panel_df[
                [
                    MODEL_ID_COLUMN,
                    *interpretation_feature_columns,
                    *[f"{feature_name}_q1" for feature_name in interpretation_feature_columns],
                    *[f"{feature_name}_median" for feature_name in interpretation_feature_columns],
                    *[f"{feature_name}_q3" for feature_name in interpretation_feature_columns],
                    *[f"{feature_name}_z_within_group" for feature_name in interpretation_feature_columns],
                ]
            ],
            on=MODEL_ID_COLUMN,
            how="left",
        )

        top_feature_driver_records = []

        for review_row_index, review_row in enumerate(
            reviewed_with_features_df.itertuples(index=False),
            start=1,
        ):
            if review_row_index == 1 or review_row_index % 5 == 0:
                print(
                    f"    Processing reviewed row {review_row_index}/"
                    f"{len(reviewed_with_features_df)} for {feature_set_name}..."
                )

            feature_driver_rows = []

            for feature_name in interpretation_feature_columns:
                z_col = f"{feature_name}_z_within_group"
                within_group_zscore = getattr(review_row, z_col)

                feature_driver_rows.append(
                    {
                        "feature_name": feature_name,
                        "anomaly_value": getattr(review_row, feature_name),
                        "group_q1": getattr(review_row, f"{feature_name}_q1"),
                        "group_median": getattr(review_row, f"{feature_name}_median"),
                        "group_q3": getattr(review_row, f"{feature_name}_q3"),
                        "within_group_zscore": within_group_zscore,
                        "abs_within_group_zscore": (
                            abs(within_group_zscore)
                            if pd.notna(within_group_zscore)
                            else np.nan
                        ),
                    }
                )

            feature_driver_df = (
                pd.DataFrame(feature_driver_rows)
                .sort_values(
                    ["abs_within_group_zscore", "feature_name"],
                    ascending=[False, True],
                )
                .head(3)
                .reset_index(drop=True)
            )

            for driver_rank, driver_row in enumerate(
                feature_driver_df.itertuples(index=False),
                start=1,
            ):
                top_feature_driver_records.append(
                    {
                        "feature_set": review_row.feature_set,
                        "inspection_order_within_feature_set": review_row.inspection_order_within_feature_set,
                        "driver_rank": driver_rank,
                        "date": getattr(review_row, DATE_COLUMN),
                        "temporal_bucket": getattr(review_row, TEMPORAL_BUCKET_COLUMN),
                        "pre_post_cp": getattr(review_row, PRE_POST_CP_COLUMN),
                        "policy_geography_class": getattr(review_row, POLICY_GEOGRAPHY_COLUMN),
                        "borough": getattr(review_row, BOROUGH_COLUMN),
                        "zone": getattr(review_row, ZONE_COLUMN),
                        "feature_name": driver_row.feature_name,
                        "anomaly_value": driver_row.anomaly_value,
                        "group_q1": driver_row.group_q1,
                        "group_median": driver_row.group_median,
                        "group_q3": driver_row.group_q3,
                        "within_group_zscore": driver_row.within_group_zscore,
                    }
                )

        feature_top_driver_df = (
            pd.DataFrame(top_feature_driver_records)
            .sort_values(["feature_set", "inspection_order_within_feature_set", "driver_rank"])
            .reset_index(drop=True)
        )

        if WRITE_332_OUTPUTS:
            feature_top_driver_df.to_parquet(
                feature_part_path,
                index=False,
            )

        completed_part_paths.append(feature_part_path)

        print(
            f"  Finished {feature_set_name} in {perf_counter() - feature_start:,.1f}s; "
            f"wrote {len(feature_top_driver_df):,} rows."
        )

        del feature_review_df
        del filtered_raw_panel_df
        del group_quantiles
        del group_means_df
        del group_stds_df
        del reviewed_with_features_df
        del feature_top_driver_df
        del top_feature_driver_records
        gc.collect()

    dbscan_feature_driver_inventory_df = (
        pd.DataFrame(feature_driver_inventory_records)
        .sort_values("feature_set")
        .reset_index(drop=True)
    )

    if completed_part_paths:
        dbscan_representative_top_feature_drivers_df = (
            pd.concat(
                [pd.read_parquet(part_path) for part_path in completed_part_paths],
                ignore_index=True,
            )
            .sort_values(["feature_set", "inspection_order_within_feature_set", "driver_rank"])
            .reset_index(drop=True)
        )
    else:
        dbscan_representative_top_feature_drivers_df = pd.DataFrame()

    dbscan_top_feature_driver_validation_df = pd.DataFrame(
        [
            {
                "rows_in_driver_table": len(dbscan_representative_top_feature_drivers_df),
                "expected_rows": len(semantic_review_df) * 3,
                "feature_sets_covered": (
                    dbscan_representative_top_feature_drivers_df["feature_set"].nunique()
                    if not dbscan_representative_top_feature_drivers_df.empty
                    else 0
                ),
                "expected_feature_sets": len(MODEL_FEATURE_SET_NAMES),
                "status": (
                    "pass"
                    if len(dbscan_representative_top_feature_drivers_df) == len(semantic_review_df) * 3
                    and (
                        dbscan_representative_top_feature_drivers_df["feature_set"].nunique()
                        if not dbscan_representative_top_feature_drivers_df.empty
                        else 0
                    ) == len(MODEL_FEATURE_SET_NAMES)
                    else "review"
                ),
            }
        ]
    )

    if WRITE_332_OUTPUTS:
        dbscan_representative_top_feature_drivers_df.to_parquet(
            dbscan_top_feature_driver_path,
            index=False,
        )
        dbscan_top_feature_driver_validation_df.to_parquet(
            dbscan_top_feature_driver_validation_path_local,
            index=False,
        )

    print(
        f"Finished rebuilding top-feature-driver outputs in "
        f"{perf_counter() - rebuild_start:,.1f}s."
    )

display(dbscan_feature_driver_inventory_df)

display(
    dbscan_representative_top_feature_drivers_df[
        [
            "feature_set",
            "inspection_order_within_feature_set",
            "driver_rank",
            "date",
            "temporal_bucket",
            "policy_geography_class",
            "borough",
            "zone",
            "feature_name",
            "anomaly_value",
            "group_q1",
            "group_median",
            "group_q3",
            "within_group_zscore",
        ]
    ].round(3)
)

display(dbscan_top_feature_driver_validation_df)

assert dbscan_top_feature_driver_validation_df["status"].eq("pass").all(), (
    "One or more reviewed anomalies is missing top interpreted feature drivers."
)

Loading cached DBSCAN top-feature-driver outputs...
Loaded cached top-feature-driver outputs in 3.4s.


,feature_set,candidate_interpretation_features,feature_count
0,bus,"bus_trip_count_deviation_abs_zscore, bus_trip_count_fourier20_residual_reconstructed, bus_trip_count, bus_trip_count_residual, bus_trip_count_residual_abs, bus_trip_count_residual_zscore, bus_trip_count_seasonal, bus_trip_count_abs_change_1_same_bucket, bus_trip_count_cp_abs_delta_same_bucket, bus_trip_count_cp_pct_delta_same_bucket, bus_trip_count_lag_1_same_bucket, bus_trip_count_pct_change_1_same_bucket",12
1,fhvhv,"fhvhv_avg_trip_speed_deviation_abs_zscore, fhvhv_avg_trip_speed_deviation_abs_zscore_missing_indicator, fhvhv_trip_count_deviation_abs_zscore, fhvhv_avg_trip_speed_fourier20_residual_reconstructed, fhvhv_avg_trip_speed_fourier20_residual_reconstructed_missing_indicator, fhvhv_trip_count_fourier20_residual_reconstructed, fhvhv_avg_trip_distance, fhvhv_avg_trip_speed, fhvhv_trip_count, borough_mean_fhvhv_avg_trip_speed, cp_zone_mean_fhvhv_trip_count, fhvhv_avg_trip_speed_residual",12
2,multimodal,"borough_mean_fhvhv_avg_trip_speed, borough_mean_taxi_avg_trip_speed, borough_mean_taxi_avg_trip_speed_missing_indicator, bus_trip_count, bus_trip_count_abs_change_1_same_bucket, bus_trip_count_abs_change_1_same_bucket_missing_indicator, bus_trip_count_cp_abs_delta_same_bucket, bus_trip_count_cp_abs_delta_same_bucket_missing_indicator, bus_trip_count_cp_pct_delta_same_bucket, bus_trip_count_cp_pct_delta_same_bucket_missing_indicator, bus_trip_count_deviation_abs_zscore, bus_trip_count_deviation_abs_zscore_missing_indicator",12
3,subway,"subway_ridership_deviation_abs_zscore, subway_ridership_fourier20_residual_reconstructed, subway_ridership, borough_mean_subway_ridership, subway_ridership_residual, subway_ridership_residual_abs, subway_ridership_residual_zscore, subway_ridership_seasonal, subway_ridership_abs_change_1_same_bucket, subway_ridership_cp_abs_delta_same_bucket, subway_ridership_cp_pct_delta_same_bucket, subway_ridership_pct_change_1_same_bucket",12
4,taxi,"taxi_avg_trip_speed_deviation_abs_zscore, taxi_avg_trip_speed_deviation_abs_zscore_missing_indicator, taxi_trip_count_deviation_abs_zscore, taxi_avg_trip_speed_fourier20_residual_reconstructed, taxi_avg_trip_speed_fourier20_residual_reconstructed_missing_indicator, taxi_trip_count_fourier20_residual_reconstructed, taxi_avg_trip_distance, taxi_avg_trip_distance_missing_indicator, taxi_avg_trip_speed, taxi_avg_trip_speed_missing_indicator, taxi_trip_count, borough_mean_taxi_avg_trip_speed",12


,feature_set,inspection_order_within_feature_set,driver_rank,date,temporal_bucket,policy_geography_class,borough,zone,feature_name,anomaly_value,group_q1,group_median,group_q3,within_group_zscore
0,bus,1,1,2025-05-19,weekday_am_peak,cbd,Manhattan,Greenwich Village South,bus_trip_count_residual_zscore,-1.991,-0.654,-0.027,0.692,-1.991
1,bus,1,2,2025-05-19,weekday_am_peak,cbd,Manhattan,Greenwich Village South,bus_trip_count_pct_change_1_same_bucket,-1.317,-0.279,-0.055,0.382,-1.730
2,bus,1,3,2025-05-19,weekday_am_peak,cbd,Manhattan,Greenwich Village South,bus_trip_count_fourier20_residual_reconstructed,-0.790,-0.260,-0.000,0.252,-1.188
3,bus,2,1,2024-03-12,weekday_evening,cbd,Manhattan,SoHo,bus_trip_count_deviation_abs_zscore,1.122,-0.760,-0.088,0.680,1.124
4,bus,2,2,2024-03-12,weekday_evening,cbd,Manhattan,SoHo,bus_trip_count_pct_change_1_same_bucket,-0.701,-0.372,-0.080,0.209,-0.915
5,bus,2,3,2024-03-12,weekday_evening,cbd,Manhattan,SoHo,bus_trip_count,-0.782,-0.742,-0.602,-0.377,-0.911
6,bus,3,1,2025-04-21,weekday_midday,gateway_or_adjacent,Brooklyn,Carroll Gardens,bus_trip_count_deviation_abs_zscore,-0.823,-0.685,-0.033,0.667,-0.906
7,bus,3,2,2025-04-21,weekday_midday,gateway_or_adjacent,Brooklyn,Carroll Gardens,bus_trip_count_lag_1_same_bucket,-0.679,-0.454,-0.131,0.348,-0.871
8,bus,3,3,2025-04-21,weekday_midday,gateway_or_adjacent,Brooklyn,Carroll Gardens,bus_trip_count,-0.678,-0.454,-0.131,0.348,-0.870
9,fhvhv,1,1,2025-12-31,weekday_am_peak,cbd,Manhattan,Battery Park,fhvhv_avg_trip_speed_residual,10.037,-0.197,-0.035,0.139,9.526


,rows_in_driver_table,expected_rows,feature_sets_covered,expected_feature_sets,status
0,45,45,5,5,pass


Findings\. The feature\-driver inventory confirms that each modality is being interpreted through a focused set of 12 engineered features, and those inventories are clearly tailored to the structure of each mobility system rather than recycled across all five branches\. Bus and Subway lean heavily on deviation, residual, seasonal, and same\-bucket change features tied to volume behavior; Taxi and FHVHV mix trip\-count signals with speed, distance, and missingness indicators; and Multimodal is intentionally more cross\-system, combining borough\-level vehicle\-speed context with bus change and CP\-delta features\. So the important takeaway is not just that the semantic review is constrained, but that it is constrained in a modality\-aware way, which should make the anomaly explanations in the next block much easier to interpret and more defensible\.

Visualize the strongest interpreted feature drivers across the reviewed anomalies\. The table gives us the exact values, but this visual is meant to make the semantic pattern easier to scan: for each reviewed anomaly, we’ll show the top three interpreted feature drivers and how extreme they are relative to the anomaly’s own temporal\_bucket \| policy\_geography\_class comparison group\. The key value to watch is the within\-group z\-score: larger absolute values mean that feature is farther from the local norm and is therefore a more plausible reason the row was flagged as anomalous\.

In [52]:
# ---------------------------------------------------------------------
# Visualize the strongest interpreted feature drivers across reviewed anomalies
# ---------------------------------------------------------------------

top_feature_driver_path_local = (
    INTERMEDIATE_332_DIR / "dbscan_representative_top_feature_drivers.parquet"
)

if "dbscan_representative_top_feature_drivers_df" in globals():
    top_feature_driver_plot_df = dbscan_representative_top_feature_drivers_df.copy()
elif top_feature_driver_path_local.exists():
    top_feature_driver_plot_df = pd.read_parquet(top_feature_driver_path_local)
else:
    raise AssertionError(
        "Could not find the top interpreted feature-driver table in memory or on disk. "
        "Please run the semantic feature explanation block first."
    )

top_feature_driver_plot_df = top_feature_driver_plot_df.copy()
top_feature_driver_plot_df["review_label"] = (
    top_feature_driver_plot_df["feature_set"].str.upper()
    + " #"
    + top_feature_driver_plot_df["inspection_order_within_feature_set"].astype(str)
    + " | "
    + top_feature_driver_plot_df["zone"].astype(str)
)

top_feature_driver_plot_df["feature_driver_label"] = (
    "Rank "
    + top_feature_driver_plot_df["driver_rank"].astype(str)
    + ": "
    + top_feature_driver_plot_df["feature_name"].astype(str)
)

top_feature_driver_plot_df["driver_rank_label"] = (
    "Rank " + top_feature_driver_plot_df["driver_rank"].astype(str)
)

top_feature_driver_plot_df["abs_within_group_zscore"] = (
    top_feature_driver_plot_df["within_group_zscore"].abs()
)

top_feature_driver_plot_df = top_feature_driver_plot_df.sort_values(
    [
        "feature_set",
        "inspection_order_within_feature_set",
        "driver_rank",
    ]
).reset_index(drop=True)

feature_set_order = MODEL_FEATURE_SET_NAMES
review_order = (
    top_feature_driver_plot_df[
        ["feature_set", "inspection_order_within_feature_set", "review_label"]
    ]
    .drop_duplicates()
    .sort_values(
        ["feature_set", "inspection_order_within_feature_set"],
        key=lambda s: s.map({name: idx for idx, name in enumerate(feature_set_order)})
        if s.name == "feature_set"
        else s,
    )["review_label"]
    .tolist()
)

fig = px.scatter(
    top_feature_driver_plot_df,
    x="within_group_zscore",
    y="review_label",
    color="driver_rank_label",
    symbol="feature_set",
    color_discrete_map={
        "Rank 1": BRAND_COLORS["dark_teal"],
        "Rank 2": BRAND_COLORS["seafoam"],
        "Rank 3": BRAND_COLORS["terracotta"],
    },
    hover_data={
        "feature_set": True,
        "inspection_order_within_feature_set": True,
        "date": True,
        "temporal_bucket": True,
        "policy_geography_class": True,
        "borough": True,
        "zone": True,
        "feature_name": True,
        "anomaly_value": ":.3f",
        "group_q1": ":.3f",
        "group_median": ":.3f",
        "group_q3": ":.3f",
        "within_group_zscore": ":.3f",
        "driver_rank": True,
        "driver_rank_label": False,
        "review_label": False,
    },
    category_orders={
        "review_label": review_order,
        "driver_rank_label": ["Rank 1", "Rank 2", "Rank 3"],
    },
    labels={
        "within_group_zscore": "Within-group z-score",
        "review_label": "Reviewed anomaly",
        "driver_rank_label": "Driver rank",
    },
    title="Top Interpreted Feature Drivers Across Reviewed DBSCAN Anomalies",
)

fig.update_traces(
    marker=dict(
        size=11,
        line=dict(width=0.6, color=BRAND_COLORS["dark_teal"]),
    ),
)

fig.add_vline(
    x=0,
    line_width=1,
    line_dash="dash",
    line_color=BRAND_COLORS["seafoam"],
)

apply_brand_plotly_layout(
    fig,
    title="Top Interpreted Feature Drivers Across Reviewed DBSCAN Anomalies",
    width=960,
    height=700,
    legend_title="Driver rank",
    margin={"l": 80, "r": 30, "t": 70, "b": 110},
)

fig.update_yaxes(categoryorder="array", categoryarray=review_order[::-1])
fig.update_yaxes(ticklabelstandoff=12)
fig.show()

print("Reviewed anomaly feature-driver visualization rendered.")

Reviewed anomaly feature-driver visualization rendered.


Findings\. Read this chart row by row\. Each row is one reviewed anomaly, and the three marks are its top three interpreted feature drivers\. The dashed vertical line at 0 is the local norm for that anomaly’s own temporal\_bucket \| policy\_geography\_class comparison group\. Marks far from 0, whether to the left or the right, indicate features that are unusually low or unusually high relative to that local baseline and therefore give us a much stronger semantic explanation for why the row was flagged\. Marks close to 0 are weaker drivers, even if they still rank in the top three for that row\. So the practical way to use this visual is to look first for anomalies whose top drivers sit far from zero, because those are the rows where DBSCAN’s noise assignment is easiest to explain in interpretable feature terms\. In this panel, Taxi, FHVHV, and several Subway and Bus examples show especially strong driver separation, while some Multimodal rows appear more moderately displaced, which suggests a subtler but still structured anomaly story\.

> 📝 Note\. What these two hovered examples mean in plain language\. For Bus \#3 in Jamaica, the feature bus\_trip\_count\_lag\_1\_same\_bucket suggests that this anomaly happened when bus activity was much higher than what that zone would normally inherit from its immediately preceding same\-bucket pattern\. In regular terms, Jamaica is behaving like it is carrying forward far more bus volume or momentum than similar weekday\-evening, outside\-zone observations usually do, which makes the row look unusually intense for its local context\. For Subway \#3 in Fort Greene, the feature subway\_ridership\_residual points the other way: ridership is much lower than what the local pattern would normally predict after accounting for the usual structure in that bucket\. In regular terms, Fort Greene looks like a sharper\-than\-expected subway shortfall, not just a mildly weak day\. So one anomaly reflects an unusually strong carry\-forward bus level, while the other reflects an unusually weak subway outcome relative to local expectation\.

## 3\.3\.2\.6 Generate and Export Candidate DBSCAN Anomalies

This final section turns the calibrated DBSCAN settings into concrete anomaly outputs\. We apply the selected modality\-specific PCA configurations back to each modality’s full modeling universe, write metadata\-enriched row\-level anomaly tables, and export compact summaries that downstream notebooks can use directly for comparison, interpretation, and later cross\-framework analysis\.

### Generate and export row\-level candidate DBSCAN anomalies

Now that each modality has one final PCA\-based DBSCAN configuration, we can write the actual anomaly surfaces\. This block applies those selected settings to the full modality\-level modeling panel, preserves the row metadata needed for interpretation, and exports one row\-level candidate\-anomaly table per modality\. It also writes a compact manifest so the rest of the section can build summaries from the exported outputs rather than rerunning DBSCAN\.

To keep the export stable on large panels, it processes one local comparison group at a time, writes temporary parquet parts as it goes, and only assembles the full modality\-level row table after all groups have been labeled\. Progress and elapsed time are printed throughout so the long\-running export is easier to monitor\.

In [53]:
(FINAL_332_DIR / "dbscan_candidate_anomaly_row_level_parts").mkdir(parents=True, exist_ok=True)
(INTERMEDIATE_332_DIR / "dbscan_final_row_level_temp_parts").mkdir(parents=True, exist_ok=True)

This helper block introduces a computational safeguard for the final DBSCAN export without changing the analytical comparison framework used throughout the notebook\. The retained analytical groups still live at the shared temporal\_bucket × policy\_geography\_class level, but extremely large groups can be split temporarily into smaller compute\-time chunks so DBSCAN can run to completion without exhausting notebook memory\.

The split logic is designed to stay faithful to the mobility context rather than using arbitrary technical partitions\. Oversized groups are broken apart first by borough, then by pre/post congestion\-pricing regime if needed, and finally by calendar quarter only as a last resort\. The original analytical group label is preserved on every exported row, so downstream summaries and interpretation still treat these anomalies as belonging to the same higher\-level operating context\.

In [54]:
# ---------------------------------------------------------------------
# Throwaway fix: reset only multimodal DBSCAN partial outputs
# ---------------------------------------------------------------------

multimodal_parts_dir = INTERMEDIATE_332_DIR / "dbscan_final_row_level_temp_parts" / "multimodal"
multimodal_output_path = FINAL_332_DIR / "dbscan_candidate_anomaly_row_level_parts" / "multimodal_dbscan_candidate_anomalies.parquet"

if multimodal_parts_dir.exists():
    shutil.rmtree(multimodal_parts_dir)

if multimodal_output_path.exists():
    multimodal_output_path.unlink()

print("Reset multimodal DBSCAN partial outputs only.")

Reset multimodal DBSCAN partial outputs only.


In [55]:
# ---------------------------------------------------------------------
# Define helper logic for oversized DBSCAN compute-group splitting
# ---------------------------------------------------------------------

DBSCAN_MAX_COMPUTE_GROUP_ROWS = 40_000
DBSCAN_MAX_COMPUTE_GROUP_ROWS_MULTIMODAL = 25_000
DBSCAN_MIN_YEAR_SPLIT_GROUP_ROWS = 15_000

def _pack_contiguous_year_buckets(
    year_counts_df: pd.DataFrame,
    *,
    max_group_rows: int,
) -> pd.DataFrame:
    """
    Pack contiguous years into buckets whose total row counts do not exceed
    max_group_rows.

    Expects columns:
    - comparison_group_key
    - borough
    - pre_post_cp
    - split_year
    - group_rows
    """
    records = []

    for parent_keys, parent_df in year_counts_df.groupby(
        ["comparison_group_key", "borough", "pre_post_cp"],
        dropna=False,
        sort=False,
    ):
        comparison_group_key, borough, pre_post_cp = parent_keys
        parent_df = parent_df.sort_values("split_year").reset_index(drop=True)

        current_years = []
        current_rows = 0

        for row in parent_df.itertuples(index=False):
            split_year = int(row.split_year)
            group_rows = int(row.group_rows)

            if group_rows > max_group_rows:
                if current_years:
                    records.append(
                        {
                            "comparison_group_key": comparison_group_key,
                            "borough": borough,
                            "pre_post_cp": pre_post_cp,
                            "year_bucket_start": min(current_years),
                            "year_bucket_end": max(current_years),
                            "year_bucket_label": ",".join(str(y) for y in current_years),
                            "group_rows": current_rows,
                        }
                    )
                    current_years = []
                    current_rows = 0

                records.append(
                    {
                        "comparison_group_key": comparison_group_key,
                        "borough": borough,
                        "pre_post_cp": pre_post_cp,
                        "year_bucket_start": split_year,
                        "year_bucket_end": split_year,
                        "year_bucket_label": str(split_year),
                        "group_rows": group_rows,
                    }
                )
                continue

            if current_rows + group_rows <= max_group_rows:
                current_years.append(split_year)
                current_rows += group_rows
            else:
                records.append(
                    {
                        "comparison_group_key": comparison_group_key,
                        "borough": borough,
                        "pre_post_cp": pre_post_cp,
                        "year_bucket_start": min(current_years),
                        "year_bucket_end": max(current_years),
                        "year_bucket_label": ",".join(str(y) for y in current_years),
                        "group_rows": current_rows,
                    }
                )
                current_years = [split_year]
                current_rows = group_rows

        if current_years:
            records.append(
                {
                    "comparison_group_key": comparison_group_key,
                    "borough": borough,
                    "pre_post_cp": pre_post_cp,
                    "year_bucket_start": min(current_years),
                    "year_bucket_end": max(current_years),
                    "year_bucket_label": ",".join(str(y) for y in current_years),
                    "group_rows": current_rows,
                }
            )

    return pd.DataFrame(records)

def build_dbscan_compute_group_plan(
    prepared_panel_path: Path,
    *,
    max_group_rows: int = DBSCAN_MAX_COMPUTE_GROUP_ROWS,
) -> pd.DataFrame:
    """
    Build a compute-time grouping plan for DBSCAN.

    The analytical comparison group remains `comparison_group_key`.
    We only split very large groups into smaller computational chunks so
    sklearn DBSCAN can finish without blowing up memory.

    Split order:
    1. no split
    2. borough
    3. borough + pre_post_cp
    4. multimodal only: borough + pre_post_cp + contiguous year buckets
    """

    assert prepared_panel_path.exists(), (
        f"Prepared panel does not exist: {prepared_panel_path}"
    )

    prepared_panel_name = prepared_panel_path.name.lower()
    is_multimodal_panel = "multimodal" in prepared_panel_name

    effective_max_group_rows = (
        DBSCAN_MAX_COMPUTE_GROUP_ROWS_MULTIMODAL
        if is_multimodal_panel
        else max_group_rows
    )

    with duckdb.connect() as con:
        base_group_sizes_df = con.execute(f"""
            SELECT
                comparison_group_key,
                COUNT(*) AS group_rows
            FROM read_parquet('{sql_path(prepared_panel_path)}')
            GROUP BY comparison_group_key
            ORDER BY comparison_group_key
        """).fetchdf()

    small_groups_df = base_group_sizes_df.loc[
        base_group_sizes_df["group_rows"].le(effective_max_group_rows)
    ].copy()

    small_groups_df["dbscan_compute_group_key"] = small_groups_df["comparison_group_key"]
    small_groups_df["compute_split_level"] = "none"

    oversized_groups = (
        base_group_sizes_df.loc[
            base_group_sizes_df["group_rows"].gt(effective_max_group_rows),
            "comparison_group_key",
        ]
        .astype(str)
        .tolist()
    )

    split_plan_frames = [small_groups_df]

    for comparison_group_key in oversized_groups:
        safe_group_key = str(comparison_group_key).replace("'", "''")

        with duckdb.connect() as con:
            borough_split_df = con.execute(f"""
                WITH scoped_rows AS (
                    SELECT
                        comparison_group_key,
                        {duckdb_identifier(BOROUGH_COLUMN)} AS borough
                    FROM read_parquet('{sql_path(prepared_panel_path)}')
                    WHERE comparison_group_key = '{safe_group_key}'
                )
                SELECT
                    comparison_group_key,
                    comparison_group_key || ' || borough=' || COALESCE(borough, 'missing') AS dbscan_compute_group_key,
                    COUNT(*) AS group_rows,
                    'borough' AS compute_split_level
                FROM scoped_rows
                GROUP BY comparison_group_key, borough
                ORDER BY dbscan_compute_group_key
            """).fetchdf()

        if borough_split_df["group_rows"].max() <= effective_max_group_rows:
            split_plan_frames.append(borough_split_df)
            continue

        with duckdb.connect() as con:
            borough_prepost_split_df = con.execute(f"""
                WITH scoped_rows AS (
                    SELECT
                        comparison_group_key,
                        {duckdb_identifier(BOROUGH_COLUMN)} AS borough,
                        {duckdb_identifier(PRE_POST_CP_COLUMN)} AS pre_post_cp
                    FROM read_parquet('{sql_path(prepared_panel_path)}')
                    WHERE comparison_group_key = '{safe_group_key}'
                )
                SELECT
                    comparison_group_key,
                    comparison_group_key
                        || ' || borough=' || COALESCE(borough, 'missing')
                        || ' || prepost=' || COALESCE(pre_post_cp, 'missing') AS dbscan_compute_group_key,
                    COUNT(*) AS group_rows,
                    'borough_prepost' AS compute_split_level
                FROM scoped_rows
                GROUP BY comparison_group_key, borough, pre_post_cp
                ORDER BY dbscan_compute_group_key
            """).fetchdf()

        if borough_prepost_split_df["group_rows"].max() <= effective_max_group_rows:
            split_plan_frames.append(borough_prepost_split_df)
            continue

        if not is_multimodal_panel:
            split_plan_frames.append(borough_prepost_split_df)
            continue

        with duckdb.connect() as con:
            borough_prepost_year_counts_df = con.execute(f"""
                WITH scoped_rows AS (
                    SELECT
                        comparison_group_key,
                        {duckdb_identifier(BOROUGH_COLUMN)} AS borough,
                        {duckdb_identifier(PRE_POST_CP_COLUMN)} AS pre_post_cp,
                        YEAR({duckdb_identifier(DATE_COLUMN)}) AS split_year
                    FROM read_parquet('{sql_path(prepared_panel_path)}')
                    WHERE comparison_group_key = '{safe_group_key}'
                )
                SELECT
                    comparison_group_key,
                    COALESCE(CAST(borough AS VARCHAR), 'missing') AS borough,
                    COALESCE(CAST(pre_post_cp AS VARCHAR), 'missing') AS pre_post_cp,
                    CAST(split_year AS INTEGER) AS split_year,
                    COUNT(*) AS group_rows
                FROM scoped_rows
                GROUP BY comparison_group_key, borough, pre_post_cp, split_year
                ORDER BY comparison_group_key, borough, pre_post_cp, split_year
            """).fetchdf()

        year_bucket_df = _pack_contiguous_year_buckets(
            borough_prepost_year_counts_df,
            max_group_rows=effective_max_group_rows,
        )

        year_bucket_df["dbscan_compute_group_key"] = (
            year_bucket_df["comparison_group_key"]
            + " || borough=" + year_bucket_df["borough"]
            + " || prepost=" + year_bucket_df["pre_post_cp"]
            + " || years=" + year_bucket_df["year_bucket_label"]
        )
        year_bucket_df["compute_split_level"] = "borough_prepost_year_bucket"

        split_plan_frames.append(
            year_bucket_df[
                [
                    "comparison_group_key",
                    "dbscan_compute_group_key",
                    "group_rows",
                    "compute_split_level",
                ]
            ]
        )

    compute_group_plan_df = (
        pd.concat(split_plan_frames, ignore_index=True)
        .sort_values(["comparison_group_key", "dbscan_compute_group_key"])
        .reset_index(drop=True)
    )

    return compute_group_plan_df

This block turns the retained DBSCAN settings into full\-universe row\-level anomaly outputs\. For each modality, it rebuilds the final PCA\-aligned modeling panel if needed, generates a compute\-group plan, runs DBSCAN one compute group at a time, and writes each completed chunk immediately to disk before stitching the finished parts into one final modality\-level export\.

The workflow is intentionally designed to be restart\-friendly\. Completed compute groups are preserved across interrupted runs, completed modalities are reused when their final exports already exist, and only missing pieces are recomputed\. That makes the final export much safer to run in a notebook environment while still producing the full metadata\-enriched anomaly surfaces needed by downstream notebooks\.

In [56]:
# ---------------------------------------------------------------------
# Generate and export row-level candidate DBSCAN anomalies
# Rerunnable at the per-compute-group level
# ---------------------------------------------------------------------

assert "dbscan_final_pca_settings_df" in globals(), (
    "dbscan_final_pca_settings_df is not in memory. "
    "Please run the final setting-selection block first."
)

assert "build_dbscan_compute_group_plan" in globals(), (
    "build_dbscan_compute_group_plan is not in memory. "
    "Please run the oversized-group helper block first."
)

dbscan_final_row_level_dir = FINAL_332_DIR / "dbscan_candidate_anomaly_row_level_parts"
dbscan_final_row_level_dir.mkdir(parents=True, exist_ok=True)

dbscan_final_temp_parts_dir = INTERMEDIATE_332_DIR / "dbscan_final_row_level_temp_parts"
dbscan_final_temp_parts_dir.mkdir(parents=True, exist_ok=True)

dbscan_final_prepared_panel_dir = INTERMEDIATE_332_DIR / "dbscan_final_prepared_panels"
dbscan_final_prepared_panel_dir.mkdir(parents=True, exist_ok=True)

group_metadata_columns = [
    MODEL_ID_COLUMN,
    DATE_COLUMN,
    TEMPORAL_BUCKET_COLUMN,
    PRE_POST_CP_COLUMN,
    ZONE_ID_COLUMN,
    ZONE_COLUMN,
    BOROUGH_COLUMN,
    POLICY_GEOGRAPHY_COLUMN,
    "comparison_group_key",
]

dbscan_export_settings_df = (
    dbscan_final_pca_settings_df.copy()
    .sort_values(["feature_set"])
    .reset_index(drop=True)
)

final_row_level_manifest_records = []
overall_start = perf_counter()

for export_index, export_row in enumerate(
    dbscan_export_settings_df.itertuples(index=False),
    start=1,
):
    feature_start = perf_counter()

    feature_set_name = export_row.feature_set
    output_path = dbscan_final_row_level_dir / f"{feature_set_name}_dbscan_candidate_anomalies.parquet"
    feature_parts_dir = dbscan_final_temp_parts_dir / feature_set_name
    prepared_panel_path = dbscan_final_prepared_panel_dir / f"{feature_set_name}_dbscan_prepared_panel.parquet"
    feature_state_path = feature_parts_dir / "_run_state.json"

    if REBUILD_DBSCAN_FINAL_OUTPUTS:
        if feature_parts_dir.exists():
            shutil.rmtree(feature_parts_dir)
        if output_path.exists():
            output_path.unlink()

    feature_parts_dir.mkdir(parents=True, exist_ok=True)

    expected_state = {
        "feature_set": feature_set_name,
        "representation_name": export_row.representation_name,
        "min_samples": int(export_row.min_samples),
        "eps_band": export_row.eps_band,
        "eps_value": float(export_row.eps_value),
        "max_compute_group_rows": int(DBSCAN_MAX_COMPUTE_GROUP_ROWS),
        "compute_split_strategy": "none_or_borough_or_borough_prepost",
    }

    if feature_state_path.exists():
        with open(feature_state_path, "r", encoding="utf-8") as f:
            existing_state = json.load(f)

        if existing_state != expected_state:
            print(
                f"[{export_index}/{len(dbscan_export_settings_df)}] "
                f"State mismatch for {feature_set_name}; clearing stale partial parts."
            )
            shutil.rmtree(feature_parts_dir)
            feature_parts_dir.mkdir(parents=True, exist_ok=True)

    with open(feature_state_path, "w", encoding="utf-8") as f:
        json.dump(expected_state, f, indent=2)

    if output_path.exists() and not REBUILD_DBSCAN_FINAL_OUTPUTS:
        with duckdb.connect() as con:
            existing_summary_df = con.execute(f"""
                SELECT
                    COUNT(*) AS rows_written,
                    SUM(CASE WHEN is_anomaly_like THEN 1 ELSE 0 END) AS anomaly_like_rows,
                    COUNT(DISTINCT comparison_group_key) AS groups_present
                FROM read_parquet('{sql_path(output_path)}')
            """).fetchdf()

        final_row_level_manifest_records.append(
            {
                "feature_set": feature_set_name,
                "representation_name": export_row.representation_name,
                "min_samples": int(export_row.min_samples),
                "eps_band": export_row.eps_band,
                "eps_value": float(export_row.eps_value),
                "rows_written": int(existing_summary_df.loc[0, "rows_written"]),
                "anomaly_like_rows": int(existing_summary_df.loc[0, "anomaly_like_rows"] or 0),
                "groups_present": int(existing_summary_df.loc[0, "groups_present"]),
                "status": "loaded_existing",
            }
        )

        print(
            f"[{export_index}/{len(dbscan_export_settings_df)}] "
            f"Loaded existing candidate anomalies for {feature_set_name} "
            f"in {perf_counter() - feature_start:,.1f}s."
        )
        continue

    print(
        f"[{export_index}/{len(dbscan_export_settings_df)}] "
        f"Generating candidate anomalies for {feature_set_name}..."
    )

    prep_start = perf_counter()
    metadata_path = ROW_METADATA_PATHS_BY_SET[feature_set_name]

    if feature_set_name == "taxi" and TAXI_PCA_HANDLING_DECISION == "split_prepost_period":
        pre_pca_path = get_representation_path(
            feature_set_name=feature_set_name,
            representation_name="pca_80pct",
            pre_post_cp="pre_cp",
        )
        post_pca_path = get_representation_path(
            feature_set_name=feature_set_name,
            representation_name="pca_80pct",
            pre_post_cp="post_cp",
        )
        representation_columns = get_numeric_representation_columns(pre_pca_path)

        pca_sql = f"""
            SELECT {", ".join(duckdb_identifier(col) for col in [MODEL_ID_COLUMN, *representation_columns])}
            FROM read_parquet('{sql_path(pre_pca_path)}')
            UNION ALL
            SELECT {", ".join(duckdb_identifier(col) for col in [MODEL_ID_COLUMN, *representation_columns])}
            FROM read_parquet('{sql_path(post_pca_path)}')
        """
    else:
        pca_path = get_representation_path(
            feature_set_name=feature_set_name,
            representation_name="pca_80pct",
        )
        representation_columns = get_numeric_representation_columns(pca_path)

        pca_sql = f"""
            SELECT {", ".join(duckdb_identifier(col) for col in [MODEL_ID_COLUMN, *representation_columns])}
            FROM read_parquet('{sql_path(pca_path)}')
        """

    if not prepared_panel_path.exists():
        print(f"  Building prepared panel for {feature_set_name}...")
        with duckdb.connect() as con:
            con.execute(f"""
                COPY (
                    WITH metadata_rows AS (
                        SELECT
                            {", ".join(duckdb_identifier(col) for col in DEFAULT_METADATA_COLUMNS)},
                            CAST({duckdb_identifier(TEMPORAL_BUCKET_COLUMN)} AS VARCHAR)
                                || ' | ' ||
                            CAST({duckdb_identifier(POLICY_GEOGRAPHY_COLUMN)} AS VARCHAR)
                                AS comparison_group_key
                        FROM read_parquet('{sql_path(metadata_path)}')
                    ),
                    pca_rows AS (
                        {pca_sql}
                    )
                    SELECT
                        m.{duckdb_identifier(MODEL_ID_COLUMN)},
                        m.{duckdb_identifier(DATE_COLUMN)},
                        m.{duckdb_identifier(TEMPORAL_BUCKET_COLUMN)},
                        m.{duckdb_identifier(PRE_POST_CP_COLUMN)},
                        m.{duckdb_identifier(ZONE_ID_COLUMN)},
                        m.{duckdb_identifier(ZONE_COLUMN)},
                        m.{duckdb_identifier(BOROUGH_COLUMN)},
                        m.{duckdb_identifier(POLICY_GEOGRAPHY_COLUMN)},
                        m.comparison_group_key,
                        {", ".join(
                            f"CAST(p.{duckdb_identifier(col)} AS FLOAT) AS {duckdb_identifier(col)}"
                            for col in representation_columns
                        )}
                    FROM metadata_rows AS m
                    INNER JOIN pca_rows AS p
                        ON m.{duckdb_identifier(MODEL_ID_COLUMN)} = p.{duckdb_identifier(MODEL_ID_COLUMN)}
                )
                TO '{sql_path(prepared_panel_path)}' (FORMAT PARQUET)
            """)

    assert prepared_panel_path.exists(), (
        f"Prepared panel was not created for {feature_set_name}: {prepared_panel_path}"
    )

    with duckdb.connect() as con:
        prep_summary_df = con.execute(f"""
            SELECT
                COUNT(*) AS row_count,
                COUNT(DISTINCT comparison_group_key) AS group_count
            FROM read_parquet('{sql_path(prepared_panel_path)}')
        """).fetchdf()

    row_count = int(prep_summary_df.loc[0, "row_count"])
    group_count = int(prep_summary_df.loc[0, "group_count"])

    print(
        f"  Prepared {feature_set_name}: {row_count:,} rows across {group_count:,} groups "
        f"in {perf_counter() - prep_start:,.1f}s."
    )

    compute_group_plan_df = build_dbscan_compute_group_plan(
        prepared_panel_path,
        max_group_rows=DBSCAN_MAX_COMPUTE_GROUP_ROWS,
    )

    compute_group_count = len(compute_group_plan_df)

    print(
        f"  Compute plan for {feature_set_name}: {compute_group_count:,} compute groups "
        f"from {group_count:,} analytical groups."
    )

    for group_index, group_row in enumerate(
        compute_group_plan_df.itertuples(index=False),
        start=1,
    ):
        group_start = perf_counter()

        comparison_group_key = group_row.comparison_group_key
        compute_group_key = group_row.dbscan_compute_group_key
        compute_split_level = group_row.compute_split_level
        group_row_count = int(group_row.group_rows)
        part_path = feature_parts_dir / f"group_{group_index:04d}.parquet"

        if part_path.exists() and not REBUILD_DBSCAN_FINAL_OUTPUTS:
            print(
                f"    Compute group {group_index}/{compute_group_count} | "
                f"analytical={comparison_group_key} | "
                f"split={compute_split_level} | "
                f"value={str(compute_group_key).replace(str(comparison_group_key) + ' || ', '')} | "
                f"rows={group_row_count:,} | skipped_existing"
            )
            continue

        print(
            f"    Compute group {group_index}/{compute_group_count} | "
            f"analytical={comparison_group_key} | "
            f"split={compute_split_level} | "
            f"value={str(compute_group_key).replace(str(comparison_group_key) + ' || ', '')} | "
            f"rows={group_row_count:,}"
        )

        safe_group_key = str(comparison_group_key).replace("'", "''")

        if compute_split_level == "none":
            where_sql = f"comparison_group_key = '{safe_group_key}'"

        elif compute_split_level == "borough":
            borough_value = compute_group_key.split(" || borough=", 1)[1]
            safe_borough = borough_value.replace("'", "''")
            where_sql = (
                f"comparison_group_key = '{safe_group_key}' "
                f"AND COALESCE(CAST({duckdb_identifier(BOROUGH_COLUMN)} AS VARCHAR), 'missing') = '{safe_borough}'"
            )

        elif compute_split_level == "borough_prepost":
            borough_part = compute_group_key.split(" || borough=", 1)[1]
            borough_value, prepost_value = borough_part.split(" || prepost=", 1)
            safe_borough = borough_value.replace("'", "''")
            safe_prepost = prepost_value.replace("'", "''")
            where_sql = (
                f"comparison_group_key = '{safe_group_key}' "
                f"AND COALESCE(CAST({duckdb_identifier(BOROUGH_COLUMN)} AS VARCHAR), 'missing') = '{safe_borough}' "
                f"AND COALESCE(CAST({duckdb_identifier(PRE_POST_CP_COLUMN)} AS VARCHAR), 'missing') = '{safe_prepost}'"
            )

        elif compute_split_level == "borough_prepost_year_bucket":
            borough_part = compute_group_key.split(" || borough=", 1)[1]
            borough_value, remainder = borough_part.split(" || prepost=", 1)
            prepost_value, years_value = remainder.split(" || years=", 1)

            safe_borough = borough_value.replace("'", "''")
            safe_prepost = prepost_value.replace("'", "''")
            year_values = [int(year_str) for year_str in years_value.split(",")]

            where_sql = (
                f"comparison_group_key = '{safe_group_key}' "
                f"AND COALESCE(CAST({duckdb_identifier(BOROUGH_COLUMN)} AS VARCHAR), 'missing') = '{safe_borough}' "
                f"AND COALESCE(CAST({duckdb_identifier(PRE_POST_CP_COLUMN)} AS VARCHAR), 'missing') = '{safe_prepost}' "
                f"AND EXTRACT(YEAR FROM {duckdb_identifier(DATE_COLUMN)}) IN ({', '.join(str(y) for y in year_values)})"
            )

        else:
            raise ValueError(f"Unsupported compute_split_level: {compute_split_level}")

        with duckdb.connect() as con:
            metadata_df = con.execute(f"""
                SELECT {", ".join(duckdb_identifier(col) for col in group_metadata_columns)}
                FROM read_parquet('{sql_path(prepared_panel_path)}')
                WHERE {where_sql}
            """).fetchdf()

            X_df = con.execute(f"""
                SELECT {", ".join(duckdb_identifier(col) for col in representation_columns)}
                FROM read_parquet('{sql_path(prepared_panel_path)}')
                WHERE {where_sql}
            """).fetchdf()

        X = np.asarray(X_df, dtype=np.float32, order="C")
        del X_df
        gc.collect()

        dbscan_model = DBSCAN(
            eps=float(export_row.eps_value),
            min_samples=int(export_row.min_samples),
            metric="euclidean",
        )
        cluster_labels = dbscan_model.fit_predict(X)

        metadata_df["feature_set"] = feature_set_name
        metadata_df["representation_name"] = export_row.representation_name
        metadata_df["min_samples"] = int(export_row.min_samples)
        metadata_df["eps_band"] = export_row.eps_band
        metadata_df["eps_value"] = float(export_row.eps_value)
        metadata_df["cluster_label"] = cluster_labels.astype(np.int32, copy=False)
        metadata_df["is_anomaly_like"] = metadata_df["cluster_label"].eq(-1)

        metadata_df.to_parquet(part_path, index=False)

        print(
            f"      done | anomaly_like_rows={int(metadata_df['is_anomaly_like'].sum()):,} | "
            f"group_time={perf_counter() - group_start:,.1f}s | "
            f"total_elapsed={perf_counter() - feature_start:,.1f}s"
        )

        del metadata_df
        del X
        del dbscan_model
        del cluster_labels
        gc.collect()

    expected_part_paths = [
        feature_parts_dir / f"group_{group_index:04d}.parquet"
        for group_index in range(1, compute_group_count + 1)
    ]
    missing_parts = [path for path in expected_part_paths if not path.exists()]
    assert not missing_parts, (
        f"Cannot stitch {feature_set_name}; missing {len(missing_parts)} compute-group part(s). "
        f"First missing: {missing_parts[0]}"
    )

    stitch_start = perf_counter()

    with duckdb.connect() as con:
        con.execute(f"""
            COPY (
                SELECT *
                FROM read_parquet('{sql_path(feature_parts_dir)}/*.parquet')
            )
            TO '{sql_path(output_path)}' (FORMAT PARQUET)
        """)

        feature_summary_df = con.execute(f"""
            SELECT
                COUNT(*) AS rows_written,
                SUM(CASE WHEN is_anomaly_like THEN 1 ELSE 0 END) AS anomaly_like_rows,
                COUNT(DISTINCT comparison_group_key) AS groups_present
            FROM read_parquet('{sql_path(output_path)}')
        """).fetchdf()

    rows_written = int(feature_summary_df.loc[0, "rows_written"])
    anomaly_like_rows = int(feature_summary_df.loc[0, "anomaly_like_rows"] or 0)
    groups_present = int(feature_summary_df.loc[0, "groups_present"])

    final_row_level_manifest_records.append(
        {
            "feature_set": feature_set_name,
            "representation_name": export_row.representation_name,
            "min_samples": int(export_row.min_samples),
            "eps_band": export_row.eps_band,
            "eps_value": float(export_row.eps_value),
            "rows_written": rows_written,
            "anomaly_like_rows": anomaly_like_rows,
            "groups_present": groups_present,
            "status": "written",
        }
    )

    print(
        f"  Finished {feature_set_name}: wrote {rows_written:,} rows "
        f"with {anomaly_like_rows:,} anomaly-like rows | "
        f"stitch_time={perf_counter() - stitch_start:,.1f}s | "
        f"feature_total={perf_counter() - feature_start:,.1f}s"
    )

    del prep_summary_df
    del compute_group_plan_df
    del feature_summary_df
    gc.collect()

dbscan_final_row_level_manifest_df = (
    pd.DataFrame(final_row_level_manifest_records)
    .sort_values(["feature_set"])
    .reset_index(drop=True)
)

dbscan_final_row_level_manifest_df.to_csv(
    DBSCAN_ANOMALY_EXPORT_MANIFEST_PATH,
    index=False,
)

print(f"Completed DBSCAN final row-level export in {perf_counter() - overall_start:,.1f}s.")

display(
    dbscan_final_row_level_manifest_df[
        [
            "feature_set",
            "representation_name",
            "min_samples",
            "eps_band",
            "eps_value",
            "rows_written",
            "anomaly_like_rows",
            "groups_present",
            "status",
        ]
    ].round({"eps_value": 3})
)

[1/5] Loaded existing candidate anomalies for bus in 1.8s.
[2/5] Loaded existing candidate anomalies for fhvhv in 1.8s.
[3/5] Generating candidate anomalies for multimodal...
  Prepared multimodal: 1,541,800 rows across 30 groups in 3.6s.
  Compute plan for multimodal: 110 compute groups from 30 analytical groups.
    Compute group 1/110 | analytical=weekday_am_peak | cbd | split=borough_prepost | value=borough=Manhattan || prepost=post_cp | rows=12,880
      done | anomaly_like_rows=63 | group_time=3.5s | total_elapsed=13.1s
    Compute group 2/110 | analytical=weekday_am_peak | cbd | split=borough_prepost | value=borough=Manhattan || prepost=pre_cp | rows=21,000
      done | anomaly_like_rows=59 | group_time=5.7s | total_elapsed=18.9s
    Compute group 3/110 | analytical=weekday_am_peak | gateway_or_adjacent | split=none | value=weekday_am_peak | gateway_or_adjacent | rows=16,093
      done | anomaly_like_rows=11 | group_time=4.2s | total_elapsed=23.3s
    Compute group 4/110 | analy

,feature_set,representation_name,min_samples,eps_band,eps_value,rows_written,anomaly_like_rows,groups_present,status
0,bus,pca_80pct,15,balanced,5.281,1472947,3074,30,loaded_existing
1,fhvhv,pca_80pct,15,balanced,5.116,1541800,6086,30,loaded_existing
2,multimodal,pca_80pct,15,balanced,13.558,1541800,4065,30,written
3,subway,pca_80pct,15,balanced,2.745,911455,5853,30,loaded_existing
4,taxi,pca_80pct,20,tight,4.812,1541800,7027,30,loaded_existing


Findings\. The full\-universe DBSCAN export completed successfully for all five modality branches, with one retained PCA configuration per branch and complete coverage of all 30 analytical comparison groups in every case\. In raw anomaly volume, Taxi is the most expansive DBSCAN surface with 7,027 anomaly\-like rows, followed by FHVHV with 6,086, Subway with 5,853, Multimodal with 4,065, and Bus with 3,074\. So even after moving from calibration to the full modeling universe, DBSCAN remains a relatively selective detector: each branch flags only a small fraction of its total rows rather than flooding the panel with noise\.

The branch differences are also substantively interesting\. Taxi and FHVHV continue to look like the richest anomaly environments under DBSCAN, which fits the idea that road\-based for\-hire mobility systems are especially sensitive to local disruptions and regime changes\. Subway remains fairly anomaly\-dense relative to its smaller row universe, suggesting that ridership patterns still generate a strong unusualness signal even without the same row count as the road\-based branches\. Multimodal is more selective than Taxi, FHVHV, or Subway despite having the full 1\.54 million\-row universe available, which is a useful sign that the fused feature space is not simply firing everywhere\. Bus is the sparsest branch, reinforcing the earlier impression that bus anomalies are real but comparatively narrower and harder to activate under the retained DBSCAN settings\.

## 3\.3\.2\.7 Review aggregate anomaly behavior

This part shifts from anomaly generation to anomaly validation at aggregate scale\. With the full DBSCAN candidate surfaces now exported for all five modality branches, the goal is to check whether those surfaces behave sensibly across temporal buckets, policy geographies, geometric separation diagnostics, and spatial concentration patterns before we carry them forward into final comparison work\.

In other words, this is the first full\-surface quality check on the exported DBSCAN outputs\. Rather than focusing on a few representative rows, we are asking whether the anomaly surfaces as a whole look coherent, selective, and geographically interpretable enough to trust as retained modality\-level candidates\.

Summarize final anomaly prevalence across temporal buckets and policy geographies\. Let’s start with the simplest aggregate check\. This block shows where the final anomaly outputs are appearing across the retained temporal\_bucket \| policy\_geography\_class comparison universes, so we can see whether DBSCAN is reasonably distributed across operating regimes or is collapsing too heavily into one slice of the city’s mobility\-state surface\.

In [57]:
# ---------------------------------------------------------------------
# Summarize final anomaly prevalence across temporal buckets and policy geographies
# ---------------------------------------------------------------------

assert "dbscan_final_pca_settings_df" in globals(), (
    "dbscan_final_pca_settings_df is not in memory. "
    "Please run the final DBSCAN setting-selection block first."
)

final_row_level_dir = FINAL_332_DIR / "dbscan_candidate_anomaly_row_level_parts"
final_row_level_dir.mkdir(parents=True, exist_ok=True)

# Use the selected settings table as the upstream source of which modalities
# should exist. Then summarize whichever row-level files are already available.
expected_feature_sets = (
    dbscan_final_pca_settings_df["feature_set"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

available_feature_sets = []
missing_feature_sets = []

for feature_set_name in expected_feature_sets:
    row_level_path = final_row_level_dir / f"{feature_set_name}_dbscan_candidate_anomalies.parquet"
    if row_level_path.exists():
        available_feature_sets.append(feature_set_name)
    else:
        missing_feature_sets.append(feature_set_name)

assert available_feature_sets, (
    "No DBSCAN row-level anomaly parquet files are available yet. "
    "Please run the candidate-anomaly export block first."
)

if missing_feature_sets:
    print(
        "Warning: some DBSCAN row-level outputs are not available yet and will be "
        f"excluded from this summary: {missing_feature_sets}"
    )

aggregate_prevalence_records = []

for feature_set_name in available_feature_sets:
    row_level_path = final_row_level_dir / f"{feature_set_name}_dbscan_candidate_anomalies.parquet"
    row_level_df = pd.read_parquet(row_level_path)

    prevalence_df = (
        row_level_df.groupby(
            [TEMPORAL_BUCKET_COLUMN, POLICY_GEOGRAPHY_COLUMN],
            dropna=False,
            observed=False,
        )
        .agg(
            total_rows=("is_anomaly_like", "size"),
            anomaly_rows=("is_anomaly_like", "sum"),
        )
        .reset_index()
    )

    prevalence_df["feature_set"] = feature_set_name
    prevalence_df["anomaly_share"] = (
        prevalence_df["anomaly_rows"] / prevalence_df["total_rows"]
    )

    aggregate_prevalence_records.append(prevalence_df)

dbscan_aggregate_prevalence_df = (
    pd.concat(aggregate_prevalence_records, ignore_index=True)
    .sort_values(["feature_set", TEMPORAL_BUCKET_COLUMN, POLICY_GEOGRAPHY_COLUMN])
    .reset_index(drop=True)
)

dbscan_aggregate_prevalence_summary_df = (
    dbscan_aggregate_prevalence_df.groupby("feature_set", dropna=False, observed=False)
    .agg(
        comparison_groups=("anomaly_share", "size"),
        min_anomaly_share=("anomaly_share", "min"),
        median_anomaly_share=("anomaly_share", "median"),
        max_anomaly_share=("anomaly_share", "max"),
    )
    .reset_index()
)

display(dbscan_aggregate_prevalence_summary_df.round(3))
display(dbscan_aggregate_prevalence_df.round(3))

,feature_set,comparison_groups,min_anomaly_share,median_anomaly_share,max_anomaly_share
0,bus,30,0.000,0.002,0.008
1,fhvhv,30,0.000,0.003,0.011
2,multimodal,30,0.001,0.003,0.015
3,subway,30,0.001,0.003,0.061
4,taxi,30,0.002,0.007,0.023


,temporal_bucket,policy_geography_class,total_rows,anomaly_rows,feature_set,anomaly_share
0,weekday_am_peak,cbd,32148,16,bus,0.000
1,weekday_am_peak,gateway_or_adjacent,15228,11,bus,0.001
2,weekday_am_peak,outside,164970,392,bus,0.002
3,weekday_evening,cbd,31302,3,bus,0.000
4,weekday_evening,gateway_or_adjacent,14382,0,bus,0.000
...,...,...,...,...,...,...
145,weekend_overnight,gateway_or_adjacent,6441,32,taxi,0.005
146,weekend_overnight,outside,68139,312,taxi,0.005
147,weekend_pm_peak,cbd,13560,99,taxi,0.007
148,weekend_pm_peak,gateway_or_adjacent,6441,116,taxi,0.018


Findings\. Across the full 30\-slice comparison surface for each modality, DBSCAN remains selective but clearly not uniform\. Taxi has the highest typical anomaly prevalence, with a median anomaly share of 0\.7% and a maximum of 2\.3%, which makes it the broadest high\-frequency anomaly surface among the five branches\. Multimodal and FHVHV sit in the middle with median shares around 0\.3%, while Bus is the sparsest branch with a median of 0\.2% and a maximum of 0\.8%\. Subway is the most uneven branch: its median anomaly share is only 0\.3%, but its maximum jumps all the way to 6\.1%, which means its anomaly behavior is far more concentrated into a small number of especially intense operating contexts rather than spread evenly across slices\.

The detailed 150\-row table makes that pattern concrete\. For Subway, the standout slice is weekday\_pm\_peak \| cbd, where 1,461 of 24,006 rows are flagged, yielding a 6\.1% anomaly share\. Subway also shows elevated anomaly concentrations in weekend\_midday \| cbd and weekend\_pm\_peak \| cbd at 3\.0%, and in weekday\_pm\_peak \| gateway\_or\_adjacent at 2\.2%, while most outside slices remain close to 0\.1% to 0\.4%\. Taxi is less extreme but more consistently elevated across multiple high\-intensity contexts, especially weekend\_overnight \| cbd at 2\.3%, weekend\_midday \| gateway\_or\_adjacent at 2\.3%, weekend\_pm\_peak \| gateway\_or\_adjacent at 1\.8%, and weekend\_evening \| cbd at 1\.6%\. So the practical takeaway is that DBSCAN is surfacing a meaningful concentration structure: Subway’s anomalies are sharply concentrated into a few central commuter slices, while Taxi’s anomalies are more broadly distributed across core urban and gateway contexts\.

In [58]:
# ---------------------------------------------------------------------
# Visualize final anomaly prevalence across temporal buckets and policy geographies
# ---------------------------------------------------------------------

heatmap_df = dbscan_aggregate_prevalence_df.copy()

temporal_bucket_order = [
    "weekday_am_peak",
    "weekday_midday",
    "weekday_evening",
    "weekday_pm_peak",
    "weekday_overnight",
    "weekend_am_peak",
    "weekend_midday",
    "weekend_evening",
    "weekend_pm_peak",
    "weekend_overnight",
]

policy_geo_order = ["cbd", "gateway_or_adjacent", "outside"]
feature_set_order = ["bus", "subway", "taxi", "fhvhv", "multimodal"]

heatmap_df[TEMPORAL_BUCKET_COLUMN] = pd.Categorical(
    heatmap_df[TEMPORAL_BUCKET_COLUMN],
    categories=temporal_bucket_order,
    ordered=True,
)
heatmap_df[POLICY_GEOGRAPHY_COLUMN] = pd.Categorical(
    heatmap_df[POLICY_GEOGRAPHY_COLUMN],
    categories=policy_geo_order,
    ordered=True,
)
heatmap_df["feature_set"] = pd.Categorical(
    heatmap_df["feature_set"],
    categories=feature_set_order,
    ordered=True,
)

brand_prevalence_scale = [
    [0.00, BRAND_COLORS["ice"]],
    [0.35, BRAND_COLORS["pale_peach"]],
    [0.70, BRAND_COLORS["terracotta"]],
    [1.00, BRAND_COLORS["dark_teal"]],
]

fig = px.density_heatmap(
    heatmap_df,
    x=POLICY_GEOGRAPHY_COLUMN,
    y=TEMPORAL_BUCKET_COLUMN,
    z="anomaly_share",
    facet_col="feature_set",
    facet_col_wrap=3,
    histfunc="avg",
    text_auto=".3f",
    color_continuous_scale=brand_prevalence_scale,
    category_orders={
        POLICY_GEOGRAPHY_COLUMN: policy_geo_order,
        TEMPORAL_BUCKET_COLUMN: temporal_bucket_order,
        "feature_set": feature_set_order,
    },
    labels={
        POLICY_GEOGRAPHY_COLUMN: "Policy geography",
        TEMPORAL_BUCKET_COLUMN: "Temporal bucket",
        "anomaly_share": "Anomaly share",
        "feature_set": "Feature set",
    },
    title="Final DBSCAN Anomaly Share Across Temporal Buckets and Policy Geographies",
)

fig.update_layout(
    template="plotly_white",
    paper_bgcolor="white",
    plot_bgcolor=BRAND_COLORS["ice"],
    font=dict(color=BRAND_COLORS["dark_teal"]),
    title=dict(
        text="Final DBSCAN Anomaly Share Across Temporal Buckets and Policy Geographies",
        font=dict(color=BRAND_COLORS["dark_teal"]),
    ),
    width=960,
    height=660,
    margin=dict(l=80, r=30, t=80, b=60),
    coloraxis_colorbar=dict(
        title=dict(
            text="Anomaly share",
            font=dict(color=BRAND_COLORS["dark_teal"]),
        ),
        tickfont=dict(color=BRAND_COLORS["dark_teal"]),
    ),
)

fig.for_each_annotation(
    lambda a: a.update(
        text=a.text.split("=")[-1].upper(),
        font=dict(color=BRAND_COLORS["dark_teal"]),
    )
)
fig.update_xaxes(
    matches=None,
    gridcolor="rgba(11, 78, 74, 0.14)",
    tickfont=dict(color=BRAND_COLORS["dark_teal"]),
    title_font=dict(color=BRAND_COLORS["dark_teal"]),
)
fig.update_yaxes(
    matches=None,
    ticklabelstandoff=4,
    gridcolor="rgba(11, 78, 74, 0.14)",
    tickfont=dict(color=BRAND_COLORS["dark_teal"]),
    title_font=dict(color=BRAND_COLORS["dark_teal"]),
)
fig.show()

print("Aggregate anomaly-prevalence heatmap rendered.")

Aggregate anomaly-prevalence heatmap rendered.


Findings\. The heatmap adds a cross\-modality view of anomaly concentration, and the main story is contrast in how sharply each branch concentrates its unusualness\. Subway stands out as the most peaked surface by a wide margin, with one dominant weekday\_pm\_peak \| cbd hotspot and a handful of smaller but still elevated CBD and gateway slices around it\. Taxi is less extreme but much more broadly elevated across multiple CBD and gateway time buckets, which makes its anomaly surface look more distributed rather than singularly spiked\. FHVHV and Multimodal sit in the middle: both show recognizable hotspots, but neither approaches Subway’s intensity nor Taxi’s breadth\. Bus is the flattest branch, with only mild elevation across the map\. So the visual takeaway is that DBSCAN is separating the modalities not just by anomaly volume, but by anomaly shape: Subway is sharply concentrated, Taxi is broadly urban\-core facing, FHVHV and Multimodal are moderate, and Bus is comparatively diffuse and quiet\.

Validate aggregate geometric separation of anomalies versus non\-anomalies\. The representative review showed that sampled anomalies often sit meaningfully outside nearby cluster structure\. This block tests whether that pattern holds across the full anomaly surfaces by comparing anomaly and non\-anomaly rows on their distance to the nearest retained cluster centroid within the local temporal\_bucket \| policy\_geography\_class group\.

In [59]:
# ---------------------------------------------------------------------
# Validate aggregate geometric separation of anomalies versus non-anomalies
# ---------------------------------------------------------------------

final_row_level_dir = FINAL_332_DIR / "dbscan_candidate_anomaly_row_level_parts"

if "dbscan_final_row_level_manifest_df" in globals():
    geometric_manifest_df = dbscan_final_row_level_manifest_df.copy()
elif DBSCAN_ANOMALY_EXPORT_MANIFEST_PATH.exists():
    geometric_manifest_df = pd.read_csv(DBSCAN_ANOMALY_EXPORT_MANIFEST_PATH)
else:
    raise AssertionError(
        "Could not find the DBSCAN export manifest in memory or on disk. "
        "Please run the candidate-anomaly export block first."
    )

aggregate_geometric_records = []

for manifest_row in geometric_manifest_df.itertuples(index=False):
    feature_set_name = manifest_row.feature_set
    row_level_path = final_row_level_dir / f"{feature_set_name}_dbscan_candidate_anomalies.parquet"

    row_level_df = pd.read_parquet(row_level_path)
    metadata_df = load_calibration_metadata(feature_set_name)

    if feature_set_name == "taxi" and TAXI_PCA_HANDLING_DECISION == "split_prepost_period":
        representation_df, representation_columns = get_final_pca_representation_df(
            feature_set_name,
            metadata_df,
        )
    else:
        representation_df = load_representation_rows(
            feature_set_name=feature_set_name,
            representation_name="pca_80pct",
        )
        representation_columns = [
            column_name
            for column_name in representation_df.columns
            if column_name != MODEL_ID_COLUMN
        ]

    modeling_df = row_level_df.merge(
        representation_df,
        on=MODEL_ID_COLUMN,
        how="inner",
    ).copy()

    group_distance_records = []

    # Compute distances within each local comparison universe.
    for comparison_group_key, group_df in modeling_df.groupby(
        "comparison_group_key",
        dropna=False,
        sort=True,
    ):
        non_noise_df = group_df.loc[group_df["cluster_label"].ne(-1)].copy()

        # If there are no retained clusters, there is no meaningful centroid-distance check.
        if non_noise_df.empty:
            continue

        centroid_by_cluster = (
            non_noise_df.groupby("cluster_label", dropna=False)[representation_columns]
            .mean()
        )

        centroid_matrix = centroid_by_cluster.to_numpy(dtype=np.float32)
        centroid_labels = centroid_by_cluster.index.to_numpy()

        X = group_df[representation_columns].to_numpy(dtype=np.float32)
        distance_matrix = np.sqrt(((X[:, None, :] - centroid_matrix[None, :, :]) ** 2).sum(axis=2))

        nearest_centroid_idx = distance_matrix.argmin(axis=1)
        nearest_centroid_distance = distance_matrix[np.arange(len(group_df)), nearest_centroid_idx]
        nearest_cluster_label = centroid_labels[nearest_centroid_idx]

        group_result_df = group_df[
            [
                MODEL_ID_COLUMN,
                "feature_set",
                "comparison_group_key",
                "is_anomaly_like",
            ]
        ].copy()

        group_result_df["nearest_cluster_label"] = nearest_cluster_label
        group_result_df["nearest_centroid_distance"] = nearest_centroid_distance

        group_distance_records.append(group_result_df)

    feature_distance_df = pd.concat(group_distance_records, ignore_index=True)

    geometric_summary_df = (
        feature_distance_df.groupby("is_anomaly_like", dropna=False)
        .agg(
            rows=("nearest_centroid_distance", "size"),
            mean_distance=("nearest_centroid_distance", "mean"),
            median_distance=("nearest_centroid_distance", "median"),
            p95_distance=("nearest_centroid_distance", lambda s: np.quantile(s, 0.95)),
        )
        .reset_index()
    )
    geometric_summary_df["feature_set"] = feature_set_name

    # Compare anomaly distances to non-anomaly distances in one compact row.
    anomaly_row = geometric_summary_df.loc[geometric_summary_df["is_anomaly_like"]].iloc[0]
    non_anomaly_row = geometric_summary_df.loc[~geometric_summary_df["is_anomaly_like"]].iloc[0]

    aggregate_geometric_records.append(
        {
            "feature_set": feature_set_name,
            "anomaly_rows": int(anomaly_row["rows"]),
            "non_anomaly_rows": int(non_anomaly_row["rows"]),
            "anomaly_mean_distance": anomaly_row["mean_distance"],
            "non_anomaly_mean_distance": non_anomaly_row["mean_distance"],
            "anomaly_median_distance": anomaly_row["median_distance"],
            "non_anomaly_median_distance": non_anomaly_row["median_distance"],
            "distance_ratio_anomaly_vs_nonanomaly": (
                anomaly_row["mean_distance"] / non_anomaly_row["mean_distance"]
                if non_anomaly_row["mean_distance"] > 0
                else np.nan
            ),
            "anomaly_p95_distance": anomaly_row["p95_distance"],
            "non_anomaly_p95_distance": non_anomaly_row["p95_distance"],
        }
    )

dbscan_aggregate_geometric_validation_df = (
    pd.DataFrame(aggregate_geometric_records)
    .sort_values("feature_set")
    .reset_index(drop=True)
)

display(
    dbscan_aggregate_geometric_validation_df.style.format(
        {
            "anomaly_mean_distance": "{:.3f}",
            "non_anomaly_mean_distance": "{:.3f}",
            "anomaly_median_distance": "{:.3f}",
            "non_anomaly_median_distance": "{:.3f}",
            "distance_ratio_anomaly_vs_nonanomaly": "{:.3f}",
            "anomaly_p95_distance": "{:.3f}",
            "non_anomaly_p95_distance": "{:.3f}",
        }
    )
)

,feature_set,anomaly_rows,non_anomaly_rows,anomaly_mean_distance,non_anomaly_mean_distance,anomaly_median_distance,non_anomaly_median_distance,distance_ratio_anomaly_vs_nonanomaly,anomaly_p95_distance,non_anomaly_p95_distance
0,bus,3074,1469873,19.927,4.091,17.887,3.640,4.871,35.579,7.538
1,fhvhv,6086,1535714,20.042,3.625,18.711,3.076,5.528,34.702,7.459
2,multimodal,4065,1537735,41.389,7.795,36.544,6.653,5.310,70.549,15.835
3,subway,5853,905602,15.330,2.222,12.321,1.907,6.901,33.364,4.637
4,taxi,216,49784,19.858,3.907,18.661,3.382,5.083,34.094,7.870


Findings\. The aggregate geometric validation remains very strong across all five modalities: DBSCAN anomalies sit much farther from retained cluster structure than ordinary rows do, both on average and at the median\. The distance ratios are now even larger than before, ranging from about 4\.9x in Bus and 5\.1x in Taxi up to 6\.9x in Subway, with FHVHV and Multimodal both above 5\.3x\. That is a strong signal that the anomaly labels are not arbitrary leftovers, but genuinely more isolated observations in representation space than the non\-anomalous background\.

The same pattern holds in the upper tail\. Across every modality, anomaly p95 distances are dramatically above non\-anomaly p95 distances, with Multimodal especially far from ordinary structure at a p95 of 70\.549 versus 15\.835 for non\-anomalies\. Subway is still the clearest separator in relative terms, while Multimodal is the most extreme in absolute distance\. So the practical takeaway is that, at full\-surface scale, DBSCAN anomalies remain geometrically distinct from non\-anomalies in a very consistent and interpretable way, which gives us good evidence that the exported anomaly surfaces are capturing real isolation in the PCA space rather than incidental noise\.

Visualize the dominant anomaly zones within each modality\. The summary table tells us how concentrated each modality’s anomaly surface is overall, but this chart shows where that concentration lives\. We limit the view to the top 10 anomaly zones per modality so the dominant hotspots are easy to scan without burying the reader in the full ranking table\.

In [60]:
# ---------------------------------------------------------------------
# Rebuild DBSCAN zone-concentration summary from exported row-level files
# if it is not already in memory, then visualize top zones by modality
# ---------------------------------------------------------------------

dbscan_final_row_level_dir = FINAL_332_DIR / "dbscan_candidate_anomaly_row_level_parts"

assert dbscan_final_row_level_dir.exists(), (
    f"Missing DBSCAN final row-level directory: {dbscan_final_row_level_dir}"
)

required_zone_columns = [
    "feature_set",
    ZONE_ID_COLUMN,
    BOROUGH_COLUMN,
    ZONE_COLUMN,
    "is_anomaly_like",
]

zone_frames = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    feature_path = dbscan_final_row_level_dir / f"{feature_set_name}_dbscan_candidate_anomalies.parquet"
    assert feature_path.exists(), (
        f"Missing DBSCAN row-level anomaly file for {feature_set_name}: {feature_path}"
    )

    feature_df = pd.read_parquet(feature_path, columns=required_zone_columns)

    feature_df = feature_df.loc[
        feature_df["is_anomaly_like"].fillna(False)
    ].copy()

    if feature_df.empty:
        continue

    zone_counts_df = (
        feature_df.groupby(
            ["feature_set", ZONE_ID_COLUMN, BOROUGH_COLUMN, ZONE_COLUMN],
            dropna=False,
        )
        .size()
        .reset_index(name="anomaly_rows")
        .sort_values(
            ["feature_set", "anomaly_rows", BOROUGH_COLUMN, ZONE_COLUMN],
            ascending=[True, False, True, True],
        )
        .reset_index(drop=True)
    )

    zone_frames.append(zone_counts_df)

assert zone_frames, "No anomaly-like DBSCAN rows were found in the exported row-level files."

dbscan_zone_concentration_rank_df = (
    pd.concat(zone_frames, ignore_index=True)
    .sort_values(
        ["feature_set", "anomaly_rows", BOROUGH_COLUMN, ZONE_COLUMN],
        ascending=[True, False, True, True],
    )
    .reset_index(drop=True)
)

dbscan_zone_concentration_rank_df["feature_total_anomaly_rows"] = (
    dbscan_zone_concentration_rank_df.groupby("feature_set", dropna=False)["anomaly_rows"]
    .transform("sum")
)

dbscan_zone_concentration_rank_df["anomaly_share_within_modality"] = (
    dbscan_zone_concentration_rank_df["anomaly_rows"]
    / dbscan_zone_concentration_rank_df["feature_total_anomaly_rows"]
)

dbscan_zone_concentration_rank_df["zone_rank"] = (
    dbscan_zone_concentration_rank_df.groupby("feature_set", dropna=False)["anomaly_rows"]
    .rank(method="first", ascending=False)
    .astype(int)
)

dbscan_zone_concentration_top5_df = (
    dbscan_zone_concentration_rank_df.loc[
        dbscan_zone_concentration_rank_df["zone_rank"].le(5)
    ]
    .copy()
    .sort_values(["feature_set", "zone_rank"])
    .reset_index(drop=True)
)

top_zone_plot_df = dbscan_zone_concentration_top5_df.copy()

top_zone_plot_df["zone_label"] = (
    top_zone_plot_df["borough"].astype(str)
    + " | "
    + top_zone_plot_df["zone"].astype(str)
)

top_zone_plot_df["facet_zone_label"] = (
    top_zone_plot_df["feature_set"].str.upper()
    + " :: "
    + top_zone_plot_df["zone_label"]
)

feature_set_order = ["bus", "subway", "taxi", "fhvhv", "multimodal"]

top_zone_plot_df["feature_set"] = pd.Categorical(
    top_zone_plot_df["feature_set"],
    categories=feature_set_order,
    ordered=True,
)

display(
    dbscan_zone_concentration_top5_df[
        [
            "feature_set",
            "zone_rank",
            BOROUGH_COLUMN,
            ZONE_COLUMN,
            "anomaly_rows",
            "anomaly_share_within_modality",
        ]
    ].style.format(
        {
            "anomaly_share_within_modality": "{:.3f}",
        }
    )
)

# ---------------------------------------------------------------------
# Visualize the dominant anomaly zones within each modality
# ---------------------------------------------------------------------

fig = px.bar(
    top_zone_plot_df,
    x="anomaly_share_within_modality",
    y="facet_zone_label",
    facet_col="feature_set",
    facet_col_wrap=2,
    orientation="h",
    text="anomaly_share_within_modality",
    category_orders={"feature_set": feature_set_order},
    labels={
        "anomaly_share_within_modality": "Share of anomaly rows",
        "facet_zone_label": "Taxi zone",
        "feature_set": "Feature set",
    },
    title="Top 5 Anomaly Zones by Modality",
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside",
    cliponaxis=False,
)

fig.update_layout(
    width=980,
    height=680,
    margin=dict(l=170, r=40, t=80, b=60),
    showlegend=False,
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1].upper()))
fig.update_yaxes(autorange="reversed", matches=None)

for axis_name in fig.layout:
    if axis_name.startswith("yaxis"):
        axis = fig.layout[axis_name]
        if hasattr(axis, "ticktext") and axis.ticktext:
            axis.ticktext = [str(t).split(" :: ", 1)[-1] for t in axis.ticktext]

fig.update_yaxes(ticklabelstandoff=4)
fig.show()

,feature_set,zone_rank,borough,zone,anomaly_rows,anomaly_share_within_modality
0,bus,1,Queens,Jamaica,761,0.248
1,bus,2,Queens,Breezy Point/Fort Tilden/Riis Beach,311,0.101
2,bus,3,Staten Island,Oakwood,229,0.074
3,bus,4,Queens,Jamaica Bay,213,0.069
4,bus,5,Manhattan,Financial District South,163,0.053
5,fhvhv,1,Queens,Jamaica Bay,660,0.108
6,fhvhv,2,Staten Island,Great Kills Park,565,0.093
7,fhvhv,3,Staten Island,Freshkills Park,537,0.088
8,fhvhv,4,Queens,LaGuardia Airport,465,0.076
9,fhvhv,5,Bronx,Rikers Island,425,0.070


Findings\. The top\-zone concentration pattern still shows that each modality has its own geographic signature rather than one shared hotspot map\. Bus remains by far the most top\-heavy branch, with Jamaica alone accounting for roughly 24\.2% of all bus anomalies, and the rest of the top five dropping off sharply after that\. Subway is still strongly Manhattan\-centered, but the dominant zones are now led by the Upper East Side South, Upper East Side North, Midtown Center, and nearby core Manhattan areas rather than a pure Times Square cluster\. That still fits the earlier story of a CBD\-heavy and Manhattan\-core anomaly surface, even if the exact leading zones have shifted\.

Taxi and FHVHV continue to look like broader road\-network and travel\-corridor surfaces, but with somewhat different emphasis\. Taxi is led by Upper East Side South, Upper East Side North, Midtown Center, JFK Airport, and LaGuardia Airport, which mixes core Manhattan demand with major airport access zones\. FHVHV is even more airport\- and outer\-borough\-facing, with its largest shares in Queens airport\-adjacent and Brooklyn/Staten Island zones rather than Manhattan alone\. Multimodal blends both patterns: it retains a Manhattan core through Times Square/Theatre District and Midtown Center, while also pulling in Queens\-centered anomaly zones such as Jamaica and Breezy Point/Fort Tilden/Riis Beach\. So the practical takeaway is that the DBSCAN anomaly surfaces are spatially structured in clearly modality\-specific ways, which makes the upcoming choropleths substantively useful rather than just decorative repetitions of the same geography\.

### Visualize anomaly concentration on NYC taxi\-zone choropleths

The aggregate tables already told us which operating regimes carry the heaviest anomaly burden\. This subsection makes that spatial pattern legible\. First, we map where each modality’s final DBSCAN anomalies concentrate across NYC taxi zones\. Then we zoom into Taxi and rotate across temporal buckets so we can see whether Taxi’s anomaly burden is consistently airport\- and gateway\-oriented or whether that spatial pattern changes meaningfully by mobility state\.

Prepare the taxi\-zone geometry and anomaly map inputs\. Before rendering maps, we want to confirm that every exported final anomaly row can be placed onto the taxi\-zone geometry cleanly\. This block builds the two choropleth\-ready datasets: one aggregated by modality and one aggregated by Taxi temporal bucket\.

In [61]:
# ---------------------------------------------------------------------
# Prepare taxi-zone geometry and anomaly map inputs
# ---------------------------------------------------------------------

# The final export block writes one row-level DBSCAN anomaly file per modality
# into this fixed directory, so we rebuild the file paths directly here instead
# of relying on any optional path column in the manifest.
dbscan_final_row_level_dir = FINAL_332_DIR / "dbscan_candidate_anomaly_row_level_parts"


def load_harmonized_taxi_zone_geometry() -> gpd.GeoDataFrame:
    assert HARMONIZED_TAXI_ZONES_PATH.exists(), (
        f"Harmonized taxi-zone file is missing: {HARMONIZED_TAXI_ZONES_PATH}"
    )

    taxi_zones_df = pd.read_parquet(HARMONIZED_TAXI_ZONES_PATH).copy()

    required_geometry_cols = {"location_id", "zone", "borough", "geometry"}
    missing_geometry_cols = required_geometry_cols.difference(taxi_zones_df.columns)
    assert not missing_geometry_cols, (
        "Harmonized taxi-zone file is missing required columns: "
        f"{sorted(missing_geometry_cols)}"
    )

    taxi_zones_df["geometry"] = taxi_zones_df["geometry"].apply(
        lambda geom: wkb.loads(geom) if pd.notna(geom) else None
    )

    taxi_zones_gdf = gpd.GeoDataFrame(
        taxi_zones_df,
        geometry="geometry",
        crs="EPSG:4326",
    )

    taxi_zones_gdf = (
        taxi_zones_gdf.rename(columns={"location_id": "taxi_zone_id"})
        .loc[lambda df: df["geometry"].notna()]
        [["taxi_zone_id", "zone", "borough", "geometry"]]
        .drop_duplicates(subset=["taxi_zone_id"])
        .copy()
    )

    taxi_zones_gdf["taxi_zone_id"] = taxi_zones_gdf["taxi_zone_id"].astype(str)

    return taxi_zones_gdf


if "dbscan_final_row_level_manifest_df" not in globals():
    dbscan_final_row_level_manifest_df = pd.read_csv(
        DBSCAN_ANOMALY_EXPORT_MANIFEST_PATH
    )

if "dbscan_final_anomaly_rows_for_maps_df" not in globals():
    anomaly_frames = []

    for _, manifest_row in (
        dbscan_final_row_level_manifest_df
        .sort_values(["feature_set", "representation_name", "min_samples", "eps_band"])
        .iterrows()
    ):
        feature_set_name = manifest_row["feature_set"]
        branch_path = (
            dbscan_final_row_level_dir
            / f"{feature_set_name}_dbscan_candidate_anomalies.parquet"
        )

        assert branch_path.exists(), (
            f"Missing final DBSCAN row-level export: {branch_path}"
        )

        branch_df = pd.read_parquet(branch_path)

        required_cols = {"taxi_zone_id", "temporal_bucket", "cluster_label"}
        missing_cols = required_cols.difference(branch_df.columns)
        assert not missing_cols, (
            f"{branch_path.name} is missing required columns: {sorted(missing_cols)}"
        )

        branch_anomalies_df = (
            branch_df.loc[
                branch_df["cluster_label"].eq(-1),
                ["taxi_zone_id", "temporal_bucket"],
            ]
            .copy()
        )

        branch_anomalies_df["feature_set"] = manifest_row["feature_set"]
        branch_anomalies_df["representation_name"] = manifest_row["representation_name"]
        branch_anomalies_df["min_samples"] = manifest_row["min_samples"]
        branch_anomalies_df["eps_band"] = manifest_row["eps_band"]
        branch_anomalies_df["eps_value"] = manifest_row["eps_value"]

        anomaly_frames.append(branch_anomalies_df)

    dbscan_final_anomaly_rows_for_maps_df = pd.concat(
        anomaly_frames,
        ignore_index=True,
    )

taxi_zone_gdf = load_harmonized_taxi_zone_geometry()

dbscan_final_anomaly_rows_for_maps_df["taxi_zone_id"] = (
    dbscan_final_anomaly_rows_for_maps_df["taxi_zone_id"].astype(str)
)

dbscan_modality_zone_map_df = (
    dbscan_final_anomaly_rows_for_maps_df.groupby(
        ["feature_set", "taxi_zone_id"],
        dropna=False,
    )
    .size()
    .rename("anomaly_rows")
    .reset_index()
)

dbscan_modality_totals_df = (
    dbscan_modality_zone_map_df.groupby("feature_set", dropna=False)["anomaly_rows"]
    .sum()
    .rename("feature_set_anomaly_rows")
    .reset_index()
)

dbscan_modality_zone_map_df = (
    dbscan_modality_zone_map_df.merge(
        dbscan_modality_totals_df,
        on="feature_set",
        how="left",
    )
    .assign(
        anomaly_share_within_modality=lambda df: (
            df["anomaly_rows"] / df["feature_set_anomaly_rows"]
        ),
        anomaly_share_pct_within_modality=lambda df: (
            100.0 * df["anomaly_share_within_modality"]
        ),
    )
)

dbscan_taxi_temporal_zone_map_df = (
    dbscan_final_anomaly_rows_for_maps_df.loc[
        dbscan_final_anomaly_rows_for_maps_df["feature_set"].eq("taxi")
    ]
    .groupby(["temporal_bucket", "taxi_zone_id"], dropna=False)
    .size()
    .rename("anomaly_rows")
    .reset_index()
)

dbscan_taxi_temporal_totals_df = (
    dbscan_taxi_temporal_zone_map_df.groupby("temporal_bucket", dropna=False)["anomaly_rows"]
    .sum()
    .rename("temporal_bucket_anomaly_rows")
    .reset_index()
)

dbscan_taxi_temporal_zone_map_df = (
    dbscan_taxi_temporal_zone_map_df.merge(
        dbscan_taxi_temporal_totals_df,
        on="temporal_bucket",
        how="left",
    )
    .assign(
        anomaly_share_within_taxi_bucket=lambda df: (
            df["anomaly_rows"] / df["temporal_bucket_anomaly_rows"]
        ),
        anomaly_share_pct_within_taxi_bucket=lambda df: (
            100.0 * df["anomaly_share_within_taxi_bucket"]
        ),
    )
)

dbscan_modality_zone_map_df = dbscan_modality_zone_map_df.merge(
    taxi_zone_gdf[["taxi_zone_id", "borough", "zone"]],
    on="taxi_zone_id",
    how="left",
)

dbscan_taxi_temporal_zone_map_df = dbscan_taxi_temporal_zone_map_df.merge(
    taxi_zone_gdf[["taxi_zone_id", "borough", "zone"]],
    on="taxi_zone_id",
    how="left",
)

dbscan_choropleth_coverage_df = (
    dbscan_modality_zone_map_df.assign(
        zone_mapped=lambda df: df["zone"].notna()
    )
    .groupby("feature_set", dropna=False)
    .agg(
        anomaly_rows=("anomaly_rows", "sum"),
        unmapped_zone_rows=("zone_mapped", lambda s: int((~s).sum())),
        mapped_zone_count=("taxi_zone_id", "nunique"),
    )
    .reset_index()
)

dbscan_choropleth_coverage_df["status"] = np.where(
    dbscan_choropleth_coverage_df["unmapped_zone_rows"].eq(0),
    "pass",
    "review",
)

dbscan_taxi_temporal_choropleth_inventory_df = (
    dbscan_taxi_temporal_zone_map_df.groupby("temporal_bucket", dropna=False)
    .agg(
        anomaly_rows=("anomaly_rows", "sum"),
        mapped_zone_count=("taxi_zone_id", "nunique"),
    )
    .reset_index()
)

display(
    dbscan_choropleth_coverage_df.style.format(
        {
            "anomaly_rows": "{:,.0f}",
            "unmapped_zone_rows": "{:,.0f}",
            "mapped_zone_count": "{:,.0f}",
        }
    )
)

display(
    dbscan_taxi_temporal_choropleth_inventory_df.style.format(
        {
            "anomaly_rows": "{:,.0f}",
            "mapped_zone_count": "{:,.0f}",
        }
    )
)

assert dbscan_choropleth_coverage_df["status"].eq("pass").all(), (
    "One or more modality-level anomaly rows could not be mapped to taxi-zone geometry."
)

print("Taxi-zone choropleth inputs prepared.")

,feature_set,anomaly_rows,unmapped_zone_rows,mapped_zone_count,status
0,bus,"3,074",0,102,pass
1,fhvhv,"6,086",0,160,pass
2,multimodal,"4,065",0,242,pass
3,subway,"5,853",0,150,pass
4,taxi,"7,027",0,255,pass


,temporal_bucket,anomaly_rows,mapped_zone_count
0,weekday_am_peak,555,131
1,weekday_evening,"1,033",180
2,weekday_midday,874,122
3,weekday_overnight,795,201
4,weekday_pm_peak,975,144
5,weekend_am_peak,307,112
6,weekend_evening,668,133
7,weekend_midday,677,127
8,weekend_overnight,654,158
9,weekend_pm_peak,489,109


Taxi-zone choropleth inputs prepared.


Findings\. All five final DBSCAN anomaly surfaces mapped cleanly onto the taxi\-zone geometry, so the choropleths below are visualizing the full exported anomaly outputs rather than a partial spatial subset\. No modality lost any anomaly rows in the spatial join, and the mapped zone counts are substantial across every branch, ranging from 102 Bus zones up to 255 Taxi zones, which confirms that these anomaly surfaces are spatially distributed rather than confined to only a handful of locations\.

The temporal summary also shows that the mapped anomaly universe is not concentrated in just one time slice\. Weekday evening, weekday PM peak, weekday midday, and weekday overnight all carry large anomaly counts, while the weekend buckets remain meaningfully represented as well\. So the practical takeaway is that the DBSCAN maps below are complete, trustworthy spatial views of the exported anomaly surfaces, and that those surfaces span a broad enough set of zones and temporal buckets to support real geographic interpretation\.

Map overall anomaly concentration by modality\. This first choropleth answers a simple question: once we aggregate across dates and local comparison groups, where does each modality’s anomaly burden actually land? The dropdown switches modalities, and the color scale shows the share of that modality’s anomaly rows attributed to each taxi zone\.

In [62]:
# ---------------------------------------------------------------------
# Visualize anomaly concentration for one modality at a time
# ---------------------------------------------------------------------

MODALITY_TO_MAP = "bus"  # bus | subway | taxi | fhvhv | multimodal

assert MODALITY_TO_MAP in MODEL_FEATURE_SET_NAMES, (
    f"Unsupported modality: {MODALITY_TO_MAP}"
)

# Simplify geometry to keep Deepnote output size manageable.
taxi_zone_plot_gdf = taxi_zone_gdf.copy().to_crs("EPSG:2263")
taxi_zone_plot_gdf["geometry"] = taxi_zone_plot_gdf["geometry"].simplify(
    tolerance=150,
    preserve_topology=True,
)
taxi_zone_plot_gdf = taxi_zone_plot_gdf.to_crs("EPSG:4326")

taxi_zone_geojson = json.loads(
    taxi_zone_plot_gdf[["taxi_zone_id", "geometry"]].to_json()
)

minx, miny, maxx, maxy = taxi_zone_plot_gdf.total_bounds
map_center = {
    "lon": float((minx + maxx) / 2.0),
    "lat": float((miny + maxy) / 2.0),
}

feature_zone_df = (
    taxi_zone_plot_gdf[["taxi_zone_id", "geometry"]]
    .merge(
        dbscan_modality_zone_map_df.loc[
            dbscan_modality_zone_map_df["feature_set"].eq(MODALITY_TO_MAP),
            [
                "taxi_zone_id",
                "borough",
                "zone",
                "anomaly_rows",
                "anomaly_share_pct_within_modality",
            ],
        ],
        on="taxi_zone_id",
        how="left",
    )
    .fillna(
        {
            "anomaly_rows": 0,
            "anomaly_share_pct_within_modality": 0.0,
            "borough": "",
            "zone": "",
        }
    )
)

brand_map_scale = [
    [0.00, BRAND_COLORS["ice"]],
    [0.35, BRAND_COLORS["pale_peach"]],
    [0.70, BRAND_COLORS["terracotta"]],
    [1.00, BRAND_COLORS["dark_teal"]],
]

fig = go.Figure(
    go.Choroplethmapbox(
        geojson=taxi_zone_geojson,
        locations=feature_zone_df["taxi_zone_id"],
        z=feature_zone_df["anomaly_share_pct_within_modality"],
        featureidkey="properties.taxi_zone_id",
        colorscale=brand_map_scale,
        zmin=0,
        zmax=float(dbscan_modality_zone_map_df["anomaly_share_pct_within_modality"].max()),
        marker_opacity=0.85,
        marker_line_width=0.25,
        marker_line_color=BRAND_COLORS["dark_teal"],
        colorbar=dict(
            title=dict(
                text="Share of modality anomalies (%)",
                font=dict(color=BRAND_COLORS["dark_teal"]),
            ),
            tickfont=dict(color=BRAND_COLORS["dark_teal"]),
            x=1.01,
        ),
        customdata=np.column_stack(
            [
                feature_zone_df["borough"],
                feature_zone_df["zone"],
                feature_zone_df["anomaly_rows"],
                feature_zone_df["anomaly_share_pct_within_modality"],
            ]
        ),
        hovertemplate=(
            "<b>%{customdata[0]} | %{customdata[1]}</b><br>"
            "Anomaly rows: %{customdata[2]:,.0f}<br>"
            "Share of modality anomalies: %{customdata[3]:.3f}%<extra></extra>"
        ),
    )
)

fig.update_layout(
    title=dict(
        text=f"Final DBSCAN Anomaly Concentration by Taxi Zone: {MODALITY_TO_MAP.upper()}",
        font=dict(color=BRAND_COLORS["dark_teal"]),
    ),
    paper_bgcolor="white",
    font=dict(color=BRAND_COLORS["dark_teal"]),
    width=980,
    height=730,
    margin=dict(l=20, r=20, t=80, b=20),
    mapbox=dict(
        style="carto-positron",
        zoom=9.3,
        center=map_center,
    ),
)

fig.show()

print(f"{MODALITY_TO_MAP.upper()} choropleth rendered.")

/tmp/ipykernel_228/1489372598.py:63: DeprecationWarning: *choroplethmapbox* is deprecated! Use *choroplethmap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Choroplethmapbox(


BUS choropleth rendered.


Findings\. The first choropleth reinforces the same basic Bus story, but much more intuitively than the ranked table: the anomaly surface is highly localized rather than citywide\. One especially strong hotspot dominates in eastern Queens around Jamaica, and a smaller secondary concentration appears across the Jamaica Bay and Rockaway\-area zones, with a lighter but still visible pocket in Staten Island\. Most of the rest of the city remains very lightly shaded, including most of Manhattan, Brooklyn, and the Bronx\.

So the practical takeaway stays the same: the Bus DBSCAN anomaly burden is real, but it is geographically concentrated\. This is not a broad diffuse bus irregularity surface spread evenly across the network; it is a modality surface being driven by a relatively small number of distinct spatial pockets, especially in southeastern Queens\.

Map Taxi anomaly concentration by temporal bucket\. Taxi was the most regime\-sensitive modality in the aggregate review, so this second choropleth keeps the spatial lens fixed on Taxi and rotates the temporal bucket instead\. Each dropdown view is normalized within that Taxi temporal bucket, so darker zones mean that a larger share of that bucket’s Taxi anomalies landed there\.

In [81]:
# ---------------------------------------------------------------------
# Visualize Taxi anomaly concentration for one temporal bucket at a time
# ---------------------------------------------------------------------

TAXI_TEMPORAL_BUCKET_TO_MAP = "weekday_pm_peak"

available_temporal_buckets = sorted(
    dbscan_taxi_temporal_zone_map_df["temporal_bucket"].dropna().unique().tolist()
)

assert TAXI_TEMPORAL_BUCKET_TO_MAP in available_temporal_buckets, (
    f"Unsupported temporal bucket: {TAXI_TEMPORAL_BUCKET_TO_MAP}. "
    f"Available values: {available_temporal_buckets}"
)

taxi_zone_plot_gdf = taxi_zone_gdf.copy().to_crs("EPSG:2263")
taxi_zone_plot_gdf["geometry"] = taxi_zone_plot_gdf["geometry"].simplify(
    tolerance=150,
    preserve_topology=True,
)
taxi_zone_plot_gdf = taxi_zone_plot_gdf.to_crs("EPSG:4326")

taxi_zone_geojson = json.loads(
    taxi_zone_plot_gdf[["taxi_zone_id", "geometry"]].to_json()
)

minx, miny, maxx, maxy = taxi_zone_plot_gdf.total_bounds
map_center = {
    "lon": float((minx + maxx) / 2.0),
    "lat": float((miny + maxy) / 2.0),
}

bucket_zone_df = (
    taxi_zone_plot_gdf[["taxi_zone_id", "geometry"]]
    .merge(
        dbscan_taxi_temporal_zone_map_df.loc[
            dbscan_taxi_temporal_zone_map_df["temporal_bucket"].eq(TAXI_TEMPORAL_BUCKET_TO_MAP),
            [
                "taxi_zone_id",
                "borough",
                "zone",
                "anomaly_rows",
                "anomaly_share_pct_within_taxi_bucket",
            ],
        ],
        on="taxi_zone_id",
        how="left",
    )
    .fillna(
        {
            "anomaly_rows": 0,
            "anomaly_share_pct_within_taxi_bucket": 0.0,
            "borough": "",
            "zone": "",
        }
    )
)

brand_map_scale = [
    [0.00, BRAND_COLORS["ice"]],
    [0.35, BRAND_COLORS["pale_peach"]],
    [0.70, BRAND_COLORS["terracotta"]],
    [1.00, BRAND_COLORS["dark_teal"]],
]

fig = go.Figure(
    go.Choroplethmapbox(
        geojson=taxi_zone_geojson,
        locations=bucket_zone_df["taxi_zone_id"],
        z=bucket_zone_df["anomaly_share_pct_within_taxi_bucket"],
        featureidkey="properties.taxi_zone_id",
        colorscale=brand_map_scale,
        zmin=0,
        zmax=float(dbscan_taxi_temporal_zone_map_df["anomaly_share_pct_within_taxi_bucket"].max()),
        marker_opacity=0.85,
        marker_line_width=0.25,
        marker_line_color=BRAND_COLORS["dark_teal"],
        colorbar=dict(
            title=dict(
                text="Share of Taxi<br>bucket anomalies (%)",
                font=dict(color=BRAND_COLORS["dark_teal"]),
            ),
            tickfont=dict(color=BRAND_COLORS["dark_teal"]),
            x=1.01,
            y=0.65,
        ),
        customdata=np.column_stack(
            [
                bucket_zone_df["borough"],
                bucket_zone_df["zone"],
                bucket_zone_df["anomaly_rows"],
                bucket_zone_df["anomaly_share_pct_within_taxi_bucket"],
            ]
        ),
        hovertemplate=(
            "<b>%{customdata[0]} | %{customdata[1]}</b><br>"
            "Taxi anomaly rows in bucket: %{customdata[2]:,.0f}<br>"
            "Share of bucket anomalies: %{customdata[3]:.3f}%<extra></extra>"
        ),
    )
)

fig.update_layout(
    title=dict(
        text=f"Taxi DBSCAN Anomaly Concentration by Taxi Zone: {TAXI_TEMPORAL_BUCKET_TO_MAP}",
        font=dict(color=BRAND_COLORS["dark_teal"]),
    ),
    paper_bgcolor="white",
    font=dict(color=BRAND_COLORS["dark_teal"]),
    width=960,
    height=740,
    margin=dict(l=20, r=20, t=80, b=20),
    mapbox=dict(
        style="carto-positron",
        zoom=9.3,
        center=map_center,
    ),
)

fig.show()

print(f"Taxi choropleth rendered for {TAXI_TEMPORAL_BUCKET_TO_MAP}.")

/tmp/ipykernel_228/1295700618.py:67: DeprecationWarning: *choroplethmapbox* is deprecated! Use *choroplethmap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Choroplethmapbox(


Taxi choropleth rendered for weekday_pm_peak.


Findings\. The Taxi weekday\_pm\_peak choropleth still shows a concentrated anomaly surface, but the geographic balance is a bit different from the earlier write\-up\. The clearest hotspot is now a tight Manhattan\-core cluster, with the strongest intensity packed into a small set of adjacent central zones rather than spread broadly across the city\. There is also a secondary concentration in southeastern Queens and a smaller pocket in Staten Island, but these are clearly weaker than the Manhattan core in this particular bucket\.

That pattern is still exactly what we would hope to see from an interpretable Taxi anomaly surface\. The anomalies are not being scattered randomly across the network; they are concentrating in a few recurring operating environments, especially the Manhattan core during the weekday PM peak, with smaller supporting pockets elsewhere\. So the practical takeaway is that Taxi anomalies in this bucket look spatially coherent and operationally plausible rather than noisy\.

### Section Summary

This chapter confirmed that the final DBSCAN anomaly surfaces remain selective and interpretable when evaluated across the full modeling universe\. Anomaly prevalence was not uniform across the city or calendar structure; instead, it concentrated in specific temporal\-bucket and policy\-geography contexts, with especially strong structure in Subway and Taxi\. The aggregate geometric validation was also very strong, showing that anomaly rows sit much farther from retained cluster structure than non\-anomalous rows do across every modality\. Spatially, the anomaly maps reinforced that each branch carries its own geographic signature rather than reproducing one universal hotspot pattern\.

Taken together, these checks suggest that the exported DBSCAN surfaces are behaving like meaningful modality\-specific anomaly candidates rather than diffuse outlier noise\. That does not mean every flagged row is equally important, but it does mean the retained DBSCAN outputs have enough internal coherence at full\-surface scale to advance into the downstream framework comparison and synthesis stages\.

## Close

3\.3\.2 turned DBSCAN from a candidate method into a full\-universe, modality\-specific anomaly framework\. By profiling local density structure, screening parameter regions, checking stability, exporting row\-level anomaly surfaces across the full modeling panel, and then validating those outputs at aggregate scale, the notebook retained one PCA\-based DBSCAN setting for each mobility system and produced geographically and geometrically coherent anomaly surfaces for Bus, Subway, Taxi, FHVHV, and Multimodal\. These outputs now give the project a consistent DBSCAN branch for downstream framework comparison, interpretation, and later evaluation alongside the other anomaly methods\.

### 3\.3\.2 Final Tables Inventory

selected\_dbscan\_configurations\.csv
Final retained DBSCAN configuration per modality, including the selected representation, eps band, eps value, and min\_samples\.

dbscan\_anomaly\_export\_manifest\.csv
Compact downstream manifest summarizing the final DBSCAN anomaly exports by modality, including row counts, anomaly\-like row counts, group coverage, and export status\.

dbscan\_candidate\_anomaly\_row\_level\_parts/
Directory containing the final metadata\-enriched DBSCAN anomaly row\-level parquet outputs, one file per modality\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>